# EDA 04 — Análise temporal e clustering

Coorte resolvida por `resolve_cohort_path` (`coorte_final.parquet`).
Descrição temporal usa todas as fichas; tabelas com desfecho usam só
o bloco `train_ref`. Desfecho e identificadores não formam clusters.


In [ ]:
from pathlib import Path
import gc
import json
import sys
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import os
import psutil

PROCESSO_ATUAL = psutil.Process(os.getpid())

def memoria_processo_gb():
    """Retorna a memória residente do kernel atual em GB."""

    return (
        PROCESSO_ATUAL.memory_info().rss
        / (1024 ** 3)
    )

print(
    "Memória inicial do kernel:",
    f"{memoria_processo_gb():.2f} GB",
)

def encontrar_raiz_projeto() -> Path:
    pasta_atual = Path.cwd().resolve()

    for pasta in [pasta_atual, *pasta_atual.parents]:
        possui_src = (pasta / "src").is_dir()
        possui_configs = (pasta / "configs").is_dir()

        if possui_src and possui_configs:
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


ROOT_DIR = encontrar_raiz_projeto()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.dados import resolve_cohort_path
from src import config as C

TIPO_COORTE, CAMINHO_COORTE = resolve_cohort_path(
    root=ROOT_DIR,
    kind="final",
    explicit_path=None,
)

CAMINHO_COORTE = Path(CAMINHO_COORTE)
NOME_COORTE = str(TIPO_COORTE)

if not CAMINHO_COORTE.is_file():
    raise FileNotFoundError(
        "A fonte de coorte resolvida não existe: "
        f"{CAMINHO_COORTE}"
    )

print(
    "Tipo de coorte:",
    NOME_COORTE,
)
print(
    "Arquivo da coorte:",
    CAMINHO_COORTE,
)


COLUNAS_ESTRUTURA = [
    "id_registro",
    "nu_notific",
    "dt_interna_convertida",
    "ano_internacao",
    "evolucao_normalizada",
    "alvo_obito_srag",
]

COLUNAS_DEMOGRAFIA = [
    "cs_sexo",
    "sexo_categoria",
    "nu_idade_n",
    "tp_idade",
    "idade_anos",
    "faixa_etaria",
    "cs_gestant",
    "cs_raca",
    "cs_zona",
]

COLUNAS_SINTOMAS_PLANEJADAS = [
    "febre",
    "tosse",
    "garganta",
    "dispneia",
    "desc_resp",
    "saturacao",
    "diarreia",
    "vomito",
    "dor_abd",
    "fadiga",
    "perd_olft",
    "perd_pala",
    "outro_sin",
]

COLUNAS_COMORBIDADES_PLANEJADAS = [
    "puerpera",
    "cardiopati",
    "hematologi",
    "sind_down",
    "hepatica",
    "asma",
    "diabetes",
    "neurologic",
    "pneumopati",
    "imunodepre",
    "renal",
    "obesidade",
    "out_morbi",
]

COLUNAS_CONTEXTO = [
    "cs_escol_n",
    "sg_uf",
    "vacina",
    "vacina_cov",
    "ave_suino",
]

COLUNAS_INTERVALOS = [
    "dt_sin_pri",
    "dt_interna",
    "dt_sin_pri_convertida",
    "dias_sintomas_ate_internacao",
    "flag_intervalo_invalido",
]

COLUNAS_CONTADORES_EXISTENTES = [
    "n_sintomas_confirmados",
    "n_sintomas_avaliados",
    "n_comorbidades_confirmadas",
    "n_comorbidades_avaliadas",
    "flag_multimorbidade",
]

COLUNAS_CLUSTERING = [
    "sin_semana",
    "cos_semana",
]


COLUNAS_SOLICITADAS = list(
    dict.fromkeys(
        COLUNAS_ESTRUTURA
        + COLUNAS_DEMOGRAFIA
        + COLUNAS_SINTOMAS_PLANEJADAS
        + COLUNAS_COMORBIDADES_PLANEJADAS
        + COLUNAS_CONTEXTO
        + COLUNAS_INTERVALOS
        + COLUNAS_CONTADORES_EXISTENTES
        + COLUNAS_CLUSTERING
    )
)


arquivo_parquet = pq.ParquetFile(
    CAMINHO_COORTE
)

COLUNAS_DISPONIVEIS_COORTE = set(
    arquivo_parquet.schema_arrow.names
)

COLUNAS_EXISTENTES = [
    coluna
    for coluna in COLUNAS_SOLICITADAS
    if coluna in COLUNAS_DISPONIVEIS_COORTE
]

COLUNAS_AUSENTES = [
    coluna
    for coluna in COLUNAS_SOLICITADAS
    if coluna not in COLUNAS_DISPONIVEIS_COORTE
]


COLUNAS_OBRIGATORIAS_COORTE = [
    "id_registro",
    "nu_notific",
    "dt_interna",
    "alvo_obito_srag",
]

colunas_obrigatorias_ausentes = [
    coluna
    for coluna in COLUNAS_OBRIGATORIAS_COORTE
    if coluna not in COLUNAS_DISPONIVEIS_COORTE
]

if colunas_obrigatorias_ausentes:
    raise KeyError(
        "A coorte não contém as colunas obrigatórias: "
        f"{colunas_obrigatorias_ausentes}"
    )


print(
    "Colunas disponíveis no Parquet:",
    len(COLUNAS_DISPONIVEIS_COORTE),
)

print(
    "Colunas solicitadas pelo notebook:",
    len(COLUNAS_SOLICITADAS),
)

print(
    "Colunas que serão carregadas:",
    len(COLUNAS_EXISTENTES),
)

if COLUNAS_AUSENTES:
    print(
        "Colunas opcionais ausentes, que poderão ser derivadas:",
        COLUNAS_AUSENTES,
    )


t0_carregamento = time.perf_counter()

try:
    coorte = pd.read_parquet(
        CAMINHO_COORTE,
        columns=COLUNAS_EXISTENTES,
        dtype_backend="pyarrow",
    )

    BACKEND_COORTE = "pyarrow"

except TypeError:
    coorte = pd.read_parquet(
        CAMINHO_COORTE,
        columns=COLUNAS_EXISTENTES,
    )

    BACKEND_COORTE = "numpy/object"


if coorte.empty:
    raise ValueError(
        "A coorte carregada está vazia."
    )


CAMINHO_PARTICAO_MESTRA = ROOT_DIR / "data" / "analytical" / "partitions_master.parquet"
CAMINHO_MANIFESTO_PARTICAO = ROOT_DIR / "data" / "analytical" / "manifesto_particao.json"

for caminho_necessario in (CAMINHO_PARTICAO_MESTRA, CAMINHO_MANIFESTO_PARTICAO):
    if not caminho_necessario.is_file():
        raise FileNotFoundError(
            "Arquivo da partição-mestra não encontrado: "
            f"{caminho_necessario}"
        )

particao_mestra = pd.read_parquet(
    CAMINHO_PARTICAO_MESTRA,
    columns=["id_registro", "particao"],
)

if not particao_mestra["id_registro"].is_unique:
    raise ValueError(
        "A partição-mestra tem id_registro repetido."
    )

particao_mestra["particao"] = particao_mestra["particao"].astype("category")

bloco_por_registro = particao_mestra.set_index("id_registro")["particao"]

coorte["particao"] = bloco_por_registro.reindex(
    coorte["id_registro"].to_numpy()
).array

del particao_mestra
del bloco_por_registro


with open(CAMINHO_MANIFESTO_PARTICAO, encoding="utf-8") as arquivo_manifesto:
    blocos_esperados = {
        nome_bloco: int(info_bloco["n_linhas"])
        for nome_bloco, info_bloco in json.load(arquivo_manifesto)["blocos"].items()
    }

blocos_observados = (
    coorte["particao"]
    .value_counts(dropna=True)
    .astype("int64")
    .to_dict()
)

if blocos_observados != blocos_esperados:
    raise ValueError(
        "As contagens por bloco divergem do manifesto da partição. "
        f"Observado: {blocos_observados}. Esperado: {blocos_esperados}."
    )

print(
    "Fichas sem bloco na partição-mestra:",
    int(coorte["particao"].isna().sum()),
)
print(
    "Fichas em train_ref:",
    blocos_observados["train_ref"],
)


memoria_apos_leitura_mb = (
    coorte.memory_usage(
        index=True,
        deep=True,
    )
    .sum()
    / 1_048_576
)


COLUNAS_CATEGORICAS_BAIXA_CARDINALIDADE = [
    coluna
    for coluna in (
        COLUNAS_DEMOGRAFIA
        + COLUNAS_SINTOMAS_PLANEJADAS
        + COLUNAS_COMORBIDADES_PLANEJADAS
        + COLUNAS_CONTEXTO
        + [
            "evolucao_normalizada",
            "faixa_etaria",
        ]
    )
    if coluna in coorte.columns
]


relatorio_compactacao = []


for coluna in COLUNAS_CATEGORICAS_BAIXA_CARDINALIDADE:
    memoria_antes = int(
        coorte[coluna]
        .memory_usage(
            index=False,
            deep=True,
        )
    )

    quantidade_categorias = int(
        coorte[coluna]
        .nunique(
            dropna=False
        )
    )

    if quantidade_categorias <= 500:
        coorte[coluna] = (
            coorte[coluna]
            .astype("category")
        )

    memoria_depois = int(
        coorte[coluna]
        .memory_usage(
            index=False,
            deep=True,
        )
    )

    relatorio_compactacao.append(
        {
            "coluna": coluna,
            "categorias": quantidade_categorias,
            "dtype_final": str(
                coorte[coluna].dtype
            ),
            "memoria_antes_mb": (
                memoria_antes
                / 1_048_576
            ),
            "memoria_depois_mb": (
                memoria_depois
                / 1_048_576
            ),
        }
    )


if "ano_internacao" in coorte.columns:
    coorte["ano_internacao"] = (
        pd.to_numeric(
            coorte["ano_internacao"],
            errors="coerce",
        )
        .astype("Int16")
    )


if "alvo_obito_srag" in coorte.columns:
    coorte["alvo_obito_srag"] = (
        pd.to_numeric(
            coorte["alvo_obito_srag"],
            errors="coerce",
        )
        .astype("Int8")
    )


if "idade_anos" in coorte.columns:
    coorte["idade_anos"] = (
        pd.to_numeric(
            coorte["idade_anos"],
            errors="coerce",
        )
        .astype("Float32")
    )


if "dias_sintomas_ate_internacao" in coorte.columns:
    coorte["dias_sintomas_ate_internacao"] = (
        pd.to_numeric(
            coorte["dias_sintomas_ate_internacao"],
            errors="coerce",
        )
        .astype("Float32")
    )


for coluna in [
    "n_sintomas_confirmados",
    "n_sintomas_avaliados",
    "n_comorbidades_confirmadas",
    "n_comorbidades_avaliadas",
]:
    if coluna in coorte.columns:
        coorte[coluna] = (
            pd.to_numeric(
                coorte[coluna],
                errors="coerce",
            )
            .astype("Int16")
        )


for coluna in [
    "flag_intervalo_invalido",
    "flag_multimorbidade",
]:
    if coluna in coorte.columns:
        coorte[coluna] = (
            pd.to_numeric(
                coorte[coluna],
                errors="coerce",
            )
            .fillna(0)
            .astype("int8")
        )


memoria_final_mb = (
    coorte.memory_usage(
        index=True,
        deep=True,
    )
    .sum()
    / 1_048_576
)

tempo_carregamento_s = (
    time.perf_counter()
    - t0_carregamento
)

gc.collect()


print(
    "Coorte utilizada:",
    NOME_COORTE,
)

print(
    "Caminho:",
    CAMINHO_COORTE,
)

print(
    "Backend:",
    BACKEND_COORTE,
)

print(
    "Dimensão carregada:",
    coorte.shape,
)

print(
    "Memória após leitura:",
    f"{memoria_apos_leitura_mb:,.2f} MB",
)

print(
    "Memória após compactação:",
    f"{memoria_final_mb:,.2f} MB",
)

if memoria_apos_leitura_mb > 0:
    reducao_memoria_pct = (
        1
        - memoria_final_mb
        / memoria_apos_leitura_mb
    ) * 100
else:
    reducao_memoria_pct = 0.0

print(
    "Redução após compactação:",
    f"{reducao_memoria_pct:,.2f}%",
)

print(
    "Tempo de leitura e compactação:",
    f"{tempo_carregamento_s:,.2f} s",
)


relatorio_compactacao_df = (
    pd.DataFrame(
        relatorio_compactacao
    )
    .sort_values(
        [
            "memoria_antes_mb",
            "coluna",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


display(
    relatorio_compactacao_df.head(20)
)

In [ ]:
if "dt_interna_convertida" not in coorte.columns:
    coorte["dt_interna_convertida"] = pd.to_datetime(
        coorte["dt_interna"].astype("string"),
        errors="coerce",
    )

if "dt_sin_pri_convertida" not in coorte.columns and "dt_sin_pri" in coorte.columns:
    coorte["dt_sin_pri_convertida"] = pd.to_datetime(
        coorte["dt_sin_pri"].astype("string"),
        errors="coerce",
    )

if "ano_internacao" not in coorte.columns:
    coorte["ano_internacao"] = (
        coorte["dt_interna_convertida"].dt.year.astype("Int16")
    )

if "evolucao_normalizada" not in coorte.columns:
    coorte["evolucao_normalizada"] = (
        pd.to_numeric(coorte["alvo_obito_srag"], errors="coerce")
        .map({1: "obito_srag", 0: "cura"})
        .astype("category")
    )

print(
    "Datas de internação não convertidas:",
    int(coorte["dt_interna_convertida"].isna().sum()),
)
print(
    "Anos presentes:",
    sorted(coorte["ano_internacao"].dropna().unique().tolist()),
)

In [ ]:
COLUNAS_ENTRADA_FEATURES = [
    "nu_idade_n",
    "tp_idade",
    "cs_sexo",
    "dt_sin_pri",
    "dt_interna",
]

colunas_entrada_ausentes_parquet = [
    coluna
    for coluna in COLUNAS_ENTRADA_FEATURES
    if coluna not in COLUNAS_DISPONIVEIS_COORTE
]

colunas_entrada_fora_da_leitura = [
    coluna
    for coluna in COLUNAS_ENTRADA_FEATURES
    if coluna not in COLUNAS_EXISTENTES
]

print(
    "Entradas ausentes no Parquet:",
    colunas_entrada_ausentes_parquet,
)
print(
    "Entradas disponíveis, mas fora da leitura:",
    colunas_entrada_fora_da_leitura,
)

if colunas_entrada_ausentes_parquet:
    raise KeyError(
        "O Parquet não contém todas as entradas exigidas pela "
        "engenharia centralizada: "
        f"{colunas_entrada_ausentes_parquet}"
    )

if colunas_entrada_fora_da_leitura:
    raise RuntimeError(
        "Há entradas da engenharia disponíveis no Parquet, "
        "mas excluídas da leitura projetada: "
        f"{colunas_entrada_fora_da_leitura}"
    )

In [ ]:
COLUNAS_OBRIGATORIAS_COORTE = [
    "nu_notific",
    "dt_interna_convertida",
    "ano_internacao",
    "evolucao_normalizada",
    "alvo_obito_srag",
]

colunas_obrigatorias_ausentes = [
    coluna
    for coluna in COLUNAS_OBRIGATORIAS_COORTE
    if coluna not in coorte.columns
]

if colunas_obrigatorias_ausentes:
    raise KeyError(
        "A coorte não contém as colunas obrigatórias: "
        f"{colunas_obrigatorias_ausentes}"
    )

dt_interna_sem_ausencia = (
    not coorte[
        "dt_interna_convertida"
    ]
    .isna()
    .any()
)

alvo_binario = (
    coorte[
        "alvo_obito_srag"
    ]
    .isin(
        [
            0,
            1,
        ]
    )
    .all()
)

periodo_valido = (
    coorte[
        "ano_internacao"
    ]
    .between(
        2019,
        2026,
    )
    .all()
)

verificacao_contrato = pd.DataFrame(
    {
        "verificacao": [
            "coorte_nao_vazia",
            "colunas_obrigatorias_presentes",
            "dt_interna_sem_ausencia",
            "alvo_binario",
            "ano_internacao_entre_2019_e_2026",
        ],
        "resultado": [
            not coorte.empty,
            not colunas_obrigatorias_ausentes,
            dt_interna_sem_ausencia,
            alvo_binario,
            periodo_valido,
        ],
    }
)

display(
    verificacao_contrato
)

if not verificacao_contrato[
    "resultado"
].all():
    falhas = (
        verificacao_contrato.loc[
            ~verificacao_contrato[
                "resultado"
            ],
            "verificacao",
        ]
        .tolist()
    )

    raise RuntimeError(
        "O contrato mínimo da coorte falhou em: "
        f"{falhas}"
    )

print(
    "Contrato mínimo da coorte validado."
)

In [ ]:
CAMINHO_VOLUME_ANO = (
    C.TABLES_DIR
    / "04_eda_volume_por_ano.csv"
)

CAMINHO_ALVO_ANO = (
    C.TABLES_DIR
    / "04_eda_alvo_por_ano.csv"
)

CAMINHO_AUSENCIA_ANO = (
    C.TABLES_DIR
    / "04_eda_ausencia_por_ano.csv"
)

CAMINHO_RESUMO_FONTE = (
    C.TABLES_DIR
    / "04_eda_resumo_fonte.csv"
)

C.TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

C.FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

resumo_fonte = pd.DataFrame(
    {
        "metrica": [
            "nome_coorte",
            "caminho_coorte",
            "quantidade_linhas",
            "quantidade_colunas",
        ],
        "valor": [
            NOME_COORTE,
            str(
                CAMINHO_COORTE
            ),
            len(
                coorte
            ),
            coorte.shape[1],
        ],
    }
)

resumo_fonte.to_csv(
    CAMINHO_RESUMO_FONTE,
    index=False,
    encoding="utf-8-sig",
)

display(
    resumo_fonte
)

In [ ]:
import matplotlib.pyplot as plt

CAMINHO_FIGURA_VOLUME_ANO = (
    C.FIGURES_DIR
    / "04_eda_volume_por_ano.png"
)

CAMINHO_FIGURA_ALVO_ANO = (
    C.FIGURES_DIR
    / "04_eda_proporcao_obito_por_ano.png"
)

print(
    "Figura de volume:",
    CAMINHO_FIGURA_VOLUME_ANO,
)

print(
    "Figura do desfecho:",
    CAMINHO_FIGURA_ALVO_ANO,
)

In [ ]:
volume_por_ano = (
    coorte.groupby(
        "ano_internacao",
        dropna=False,
    )
    .size()
    .rename(
        "quantidade_registros"
    )
    .reset_index()
    .sort_values(
        "ano_internacao"
    )
    .reset_index(drop=True)
)

volume_por_ano[
    "percentual_coorte"
] = (
    volume_por_ano[
        "quantidade_registros"
    ]
    .div(
        len(
            coorte
        )
    )
    .mul(100)
)

volume_por_ano[
    "ano_internacao"
] = (
    volume_por_ano[
        "ano_internacao"
    ]
    .astype("Int64")
)

volume_por_ano[
    "periodo_completo"
] = (
    volume_por_ano[
        "ano_internacao"
    ]
    .ne(2026)
)

total_volume_por_ano = int(
    volume_por_ano[
        "quantidade_registros"
    ]
    .sum()
)

if total_volume_por_ano != len(
    coorte
):
    raise RuntimeError(
        "A soma do volume por ano não corresponde "
        "ao total de registros da coorte."
    )

display(
    volume_por_ano
)

In [ ]:
contagem_alvo_por_ano = (
    coorte.groupby(
        [
            "ano_internacao",
            "alvo_obito_srag",
        ],
        dropna=False,
    )
    .size()
    .unstack(
        fill_value=0
    )
)

contagem_alvo_por_ano = (
    contagem_alvo_por_ano.reindex(
        columns=[
            0,
            1,
        ],
        fill_value=0,
    )
)

contagem_alvo_por_ano = (
    contagem_alvo_por_ano.rename(
        columns={
            0: "quantidade_cura",
            1: "quantidade_obito_srag",
        }
    )
    .reset_index()
)

alvo_por_ano = (
    contagem_alvo_por_ano
    .sort_values(
        "ano_internacao"
    )
    .reset_index(drop=True)
)

alvo_por_ano[
    "quantidade_total"
] = (
    alvo_por_ano[
        "quantidade_cura"
    ]
    .add(
        alvo_por_ano[
            "quantidade_obito_srag"
        ]
    )
)

alvo_por_ano[
    "percentual_cura"
] = (
    alvo_por_ano[
        "quantidade_cura"
    ]
    .div(
        alvo_por_ano[
            "quantidade_total"
        ]
    )
    .mul(100)
)

alvo_por_ano[
    "percentual_obito_srag"
] = (
    alvo_por_ano[
        "quantidade_obito_srag"
    ]
    .div(
        alvo_por_ano[
            "quantidade_total"
        ]
    )
    .mul(100)
)

alvo_por_ano[
    "ano_internacao"
] = (
    alvo_por_ano[
        "ano_internacao"
    ]
    .astype("Int64")
)

total_alvo_por_ano = int(
    alvo_por_ano[
        "quantidade_total"
    ]
    .sum()
)

if total_alvo_por_ano != len(
    coorte
):
    raise RuntimeError(
        "A soma da distribuição do alvo por ano "
        "não corresponde ao total da coorte."
    )

display(
    alvo_por_ano
)

In [ ]:
resumo_temporal = (
    volume_por_ano.merge(
        alvo_por_ano,
        on="ano_internacao",
        how="left",
        validate="one_to_one",
    )
)

resumo_temporal[
    "diferenca_total"
] = (
    resumo_temporal[
        "quantidade_registros"
    ]
    .sub(
        resumo_temporal[
            "quantidade_total"
        ]
    )
)

if not resumo_temporal[
    "diferenca_total"
].eq(0).all():
    raise RuntimeError(
        "Existe divergência entre o volume anual "
        "e a distribuição anual do alvo."
    )

COLUNAS_RESUMO_TEMPORAL = [
    "ano_internacao",
    "quantidade_registros",
    "percentual_coorte",
    "quantidade_cura",
    "quantidade_obito_srag",
    "percentual_cura",
    "percentual_obito_srag",
    "periodo_completo",
]

resumo_temporal = (
    resumo_temporal[
        COLUNAS_RESUMO_TEMPORAL
    ]
    .copy()
)

display(
    resumo_temporal
)

In [ ]:
anos_volume = (
    resumo_temporal[
        "ano_internacao"
    ]
    .astype(int)
    .astype(str)
)

quantidades_volume = (
    resumo_temporal[
        "quantidade_registros"
    ]
)

cores_volume = [
    (
        "tab:orange"
        if ano == 2026
        else "tab:blue"
    )
    for ano in resumo_temporal[
        "ano_internacao"
    ]
]

figura_volume, eixo_volume = plt.subplots(
    figsize=(
        11,
        6,
    )
)

barras_volume = eixo_volume.bar(
    anos_volume,
    quantidades_volume,
    color=cores_volume,
)

eixo_volume.set_title(
    "Volume de registros por ano de internação"
)

eixo_volume.set_xlabel(
    "Ano da internação"
)

eixo_volume.set_ylabel(
    "Quantidade de registros"
)

eixo_volume.ticklabel_format(
    style="plain",
    axis="y",
)

for barra, quantidade in zip(
    barras_volume,
    quantidades_volume,
):
    rotulo_quantidade = (
        f"{int(quantidade):,}"
        .replace(
            ",",
            ".",
        )
    )

    eixo_volume.text(
        barra.get_x()
        + barra.get_width()
        / 2,
        barra.get_height(),
        rotulo_quantidade,
        ha="center",
        va="bottom",
        fontsize=9,
    )

figura_volume.text(
    0.01,
    0.01,
    (
        "Fonte: coorte analítica carregada pelo notebook. "
        f"N total = {len(coorte):,}"
        .replace(
            ",",
            ".",
        )
    ),
    fontsize=8,
)

figura_volume.tight_layout(
    rect=(
        0,
        0.04,
        1,
        1,
    )
)

figura_volume.savefig(
    CAMINHO_FIGURA_VOLUME_ANO,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_VOLUME_ANO,
)

In [ ]:
anos_alvo = (
    alvo_por_ano[
        "ano_internacao"
    ]
    .astype(int)
)

percentuais_obito = (
    alvo_por_ano[
        "percentual_obito_srag"
    ]
)

denominadores_ano = (
    alvo_por_ano[
        "quantidade_total"
    ]
)

figura_alvo, eixo_alvo = plt.subplots(
    figsize=(
        11,
        6,
    )
)

eixo_alvo.plot(
    anos_alvo,
    percentuais_obito,
    color="tab:red",
    marker="o",
    linewidth=2,
)

eixo_alvo.set_title(
    "Proporção observada de óbito por SRAG por ano de internação"
)

eixo_alvo.set_xlabel(
    "Ano da internação"
)

eixo_alvo.set_ylabel(
    "Óbito por SRAG"
)

eixo_alvo.set_xticks(
    anos_alvo
)

eixo_alvo.set_ylim(
    bottom=0,
)

eixo_alvo.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.1f}%"
        )
    )
)

for ano, percentual, denominador in zip(
    anos_alvo,
    percentuais_obito,
    denominadores_ano,
):
    denominador_formatado = (
        f"{int(denominador):,}"
        .replace(
            ",",
            ".",
        )
    )

    rotulo_ponto = (
        f"{percentual:.1f}%\n"
        f"n={denominador_formatado}"
    )

    eixo_alvo.annotate(
        rotulo_ponto,
        (
            ano,
            percentual,
        ),
        textcoords="offset points",
        xytext=(
            0,
            10,
        ),
        ha="center",
        fontsize=8,
    )

figura_alvo.text(
    0.01,
    0.01,
    (
        "Desfecho: evolucao = 2. "
        "Denominador anual restrito a evolucao = 1 ou 2."
    ),
    fontsize=8,
)

figura_alvo.tight_layout(
    rect=(
        0.0,
        0.04,
        1.0,
        1.0,
    )
)

figura_alvo.savefig(
    CAMINHO_FIGURA_ALVO_ANO,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_ALVO_ANO,
)

In [ ]:
volume_por_ano.to_csv(
    CAMINHO_VOLUME_ANO,
    index=False,
    encoding="utf-8-sig",
)

alvo_por_ano.to_csv(
    CAMINHO_ALVO_ANO,
    index=False,
    encoding="utf-8-sig",
)

CAMINHO_RESUMO_TEMPORAL = (
    C.TABLES_DIR
    / "04_eda_resumo_temporal.csv"
)

resumo_temporal.to_csv(
    CAMINHO_RESUMO_TEMPORAL,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Volume por ano salvo em:",
    CAMINHO_VOLUME_ANO,
)

print(
    "Distribuição do alvo salva em:",
    CAMINHO_ALVO_ANO,
)

print(
    "Resumo temporal salvo em:",
    CAMINHO_RESUMO_TEMPORAL,
)

In [ ]:
CAMINHO_IDADE_ANO = (
    C.TABLES_DIR
    / "04_eda_idade_por_ano.csv"
)

CAMINHO_FAIXA_ETARIA_ANO = (
    C.TABLES_DIR
    / "04_eda_faixa_etaria_por_ano.csv"
)

CAMINHO_SEXO_ANO = (
    C.TABLES_DIR
    / "04_eda_sexo_por_ano.csv"
)

CAMINHO_AUDITORIA_IDADE = (
    C.TABLES_DIR
    / "04_eda_auditoria_idade.csv"
)

CAMINHO_FIGURA_IDADE_ANO = (
    C.FIGURES_DIR
    / "04_eda_idade_por_ano.png"
)

CAMINHO_FIGURA_FAIXA_ETARIA = (
    C.FIGURES_DIR
    / "04_eda_faixa_etaria_por_ano.png"
)

print(
    "Tabela de idade:",
    CAMINHO_IDADE_ANO,
)

print(
    "Tabela de faixa etária:",
    CAMINHO_FAIXA_ETARIA_ANO,
)

print(
    "Tabela de sexo:",
    CAMINHO_SEXO_ANO,
)

In [ ]:
from src.features import montar_features

COLUNAS_ORIGEM_DEMOGRAFIA = [
    "nu_idade_n",
    "tp_idade",
    "cs_sexo",
]

colunas_origem_ausentes = [
    coluna
    for coluna in COLUNAS_ORIGEM_DEMOGRAFIA
    if coluna not in coorte.columns
]

if colunas_origem_ausentes:
    raise KeyError(
        "A coorte não contém as colunas de origem necessárias: "
        f"{colunas_origem_ausentes}"
    )

quantidade_linhas_antes = len(coorte)
indice_antes = coorte.index.copy()

coorte = montar_features(coorte)

COLUNAS_DEMOGRAFICAS_CENTRALIZADAS = [
    "idade_anos",
    "faixa_etaria",
    "sexo_categoria",
]

colunas_demograficas_ausentes = [
    coluna
    for coluna in COLUNAS_DEMOGRAFICAS_CENTRALIZADAS
    if coluna not in coorte.columns
]

if colunas_demograficas_ausentes:
    raise KeyError(
        "A engenharia centralizada não produziu as colunas esperadas: "
        f"{colunas_demograficas_ausentes}"
    )

if len(coorte) != quantidade_linhas_antes:
    raise RuntimeError(
        "A engenharia de atributos alterou a quantidade de registros: "
        f"{quantidade_linhas_antes} -> {len(coorte)}"
    )

if not coorte.index.equals(indice_antes):
    raise RuntimeError(
        "A engenharia de atributos alterou o índice ou a ordem dos registros."
    )

idade_valor = pd.to_numeric(
    coorte["nu_idade_n"],
    errors="coerce",
)

tipo_idade = (
    coorte["tp_idade"]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True,
    )
    .replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        }
    )
)

coorte["flag_idade_ausente"] = (
    idade_valor.isna()
    | tipo_idade.isna()
).astype("int8")

coorte["flag_tipo_idade_invalido"] = (
    tipo_idade.notna()
    & ~tipo_idade.isin(["1", "2", "3"])
).astype("int8")

coorte["flag_idade_nao_convertida"] = (
    idade_valor.notna()
    & tipo_idade.notna()
    & coorte["idade_anos"].isna()
).astype("int8")

print(
    "Idades padronizadas válidas:",
    int(coorte["idade_anos"].notna().sum()),
)
print(
    "Idades ausentes:",
    int(coorte["flag_idade_ausente"].sum()),
)
print(
    "Idades não convertidas ou implausíveis:",
    int(coorte["flag_idade_nao_convertida"].sum()),
)
print(
    "Tipos de idade inválidos:",
    int(coorte["flag_tipo_idade_invalido"].sum()),
)
print(
    "Categorias de faixa etária:",
    coorte["faixa_etaria"].cat.categories.tolist(),
)
print(
    "Categorias de sexo:",
    coorte["sexo_categoria"].cat.categories.tolist(),
)

In [ ]:
auditoria_idade = (
    pd.DataFrame(
        {
            "tipo_idade": tipo_idade,
            "idade_valor_original": idade_valor,
            "idade_anos": coorte[
                "idade_anos"
            ],
            "flag_idade_ausente": coorte[
                "flag_idade_ausente"
            ],
            "flag_idade_nao_convertida": coorte[
                "flag_idade_nao_convertida"
            ],
            "flag_tipo_idade_invalido": coorte[
                "flag_tipo_idade_invalido"
            ],
        }
    )
    .groupby(
        "tipo_idade",
        dropna=False,
    )
    .agg(
        quantidade_registros=(
            "idade_valor_original",
            "size",
        ),
        quantidade_valor_informado=(
            "idade_valor_original",
            "count",
        ),
        quantidade_idade_convertida=(
            "idade_anos",
            "count",
        ),
        quantidade_idade_ausente=(
            "flag_idade_ausente",
            "sum",
        ),
        quantidade_nao_convertida=(
            "flag_idade_nao_convertida",
            "sum",
        ),
        quantidade_tipo_invalido=(
            "flag_tipo_idade_invalido",
            "sum",
        ),
    )
    .reset_index()
)

auditoria_idade[
    "percentual_convertido"
] = (
    auditoria_idade[
        "quantidade_idade_convertida"
    ]
    .div(
        auditoria_idade[
            "quantidade_registros"
        ]
    )
    .mul(100)
)

display(
    auditoria_idade
)

In [ ]:
idade_por_ano = (
    coorte.groupby(
        "ano_internacao",
        dropna=False,
    )
    .agg(
        quantidade_registros=(
            "ano_internacao",
            "size",
        ),
        quantidade_idade_valida=(
            "idade_anos",
            "count",
        ),
        idade_media=(
            "idade_anos",
            "mean",
        ),
        idade_mediana=(
            "idade_anos",
            "median",
        ),
        idade_minima=(
            "idade_anos",
            "min",
        ),
        idade_maxima=(
            "idade_anos",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        "ano_internacao"
    )
    .reset_index(drop=True)
)

quantil_25_idade = (
    coorte.groupby(
        "ano_internacao",
        dropna=False,
    )[
        "idade_anos"
    ]
    .quantile(0.25)
    .rename(
        "idade_q1"
    )
    .reset_index()
)

quantil_75_idade = (
    coorte.groupby(
        "ano_internacao",
        dropna=False,
    )[
        "idade_anos"
    ]
    .quantile(0.75)
    .rename(
        "idade_q3"
    )
    .reset_index()
)

idade_por_ano = (
    idade_por_ano.merge(
        quantil_25_idade,
        on="ano_internacao",
        how="left",
        validate="one_to_one",
    )
    .merge(
        quantil_75_idade,
        on="ano_internacao",
        how="left",
        validate="one_to_one",
    )
)

idade_por_ano[
    "idade_iqr"
] = (
    idade_por_ano[
        "idade_q3"
    ]
    .sub(
        idade_por_ano[
            "idade_q1"
        ]
    )
)

idade_por_ano[
    "quantidade_idade_ausente"
] = (
    idade_por_ano[
        "quantidade_registros"
    ]
    .sub(
        idade_por_ano[
            "quantidade_idade_valida"
        ]
    )
)

idade_por_ano[
    "percentual_idade_ausente"
] = (
    idade_por_ano[
        "quantidade_idade_ausente"
    ]
    .div(
        idade_por_ano[
            "quantidade_registros"
        ]
    )
    .mul(100)
)

COLUNAS_IDADE_ANO = [
    "ano_internacao",
    "quantidade_registros",
    "quantidade_idade_valida",
    "quantidade_idade_ausente",
    "percentual_idade_ausente",
    "idade_media",
    "idade_mediana",
    "idade_q1",
    "idade_q3",
    "idade_iqr",
    "idade_minima",
    "idade_maxima",
]

idade_por_ano = (
    idade_por_ano[
        COLUNAS_IDADE_ANO
    ]
    .copy()
)

display(
    idade_por_ano
)

In [ ]:
COLUNAS_DEMOGRAFICAS_CENTRALIZADAS = [
    "idade_anos",
    "faixa_etaria",
    "sexo_categoria",
]

colunas_demograficas_ausentes = [
    coluna
    for coluna in COLUNAS_DEMOGRAFICAS_CENTRALIZADAS
    if coluna not in coorte.columns
]

if colunas_demograficas_ausentes:
    raise KeyError(
        "A coorte não contém todas as variáveis demográficas "
        "centralizadas necessárias para esta análise: "
        f"{colunas_demograficas_ausentes}"
    )

if not isinstance(coorte["faixa_etaria"].dtype, pd.CategoricalDtype):
    raise TypeError(
        "faixa_etaria não preservou o tipo categórico centralizado."
    )

if not isinstance(coorte["sexo_categoria"].dtype, pd.CategoricalDtype):
    raise TypeError(
        "sexo_categoria não preservou o tipo categórico centralizado."
    )

ROTULOS_FAIXA_ETARIA = (
    coorte["faixa_etaria"]
    .cat.categories
    .astype("string")
    .tolist()
)

ROTULOS_SEXO = (
    coorte["sexo_categoria"]
    .cat.categories
    .astype("string")
    .tolist()
)

ROTULO_FAIXA_AUSENTE = "idade ausente ou inválida"
ROTULO_SEXO_AUSENTE = "sexo ausente ou inesperado"

faixa_etaria_eda = (
    coorte["faixa_etaria"]
    .astype("string")
    .fillna(ROTULO_FAIXA_AUSENTE)
)

sexo_categoria_eda = (
    coorte["sexo_categoria"]
    .astype("string")
    .fillna(ROTULO_SEXO_AUSENTE)
)

ORDEM_FAIXA_ETARIA = (
    ROTULOS_FAIXA_ETARIA
    + [ROTULO_FAIXA_AUSENTE]
)

ORDEM_SEXO = (
    ROTULOS_SEXO
    + [ROTULO_SEXO_AUSENTE]
)

faixa_etaria_por_ano = (
    pd.DataFrame(
        {
            "ano_internacao": coorte["ano_internacao"],
            "faixa_etaria": faixa_etaria_eda,
        }
    )
    .groupby(
        ["ano_internacao", "faixa_etaria"],
        dropna=False,
    )
    .size()
    .rename("quantidade")
    .reset_index()
)

faixa_etaria_por_ano["percentual_ano"] = (
    faixa_etaria_por_ano["quantidade"]
    .div(
        faixa_etaria_por_ano.groupby(
            "ano_internacao"
        )["quantidade"].transform("sum")
    )
    .mul(100)
)

ordem_faixa = {
    categoria: posicao
    for posicao, categoria in enumerate(ORDEM_FAIXA_ETARIA)
}

faixa_etaria_por_ano["ordem_faixa"] = (
    faixa_etaria_por_ano["faixa_etaria"]
    .map(ordem_faixa)
    .fillna(len(ordem_faixa))
)

faixa_etaria_por_ano = (
    faixa_etaria_por_ano
    .sort_values(["ano_internacao", "ordem_faixa"])
    .drop(columns=["ordem_faixa"])
    .reset_index(drop=True)
)

sexo_por_ano = (
    pd.DataFrame(
        {
            "ano_internacao": coorte["ano_internacao"],
            "sexo_categoria": sexo_categoria_eda,
        }
    )
    .groupby(
        ["ano_internacao", "sexo_categoria"],
        dropna=False,
    )
    .size()
    .rename("quantidade")
    .reset_index()
)

sexo_por_ano["percentual_ano"] = (
    sexo_por_ano["quantidade"]
    .div(
        sexo_por_ano.groupby(
            "ano_internacao"
        )["quantidade"].transform("sum")
    )
    .mul(100)
)

ordem_sexo = {
    categoria: posicao
    for posicao, categoria in enumerate(ORDEM_SEXO)
}

sexo_por_ano["ordem_sexo"] = (
    sexo_por_ano["sexo_categoria"]
    .map(ordem_sexo)
    .fillna(len(ordem_sexo))
)

sexo_por_ano = (
    sexo_por_ano
    .sort_values(["ano_internacao", "ordem_sexo"])
    .drop(columns=["ordem_sexo"])
    .reset_index(drop=True)
)

auditoria_categorias_demograficas = pd.DataFrame(
    {
        "variavel": ["faixa_etaria", "sexo_categoria"],
        "quantidade_registros": [len(coorte), len(coorte)],
        "quantidade_categorias": [
            coorte["faixa_etaria"].nunique(dropna=True),
            coorte["sexo_categoria"].nunique(dropna=True),
        ],
        "quantidade_ausente_ou_invalida": [
            int(coorte["faixa_etaria"].isna().sum()),
            int(coorte["sexo_categoria"].isna().sum()),
        ],
    }
)

display(auditoria_categorias_demograficas)
display(faixa_etaria_por_ano.head(30))
display(sexo_por_ano)

In [ ]:
anos_idade = (
    idade_por_ano[
        "ano_internacao"
    ]
    .astype(int)
    .to_numpy()
)

idade_mediana_grafico = (
    idade_por_ano[
        "idade_mediana"
    ]
    .astype(float)
    .to_numpy()
)

idade_q1_grafico = (
    idade_por_ano[
        "idade_q1"
    ]
    .astype(float)
    .to_numpy()
)

idade_q3_grafico = (
    idade_por_ano[
        "idade_q3"
    ]
    .astype(float)
    .to_numpy()
)

figura_idade, eixo_idade = plt.subplots(
    figsize=(
        11,
        6,
    )
)

eixo_idade.plot(
    anos_idade,
    idade_mediana_grafico,
    color="tab:blue",
    marker="o",
    linewidth=2,
    label="Mediana",
)

eixo_idade.fill_between(
    anos_idade,
    idade_q1_grafico,
    idade_q3_grafico,
    color="tab:blue",
    alpha=0.2,
    label="Intervalo interquartil",
)

eixo_idade.set_title(
    "Idade mediana e intervalo interquartil por ano de internação"
)

eixo_idade.set_xlabel(
    "Ano da internação"
)

eixo_idade.set_ylabel(
    "Idade em anos"
)

eixo_idade.set_xticks(
    anos_idade
)

eixo_idade.legend()

for ano, mediana in zip(
    anos_idade,
    idade_mediana_grafico,
):
    eixo_idade.annotate(
        f"{mediana:.1f}",
        (
            ano,
            mediana,
        ),
        textcoords="offset points",
        xytext=(
            0,
            8,
        ),
        ha="center",
        fontsize=8,
    )

figura_idade.text(
    0.01,
    0.01,
    (
        "Idade derivada de nu_idade_n e tp_idade. "
        "Faixa sombreada: primeiro ao terceiro quartil."
    ),
    fontsize=8,
)

figura_idade.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1,
    ]
)

figura_idade.savefig(
    CAMINHO_FIGURA_IDADE_ANO,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_IDADE_ANO,
)

In [ ]:
if "ORDEM_FAIXA_ETARIA" not in globals():
    raise NameError(
        "A ordem centralizada das faixas etárias não foi criada. "
        "Execute novamente a célula demográfica desde o início."
    )

faixa_etaria_grafico = (
    faixa_etaria_por_ano.pivot(
        index="ano_internacao",
        columns="faixa_etaria",
        values="percentual_ano",
    )
    .reindex(
        columns=ORDEM_FAIXA_ETARIA,
        fill_value=0,
    )
    .fillna(0)
)

figura_faixa, eixo_faixa = plt.subplots(
    figsize=(
        13,
        7,
    )
)

faixa_etaria_grafico.plot(
    kind="bar",
    stacked=True,
    ax=eixo_faixa,
    width=0.85,
)

eixo_faixa.set_title(
    "Composição da coorte por faixa etária e ano de internação"
)

eixo_faixa.set_xlabel(
    "Ano da internação"
)

eixo_faixa.set_ylabel(
    "Percentual dos registros do ano"
)

eixo_faixa.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

eixo_faixa.legend(
    title="Faixa etária",
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)

figura_faixa.text(
    0.01,
    0.01,
    (
        "Cada barra representa 100% dos registros elegíveis "
        "do respectivo ano."
    ),
    fontsize=8,
)

figura_faixa.tight_layout(
    rect=[
        0,
        0.04,
        0.82,
        1,
    ]
)

figura_faixa.savefig(
    CAMINHO_FIGURA_FAIXA_ETARIA,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_FAIXA_ETARIA,
)

In [ ]:
idade_por_ano.to_csv(
    CAMINHO_IDADE_ANO,
    index=False,
    encoding="utf-8-sig",
)

faixa_etaria_por_ano.to_csv(
    CAMINHO_FAIXA_ETARIA_ANO,
    index=False,
    encoding="utf-8-sig",
)

sexo_por_ano.to_csv(
    CAMINHO_SEXO_ANO,
    index=False,
    encoding="utf-8-sig",
)

auditoria_idade.to_csv(
    CAMINHO_AUDITORIA_IDADE,
    index=False,
    encoding="utf-8-sig",
)

total_faixa_etaria = int(
    faixa_etaria_por_ano[
        "quantidade"
    ]
    .sum()
)

total_sexo = int(
    sexo_por_ano[
        "quantidade"
    ]
    .sum()
)

LIMITE_MINIMO_IDADE = 0.0
LIMITE_MAXIMO_IDADE = 120.0

idades_validas = (
    coorte["idade_anos"]
    .dropna()
)

idade_valida_entre_limites = (
    idades_validas
    .between(
        LIMITE_MINIMO_IDADE,
        LIMITE_MAXIMO_IDADE,
        inclusive="both",
    )
    .all()
)

verificacoes_demograficas = {
    "idade_por_ano_nao_vazia": (
        not idade_por_ano.empty
    ),
    "faixa_etaria_nao_vazia": (
        not faixa_etaria_por_ano.empty
    ),
    "sexo_por_ano_nao_vazio": (
        not sexo_por_ano.empty
    ),
    "total_faixa_etaria_igual_coorte": (
        total_faixa_etaria
        == len(coorte)
    ),
    "total_sexo_igual_coorte": (
        total_sexo
        == len(coorte)
    ),
    "idade_valida_entre_zero_e_120": (
        bool(idade_valida_entre_limites)
    ),
    "arquivo_idade_criado": (
        CAMINHO_IDADE_ANO.exists()
    ),
    "arquivo_faixa_etaria_criado": (
        CAMINHO_FAIXA_ETARIA_ANO.exists()
    ),
    "arquivo_sexo_criado": (
        CAMINHO_SEXO_ANO.exists()
    ),
    "arquivo_auditoria_idade_criado": (
        CAMINHO_AUDITORIA_IDADE.exists()
    ),
    "figura_idade_criada": (
        CAMINHO_FIGURA_IDADE_ANO.exists()
    ),
    "figura_faixa_etaria_criada": (
        CAMINHO_FIGURA_FAIXA_ETARIA.exists()
    ),
}

verificacoes_demograficas_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_demograficas.keys()
        ),
        "resultado": list(
            verificacoes_demograficas.values()
        ),
    }
)

display(
    verificacoes_demograficas_df
)

if not all(
    verificacoes_demograficas.values()
):
    falhas_demograficas = [
        verificacao
        for verificacao, resultado
        in verificacoes_demograficas.items()
        if not resultado
    ]

    raise RuntimeError(
        "A seção demográfica terminou com falhas: "
        f"{falhas_demograficas}"
    )

print(
    "Análise demográfica validada com sucesso."
)

In [ ]:
CAMINHO_DICIONARIO_OPERACIONAL = (
    C.CONFIGS_DIR
    / "dicionario_variaveis_v1.csv"
)

if not CAMINHO_DICIONARIO_OPERACIONAL.exists():
    raise FileNotFoundError(
        "O dicionário operacional não foi encontrado em: "
        f"{CAMINHO_DICIONARIO_OPERACIONAL}"
    )

dicionario_operacional = pd.read_csv(
    CAMINHO_DICIONARIO_OPERACIONAL,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
)

COLUNAS_DICIONARIO_NECESSARIAS = [
    "coluna",
    "nome_amigavel",
    "grupo",
    "uso_inicial_sugerido",
]

colunas_dicionario_ausentes = [
    coluna
    for coluna in COLUNAS_DICIONARIO_NECESSARIAS
    if coluna not in dicionario_operacional.columns
]

if colunas_dicionario_ausentes:
    raise KeyError(
        "O dicionário operacional não contém as colunas: "
        f"{colunas_dicionario_ausentes}"
    )

dicionario_operacional[
    "coluna"
] = (
    dicionario_operacional[
        "coluna"
    ]
    .astype("string")
    .str.strip()
)

dicionario_operacional = (
    dicionario_operacional.loc[
        dicionario_operacional[
            "coluna"
        ].ne("")
    ]
    .drop_duplicates(
        subset=[
            "coluna",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

colunas_dicionario_na_coorte = [
    coluna
    for coluna in dicionario_operacional[
        "coluna"
    ].tolist()
    if coluna in coorte.columns
]

colunas_dicionario_fora_coorte = [
    coluna
    for coluna in dicionario_operacional[
        "coluna"
    ].tolist()
    if coluna not in coorte.columns
]

if not colunas_dicionario_na_coorte:
    raise ValueError(
        "Nenhuma variável do dicionário foi encontrada na coorte."
    )

print(
    "Variáveis do dicionário encontradas na coorte:",
    len(
        colunas_dicionario_na_coorte
    ),
)

print(
    "Variáveis do dicionário ausentes na coorte:",
    len(
        colunas_dicionario_fora_coorte
    ),
)

if colunas_dicionario_fora_coorte:
    display(
        pd.DataFrame(
            {
                "coluna_ausente_na_coorte": (
                    colunas_dicionario_fora_coorte
                ),
            }
        )
    )

In [ ]:
CAMINHO_PREENCHIMENTO_ANO = (
    C.TABLES_DIR
    / "04_eda_preenchimento_por_variavel_ano.csv"
)

CAMINHO_RESUMO_PREENCHIMENTO = (
    C.TABLES_DIR
    / "04_eda_resumo_preenchimento_variaveis.csv"
)

CAMINHO_VARIACAO_PREENCHIMENTO = (
    C.TABLES_DIR
    / "04_eda_variacao_preenchimento.csv"
)

CAMINHO_FIGURA_VARIACAO_PREENCHIMENTO = (
    C.FIGURES_DIR
    / "04_eda_variacao_preenchimento.png"
)

print(
    "Tabela anual de preenchimento:",
    CAMINHO_PREENCHIMENTO_ANO,
)

print(
    "Resumo de preenchimento:",
    CAMINHO_RESUMO_PREENCHIMENTO,
)

print(
    "Figura de variação:",
    CAMINHO_FIGURA_VARIACAO_PREENCHIMENTO,
)

In [ ]:
quantidade_por_ano = (
    coorte.groupby(
        "ano_internacao",
        dropna=False,
    )
    .size()
    .rename(
        "quantidade_registros_ano"
    )
    .sort_index()
)

preenchidos_por_ano = (
    coorte.groupby(
        "ano_internacao",
        dropna=False,
    )[
        colunas_dicionario_na_coorte
    ]
    .count()
    .sort_index()
)

preenchidos_por_ano.index = (
    preenchidos_por_ano.index.astype(
        "Int64"
    )
)

quantidade_por_ano.index = (
    quantidade_por_ano.index.astype(
        "Int64"
    )
)

preenchimento_percentual_por_ano = (
    preenchidos_por_ano
    .div(
        quantidade_por_ano,
        axis=0,
    )
    .mul(100)
)

quantidade_preenchida_longa = (
    preenchidos_por_ano
    .reset_index()
    .melt(
        id_vars=[
            "ano_internacao",
        ],
        var_name="coluna",
        value_name="quantidade_preenchida",
    )
)

percentual_preenchido_longo = (
    preenchimento_percentual_por_ano
    .reset_index()
    .melt(
        id_vars=[
            "ano_internacao",
        ],
        var_name="coluna",
        value_name="percentual_preenchido",
    )
)

preenchimento_por_variavel_ano = (
    quantidade_preenchida_longa.merge(
        percentual_preenchido_longo,
        on=[
            "ano_internacao",
            "coluna",
        ],
        how="inner",
        validate="one_to_one",
    )
)

quantidade_por_ano_df = (
    quantidade_por_ano
    .reset_index()
)

preenchimento_por_variavel_ano = (
    preenchimento_por_variavel_ano.merge(
        quantidade_por_ano_df,
        on="ano_internacao",
        how="left",
        validate="many_to_one",
    )
)

preenchimento_por_variavel_ano[
    "quantidade_ausente"
] = (
    preenchimento_por_variavel_ano[
        "quantidade_registros_ano"
    ]
    .sub(
        preenchimento_por_variavel_ano[
            "quantidade_preenchida"
        ]
    )
)

preenchimento_por_variavel_ano[
    "percentual_ausente"
] = (
    100
    - preenchimento_por_variavel_ano[
        "percentual_preenchido"
    ]
)

metadados_variaveis = (
    dicionario_operacional[
        COLUNAS_DICIONARIO_NECESSARIAS
    ]
    .copy()
)

preenchimento_por_variavel_ano = (
    preenchimento_por_variavel_ano.merge(
        metadados_variaveis,
        on="coluna",
        how="left",
        validate="many_to_one",
    )
)

COLUNAS_PREENCHIMENTO_ANUAL = [
    "ano_internacao",
    "coluna",
    "nome_amigavel",
    "grupo",
    "uso_inicial_sugerido",
    "quantidade_registros_ano",
    "quantidade_preenchida",
    "quantidade_ausente",
    "percentual_preenchido",
    "percentual_ausente",
]

preenchimento_por_variavel_ano = (
    preenchimento_por_variavel_ano[
        COLUNAS_PREENCHIMENTO_ANUAL
    ]
    .sort_values(
        [
            "grupo",
            "coluna",
            "ano_internacao",
        ]
    )
    .reset_index(drop=True)
)

display(
    preenchimento_por_variavel_ano.head(30)
)

In [ ]:
quantidade_total_coorte = len(
    coorte
)

quantidade_preenchida_global = (
    coorte[
        colunas_dicionario_na_coorte
    ]
    .count()
    .rename(
        "quantidade_preenchida"
    )
)

resumo_preenchimento_variaveis = (
    quantidade_preenchida_global
    .reset_index()
    .rename(
        columns={
            "index": "coluna",
        }
    )
)

resumo_preenchimento_variaveis[
    "quantidade_registros"
] = quantidade_total_coorte

resumo_preenchimento_variaveis[
    "quantidade_ausente"
] = (
    resumo_preenchimento_variaveis[
        "quantidade_registros"
    ]
    .sub(
        resumo_preenchimento_variaveis[
            "quantidade_preenchida"
        ]
    )
)

resumo_preenchimento_variaveis[
    "percentual_preenchido"
] = (
    resumo_preenchimento_variaveis[
        "quantidade_preenchida"
    ]
    .div(
        resumo_preenchimento_variaveis[
            "quantidade_registros"
        ]
    )
    .mul(100)
)

resumo_preenchimento_variaveis[
    "percentual_ausente"
] = (
    100
    - resumo_preenchimento_variaveis[
        "percentual_preenchido"
    ]
)

resumo_preenchimento_variaveis = (
    resumo_preenchimento_variaveis.merge(
        metadados_variaveis,
        on="coluna",
        how="left",
        validate="one_to_one",
    )
)

COLUNAS_RESUMO_PREENCHIMENTO = [
    "coluna",
    "nome_amigavel",
    "grupo",
    "uso_inicial_sugerido",
    "quantidade_registros",
    "quantidade_preenchida",
    "quantidade_ausente",
    "percentual_preenchido",
    "percentual_ausente",
]

resumo_preenchimento_variaveis = (
    resumo_preenchimento_variaveis[
        COLUNAS_RESUMO_PREENCHIMENTO
    ]
    .sort_values(
        [
            "percentual_preenchido",
            "grupo",
            "coluna",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    resumo_preenchimento_variaveis.head(30)
)

In [ ]:
variacao_preenchimento = (
    preenchimento_por_variavel_ano.groupby(
        [
            "coluna",
            "nome_amigavel",
            "grupo",
            "uso_inicial_sugerido",
        ],
        dropna=False,
    )
    .agg(
        preenchimento_minimo=(
            "percentual_preenchido",
            "min",
        ),
        preenchimento_maximo=(
            "percentual_preenchido",
            "max",
        ),
        preenchimento_medio=(
            "percentual_preenchido",
            "mean",
        ),
        quantidade_anos_observados=(
            "ano_internacao",
            "nunique",
        ),
    )
    .reset_index()
)

variacao_preenchimento[
    "amplitude_pontos_percentuais"
] = (
    variacao_preenchimento[
        "preenchimento_maximo"
    ]
    .sub(
        variacao_preenchimento[
            "preenchimento_minimo"
        ]
    )
)

variacao_preenchimento[
    "flag_variacao_20_pp"
] = (
    variacao_preenchimento[
        "amplitude_pontos_percentuais"
    ]
    .ge(20)
)

variacao_preenchimento = (
    variacao_preenchimento
    .sort_values(
        [
            "amplitude_pontos_percentuais",
            "coluna",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    variacao_preenchimento.head(30)
)

In [ ]:
USOS_RELEVANTES_PARA_GRAFICO = {
    "modelo_a_nucleo",
    "modelo_b_ampliado",
    "modelo_c_sensibilidade",
    "derivar_atributo",
    "reavaliar_apos_eda",
    "reavaliar_apos_modelo_principal",
    "exploracao_subgrupo",
}

variaveis_relevantes_variacao = (
    variacao_preenchimento.loc[
        variacao_preenchimento[
            "uso_inicial_sugerido"
        ].isin(
            USOS_RELEVANTES_PARA_GRAFICO
        )
    ]
    .copy()
)

QUANTIDADE_VARIAVEIS_GRAFICO = 15

variaveis_grafico = (
    variaveis_relevantes_variacao.head(
        QUANTIDADE_VARIAVEIS_GRAFICO
    )
    .copy()
)

if variaveis_grafico.empty:
    raise ValueError(
        "Nenhuma variável foi selecionada para "
        "o gráfico de variação de preenchimento."
    )

variaveis_grafico[
    "rotulo_variavel"
] = (
    variaveis_grafico[
        "nome_amigavel"
    ]
    .where(
        variaveis_grafico[
            "nome_amigavel"
        ].str.strip().ne(""),
        variaveis_grafico[
            "coluna"
        ],
    )
)

display(
    variaveis_grafico[
        [
            "coluna",
            "rotulo_variavel",
            "grupo",
            "uso_inicial_sugerido",
            "preenchimento_minimo",
            "preenchimento_maximo",
            "amplitude_pontos_percentuais",
        ]
    ]
)

variaveis_grafico = (
    variaveis_grafico
    .sort_values(
        "amplitude_pontos_percentuais",
        ascending=True,
    )
    .reset_index(drop=True)
)

figura_variacao, eixo_variacao = plt.subplots(
    figsize=(
        12,
        8,
    )
)

barras_variacao = eixo_variacao.barh(
    variaveis_grafico[
        "rotulo_variavel"
    ],
    variaveis_grafico[
        "amplitude_pontos_percentuais"
    ],
    color="tab:purple",
)

eixo_variacao.set_title(
    "Variáveis candidatas com maior variação de preenchimento entre anos"
)

eixo_variacao.set_xlabel(
    "Amplitude do preenchimento em pontos percentuais"
)

eixo_variacao.set_ylabel(
    "Variável"
)

for barra, amplitude in zip(
    barras_variacao,
    variaveis_grafico[
        "amplitude_pontos_percentuais"
    ],
):
    eixo_variacao.text(
        barra.get_width(),
        barra.get_y()
        + barra.get_height()
        / 2,
        f"{amplitude:.1f} p.p.",
        va="center",
        ha="left",
        fontsize=8,
    )

figura_variacao.text(
    0.01,
    0.01,
    (
        "Amplitude calculada como a diferença entre o maior e o menor "
        "percentual de preenchimento anual observado."
    ),
    fontsize=8,
)

figura_variacao.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1,
    ]
)

figura_variacao.savefig(
    CAMINHO_FIGURA_VARIACAO_PREENCHIMENTO,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_VARIACAO_PREENCHIMENTO,
)

In [ ]:
preenchimento_por_variavel_ano.to_csv(
    CAMINHO_PREENCHIMENTO_ANO,
    index=False,
    encoding="utf-8-sig",
)

resumo_preenchimento_variaveis.to_csv(
    CAMINHO_RESUMO_PREENCHIMENTO,
    index=False,
    encoding="utf-8-sig",
)

variacao_preenchimento.to_csv(
    CAMINHO_VARIACAO_PREENCHIMENTO,
    index=False,
    encoding="utf-8-sig",
)

quantidade_anos_observados = (
    coorte[
        "ano_internacao"
    ]
    .nunique(
        dropna=True
    )
)

quantidade_linhas_esperada = (
    len(
        colunas_dicionario_na_coorte
    )
    * quantidade_anos_observados
)

percentuais_preenchimento_validos = (
    preenchimento_por_variavel_ano[
        "percentual_preenchido"
    ]
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

percentuais_ausencia_validos = (
    preenchimento_por_variavel_ano[
        "percentual_ausente"
    ]
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

soma_percentuais_valida = (
    preenchimento_por_variavel_ano[
        "percentual_preenchido"
    ]
    .add(
        preenchimento_por_variavel_ano[
            "percentual_ausente"
        ]
    )
    .sub(100)
    .abs()
    .lt(0.000001)
    .all()
)

verificacoes_preenchimento = {
    "tabela_anual_nao_vazia": (
        not preenchimento_por_variavel_ano.empty
    ),
    "resumo_global_nao_vazio": (
        not resumo_preenchimento_variaveis.empty
    ),
    "variacao_nao_vazia": (
        not variacao_preenchimento.empty
    ),
    "uma_linha_por_variavel_e_ano": (
        len(
            preenchimento_por_variavel_ano
        )
        == quantidade_linhas_esperada
    ),
    "todas_variaveis_no_resumo": (
        len(
            resumo_preenchimento_variaveis
        )
        == len(
            colunas_dicionario_na_coorte
        )
    ),
    "percentuais_preenchimento_validos": (
        percentuais_preenchimento_validos
    ),
    "percentuais_ausencia_validos": (
        percentuais_ausencia_validos
    ),
    "preenchimento_e_ausencia_somam_100": (
        soma_percentuais_valida
    ),
    "arquivo_anual_criado": (
        CAMINHO_PREENCHIMENTO_ANO.exists()
    ),
    "arquivo_resumo_criado": (
        CAMINHO_RESUMO_PREENCHIMENTO.exists()
    ),
    "arquivo_variacao_criado": (
        CAMINHO_VARIACAO_PREENCHIMENTO.exists()
    ),
    "figura_variacao_criada": (
        CAMINHO_FIGURA_VARIACAO_PREENCHIMENTO.exists()
    ),
}

verificacoes_preenchimento_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_preenchimento.keys()
        ),
        "resultado": list(
            verificacoes_preenchimento.values()
        ),
    }
)

display(
    verificacoes_preenchimento_df
)

if not all(
    verificacoes_preenchimento.values()
):
    falhas_preenchimento = [
        verificacao
        for verificacao, resultado
        in verificacoes_preenchimento.items()
        if not resultado
    ]

    raise RuntimeError(
        "A análise de preenchimento terminou com falhas: "
        f"{falhas_preenchimento}"
    )

print(
    "Análise de preenchimento validada com sucesso."
)

In [ ]:

COLUNAS_SINTOMAS = [
    coluna
    for coluna in COLUNAS_SINTOMAS_PLANEJADAS
    if coluna in coorte.columns
]

COLUNAS_COMORBIDADES = [
    coluna
    for coluna in COLUNAS_COMORBIDADES_PLANEJADAS
    if coluna in coorte.columns
]

SINTOMAS_AUSENTES_NA_COORTE = [
    coluna
    for coluna in COLUNAS_SINTOMAS_PLANEJADAS
    if coluna not in coorte.columns
]

COMORBIDADES_AUSENTES_NA_COORTE = [
    coluna
    for coluna in COLUNAS_COMORBIDADES_PLANEJADAS
    if coluna not in coorte.columns
]

if not COLUNAS_SINTOMAS:
    raise ValueError(
        "Nenhuma variável estruturada de sintomas "
        "foi encontrada na coorte."
    )

if not COLUNAS_COMORBIDADES:
    raise ValueError(
        "Nenhuma variável estruturada de comorbidades "
        "foi encontrada na coorte."
    )

print(
    "Sintomas encontrados:",
    len(COLUNAS_SINTOMAS),
)

print(
    "Comorbidades encontradas:",
    len(COLUNAS_COMORBIDADES),
)

if SINTOMAS_AUSENTES_NA_COORTE:
    print(
        "Sintomas ausentes na coorte:",
        SINTOMAS_AUSENTES_NA_COORTE,
    )

if COMORBIDADES_AUSENTES_NA_COORTE:
    print(
        "Comorbidades ausentes na coorte:",
        COMORBIDADES_AUSENTES_NA_COORTE,
    )

In [ ]:
nomes_amigaveis_dicionario = (
    dicionario_operacional.set_index(
        "coluna"
    )[
        "nome_amigavel"
    ]
    .to_dict()
)

CAMINHO_SINTOMAS_ANO = (
    C.TABLES_DIR
    / "04_eda_sintomas_por_ano.csv"
)

CAMINHO_COMORBIDADES_ANO = (
    C.TABLES_DIR
    / "04_eda_comorbidades_por_ano.csv"
)

CAMINHO_CONTADORES_CLINICOS_ANO = (
    C.TABLES_DIR
    / "04_eda_contadores_clinicos_por_ano.csv"
)

CAMINHO_VARIACAO_SINTOMAS = (
    C.TABLES_DIR
    / "04_eda_variacao_sintomas.csv"
)

CAMINHO_VARIACAO_COMORBIDADES = (
    C.TABLES_DIR
    / "04_eda_variacao_comorbidades.csv"
)

CAMINHO_FIGURA_SINTOMAS = (
    C.FIGURES_DIR
    / "04_eda_sintomas_por_ano.png"
)

CAMINHO_FIGURA_COMORBIDADES = (
    C.FIGURES_DIR
    / "04_eda_comorbidades_por_ano.png"
)

print(
    "Tabela de sintomas:",
    CAMINHO_SINTOMAS_ANO,
)

print(
    "Tabela de comorbidades:",
    CAMINHO_COMORBIDADES_ANO,
)

print(
    "Tabela de contadores clínicos:",
    CAMINHO_CONTADORES_CLINICOS_ANO,
)

In [ ]:
def normalizar_resposta_clinica(
    serie: pd.Series,
) -> pd.Series:
    """Normaliza respostas estruturadas sem converter ausência em não."""

    codigos_linhas, valores_distintos = pd.factorize(
        serie.astype("string"),
        use_na_sentinel=False,
    )

    codigo = (
        pd.Series(
            valores_distintos,
            dtype="string",
        )
        .str.strip()
        .str.upper()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .replace(
            {
                "": pd.NA,
                "NAN": pd.NA,
                "NONE": pd.NA,
                "<NA>": pd.NA,
            }
        )
    )

    mapa_respostas = {
        "1": "sim",
        "SIM": "sim",
        "S": "sim",
        "2": "nao",
        "NÃO": "nao",
        "NAO": "nao",
        "N": "nao",
        "9": "ignorado",
        "IGNORADO": "ignorado",
        "I": "ignorado",
    }

    resposta = (
        codigo.map(
            mapa_respostas
        )
    )

    resposta = resposta.where(
        codigo.notna(),
        "ausente",
    )

    resposta = resposta.fillna(
        "inesperado"
    )

    return pd.Series(
        resposta.to_numpy(dtype=object)[codigos_linhas],
        index=serie.index,
        dtype="string",
    )

In [ ]:
def criar_tabela_clinica_por_ano(
    dataframe: pd.DataFrame,
    colunas: list,
    grupo_clinico: str,
    nomes_amigaveis: dict,
) -> pd.DataFrame:
    """Cria uma tabela longa de sintomas ou comorbidades por ano."""

    registros = []

    anos_observados = (
        dataframe[
            "ano_internacao"
        ]
        .dropna()
        .astype(int)
        .sort_values()
        .unique()
        .tolist()
    )

    for coluna in colunas:
        resposta_normalizada = (
            normalizar_resposta_clinica(
                dataframe[
                    coluna
                ]
            )
        )

        tabela_variavel = pd.DataFrame(
            {
                "ano_internacao": dataframe[
                    "ano_internacao"
                ],
                "resposta": resposta_normalizada,
            }
        )

        contagem = (
            tabela_variavel.groupby(
                [
                    "ano_internacao",
                    "resposta",
                ],
                dropna=False,
            )
            .size()
            .unstack(
                fill_value=0
            )
        )

        contagem = contagem.reindex(
            index=anos_observados,
            fill_value=0,
        )

        contagem = contagem.reindex(
            columns=[
                "sim",
                "nao",
                "ignorado",
                "ausente",
                "inesperado",
            ],
            fill_value=0,
        )

        for ano in anos_observados:
            quantidade_sim = int(
                contagem.loc[
                    ano,
                    "sim",
                ]
            )

            quantidade_nao = int(
                contagem.loc[
                    ano,
                    "nao",
                ]
            )

            quantidade_ignorado = int(
                contagem.loc[
                    ano,
                    "ignorado",
                ]
            )

            quantidade_ausente = int(
                contagem.loc[
                    ano,
                    "ausente",
                ]
            )

            quantidade_inesperado = int(
                contagem.loc[
                    ano,
                    "inesperado",
                ]
            )

            quantidade_total = (
                quantidade_sim
                + quantidade_nao
                + quantidade_ignorado
                + quantidade_ausente
                + quantidade_inesperado
            )

            quantidade_respostas_validas = (
                quantidade_sim
                + quantidade_nao
            )

            if quantidade_respostas_validas == 0:
                percentual_sim_validos = np.nan
                percentual_nao_validos = np.nan
            else:
                percentual_sim_validos = (
                    quantidade_sim
                    / quantidade_respostas_validas
                    * 100
                )

                percentual_nao_validos = (
                    quantidade_nao
                    / quantidade_respostas_validas
                    * 100
                )

            if quantidade_total == 0:
                percentual_sem_resposta_valida = np.nan
            else:
                percentual_sem_resposta_valida = (
                    (
                        quantidade_ignorado
                        + quantidade_ausente
                        + quantidade_inesperado
                    )
                    / quantidade_total
                    * 100
                )

            registros.append(
                {
                    "ano_internacao": int(
                        ano
                    ),
                    "grupo_clinico": grupo_clinico,
                    "coluna": coluna,
                    "nome_amigavel": (
                        nomes_amigaveis.get(
                            coluna,
                            coluna,
                        )
                    ),
                    "quantidade_total": (
                        quantidade_total
                    ),
                    "quantidade_sim": (
                        quantidade_sim
                    ),
                    "quantidade_nao": (
                        quantidade_nao
                    ),
                    "quantidade_ignorado": (
                        quantidade_ignorado
                    ),
                    "quantidade_ausente": (
                        quantidade_ausente
                    ),
                    "quantidade_inesperado": (
                        quantidade_inesperado
                    ),
                    "quantidade_respostas_validas": (
                        quantidade_respostas_validas
                    ),
                    "percentual_sim_validos": (
                        percentual_sim_validos
                    ),
                    "percentual_nao_validos": (
                        percentual_nao_validos
                    ),
                    "percentual_sem_resposta_valida": (
                        percentual_sem_resposta_valida
                    ),
                }
            )

    resultado = pd.DataFrame(
        registros
    )

    resultado = (
        resultado.sort_values(
            [
                "coluna",
                "ano_internacao",
            ]
        )
        .reset_index(drop=True)
    )

    return resultado

In [ ]:
sintomas_por_ano = (
    criar_tabela_clinica_por_ano(
        dataframe=coorte,
        colunas=COLUNAS_SINTOMAS,
        grupo_clinico="sintomas",
        nomes_amigaveis=(
            nomes_amigaveis_dicionario
        ),
    )
)

comorbidades_por_ano = (
    criar_tabela_clinica_por_ano(
        dataframe=coorte,
        colunas=COLUNAS_COMORBIDADES,
        grupo_clinico="comorbidades",
        nomes_amigaveis=(
            nomes_amigaveis_dicionario
        ),
    )
)

display(
    sintomas_por_ano.head(20)
)

display(
    comorbidades_por_ano.head(20)
)

In [ ]:

import gc

def contar_respostas_clinicas(
    dataframe: pd.DataFrame,
    colunas: list[str],
    coluna_confirmados: str,
    coluna_avaliados: str,
) -> None:
    """Calcula as quantidades de respostas confirmadas e avaliadas sem manter grandes matrizes textuais simultaneamente."""

    if not colunas:
        raise ValueError(
            "A lista de colunas clínicas está vazia."
        )

    colunas_ausentes = [
        coluna
        for coluna in colunas
        if coluna not in dataframe.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            "Não foi possível calcular os contadores clínicos. "
            "As seguintes colunas não existem na coorte: "
            f"{colunas_ausentes}"
        )

    quantidade_linhas = len(
        dataframe
    )

    confirmados = np.zeros(
        quantidade_linhas,
        dtype=np.int16,
    )

    avaliados = np.zeros(
        quantidade_linhas,
        dtype=np.int16,
    )

    for coluna in colunas:
        resposta = normalizar_resposta_clinica(
            dataframe[coluna]
        )

        mascara_confirmado = (
            resposta.eq("sim")
            .to_numpy(
                dtype=bool,
                copy=False,
            )
        )

        mascara_avaliado = (
            resposta.isin(
                [
                    "sim",
                    "nao",
                ]
            )
            .to_numpy(
                dtype=bool,
                copy=False,
            )
        )

        confirmados += (
            mascara_confirmado
            .astype(
                np.int16,
                copy=False,
            )
        )

        avaliados += (
            mascara_avaliado
            .astype(
                np.int16,
                copy=False,
            )
        )

        del resposta
        del mascara_confirmado
        del mascara_avaliado

    dataframe[coluna_confirmados] = pd.Series(
        confirmados,
        index=dataframe.index,
        dtype="Int16",
    )

    dataframe[coluna_avaliados] = pd.Series(
        avaliados,
        index=dataframe.index,
        dtype="Int16",
    )

    if not dataframe[coluna_confirmados].between(
        0,
        len(colunas),
        inclusive="both",
    ).all():
        raise RuntimeError(
            f"A coluna {coluna_confirmados!r} contém "
            "valores fora do intervalo esperado."
        )

    if not dataframe[coluna_avaliados].between(
        0,
        len(colunas),
        inclusive="both",
    ).all():
        raise RuntimeError(
            f"A coluna {coluna_avaliados!r} contém "
            "valores fora do intervalo esperado."
        )

    if (
        dataframe[coluna_confirmados]
        .gt(
            dataframe[coluna_avaliados]
        )
        .any()
    ):
        raise RuntimeError(
            "A quantidade de condições confirmadas não pode "
            "ser maior que a quantidade avaliada."
        )

    del confirmados
    del avaliados

    gc.collect()


COLUNAS_CONTADORES_INCORRETAS = [
    "n_comorbidades_confirmados",
    "n_comorbidades_avaliados",
]

colunas_incorretas_presentes = [
    coluna
    for coluna in COLUNAS_CONTADORES_INCORRETAS
    if coluna in coorte.columns
]

if colunas_incorretas_presentes:
    coorte.drop(
        columns=colunas_incorretas_presentes,
        inplace=True,
    )

    print(
        "Colunas com nomes incorretos removidas:",
        colunas_incorretas_presentes,
    )


contar_respostas_clinicas(
    dataframe=coorte,
    colunas=COLUNAS_SINTOMAS,
    coluna_confirmados="n_sintomas_confirmados",
    coluna_avaliados="n_sintomas_avaliados",
)


contar_respostas_clinicas(
    dataframe=coorte,
    colunas=COLUNAS_COMORBIDADES,
    coluna_confirmados="n_comorbidades_confirmadas",
    coluna_avaliados="n_comorbidades_avaliadas",
)


coorte["flag_multimorbidade"] = (
    coorte["n_comorbidades_confirmadas"]
    .ge(2)
    .astype("int8")
)


COLUNAS_CONTADORES_CLINICOS = [
    "n_sintomas_confirmados",
    "n_sintomas_avaliados",
    "n_comorbidades_confirmadas",
    "n_comorbidades_avaliadas",
    "flag_multimorbidade",
]

colunas_contadores_ausentes = [
    coluna
    for coluna in COLUNAS_CONTADORES_CLINICOS
    if coluna not in coorte.columns
]

if colunas_contadores_ausentes:
    raise RuntimeError(
        "Os contadores clínicos não foram criados corretamente. "
        "Colunas ausentes: "
        f"{colunas_contadores_ausentes}"
    )


print(
    "Contadores clínicos criados "
    "sem matrizes textuais persistentes."
)

print(
    "Colunas criadas:",
    COLUNAS_CONTADORES_CLINICOS,
)

display(
    coorte[
        COLUNAS_CONTADORES_CLINICOS
    ]
    .describe()
)

In [ ]:
contadores_clinicos_por_ano = (
    coorte.groupby(
        "ano_internacao",
        dropna=False,
    )
    .agg(
        quantidade_registros=(
            "ano_internacao",
            "size",
        ),
        sintomas_confirmados_media=(
            "n_sintomas_confirmados",
            "mean",
        ),
        sintomas_confirmados_mediana=(
            "n_sintomas_confirmados",
            "median",
        ),
        sintomas_avaliados_media=(
            "n_sintomas_avaliados",
            "mean",
        ),
        comorbidades_confirmadas_media=(
            "n_comorbidades_confirmadas",
            "mean",
        ),
        comorbidades_confirmadas_mediana=(
            "n_comorbidades_confirmadas",
            "median",
        ),
        comorbidades_avaliadas_media=(
            "n_comorbidades_avaliadas",
            "mean",
        ),
        quantidade_multimorbidade=(
            "flag_multimorbidade",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "ano_internacao"
    )
    .reset_index(drop=True)
)

contadores_clinicos_por_ano[
    "percentual_multimorbidade"
] = (
    contadores_clinicos_por_ano[
        "quantidade_multimorbidade"
    ]
    .div(
        contadores_clinicos_por_ano[
            "quantidade_registros"
        ]
    )
    .mul(100)
)

display(
    contadores_clinicos_por_ano
)

In [ ]:
def calcular_variacao_clinica(
    tabela_clinica: pd.DataFrame,
) -> pd.DataFrame:
    """
    Resume a variação anual da proporção de respostas sim.
    """

    resultado = (
        tabela_clinica.groupby(
            [
                "grupo_clinico",
                "coluna",
                "nome_amigavel",
            ],
            dropna=False,
        )
        .agg(
            percentual_sim_minimo=(
                "percentual_sim_validos",
                "min",
            ),
            percentual_sim_maximo=(
                "percentual_sim_validos",
                "max",
            ),
            percentual_sim_medio=(
                "percentual_sim_validos",
                "mean",
            ),
            percentual_sem_resposta_valida_maximo=(
                "percentual_sem_resposta_valida",
                "max",
            ),
            quantidade_anos_observados=(
                "ano_internacao",
                "nunique",
            ),
        )
        .reset_index()
    )

    resultado[
        "amplitude_percentual_sim"
    ] = (
        resultado[
            "percentual_sim_maximo"
        ]
        .sub(
            resultado[
                "percentual_sim_minimo"
            ]
        )
    )

    resultado[
        "flag_variacao_10_pp"
    ] = (
        resultado[
            "amplitude_percentual_sim"
        ]
        .ge(10)
    )

    resultado = (
        resultado.sort_values(
            [
                "amplitude_percentual_sim",
                "coluna",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return resultado


variacao_sintomas = (
    calcular_variacao_clinica(
        sintomas_por_ano
    )
)

variacao_comorbidades = (
    calcular_variacao_clinica(
        comorbidades_por_ano
    )
)

display(
    variacao_sintomas
)

display(
    variacao_comorbidades
)

In [ ]:
QUANTIDADE_INDICADORES_GRAFICO = 8

sintomas_grafico = (
    variacao_sintomas.head(
        QUANTIDADE_INDICADORES_GRAFICO
    )[
        [
            "coluna",
            "nome_amigavel",
        ]
    ]
    .copy()
)

comorbidades_grafico = (
    variacao_comorbidades.head(
        QUANTIDADE_INDICADORES_GRAFICO
    )[
        [
            "coluna",
            "nome_amigavel",
        ]
    ]
    .copy()
)

if sintomas_grafico.empty:
    raise ValueError(
        "Nenhum sintoma foi selecionado para o gráfico."
    )

if comorbidades_grafico.empty:
    raise ValueError(
        "Nenhuma comorbidade foi selecionada para o gráfico."
    )

display(
    sintomas_grafico
)

display(
    comorbidades_grafico
)

In [ ]:
def criar_grafico_clinico_temporal(
    tabela_clinica: pd.DataFrame,
    selecao_variaveis: pd.DataFrame,
    titulo: str,
    caminho_saida: Path,
) -> None:
    """
    Cria um gráfico de linhas com a proporção anual de resposta sim.
    """

    colunas_selecionadas = (
        selecao_variaveis[
            "coluna"
        ]
        .tolist()
    )

    dados_grafico = (
        tabela_clinica.loc[
            tabela_clinica[
                "coluna"
            ].isin(
                colunas_selecionadas
            )
        ]
        .copy()
    )

    dados_grafico[
        "rotulo"
    ] = (
        dados_grafico[
            "nome_amigavel"
        ]
        .where(
            dados_grafico[
                "nome_amigavel"
            ]
            .astype("string")
            .str.strip()
            .ne(""),
            dados_grafico[
                "coluna"
            ],
        )
    )

    figura, eixo = plt.subplots(
        figsize=(
            13,
            7,
        )
    )

    for rotulo in dados_grafico[
        "rotulo"
    ].drop_duplicates():
        serie_variavel = (
            dados_grafico.loc[
                dados_grafico[
                    "rotulo"
                ].eq(
                    rotulo
                )
            ]
            .sort_values(
                "ano_internacao"
            )
        )

        eixo.plot(
            serie_variavel[
                "ano_internacao"
            ].astype(int),
            serie_variavel[
                "percentual_sim_validos"
            ],
            marker="o",
            linewidth=2,
            label=rotulo,
        )

    anos_grafico = (
        dados_grafico[
            "ano_internacao"
        ]
        .dropna()
        .astype(int)
        .sort_values()
        .unique()
    )

    eixo.set_title(
        titulo
    )

    eixo.set_xlabel(
        "Ano da internação"
    )

    eixo.set_ylabel(
        "Resposta sim entre respostas válidas"
    )

    eixo.set_xticks(
        anos_grafico
    )

    eixo.set_ylim(
        bottom=0
    )

    eixo.yaxis.set_major_formatter(
        plt.FuncFormatter(
            lambda valor, posicao: (
                f"{valor:.0f}%"
            )
        )
    )

    eixo.legend(
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
        title="Indicador",
    )

    figura.text(
        0.01,
        0.01,
        (
            "Denominador de cada indicador: respostas sim ou não. "
            "Ignorado, ausente e código inesperado não entram "
            "no denominador da proporção."
        ),
        fontsize=8,
    )

    figura.tight_layout(
        rect=[
            0,
            0.04,
            0.82,
            1,
        ]
    )

    figura.savefig(
        caminho_saida,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Figura salva em:",
        caminho_saida,
    )

In [ ]:
criar_grafico_clinico_temporal(
    tabela_clinica=sintomas_por_ano,
    selecao_variaveis=sintomas_grafico,
    titulo=(
        "Sintomas com maior variação temporal "
        "entre os registros com resposta válida"
    ),
    caminho_saida=(
        CAMINHO_FIGURA_SINTOMAS
    ),
)

criar_grafico_clinico_temporal(
    tabela_clinica=comorbidades_por_ano,
    selecao_variaveis=(
        comorbidades_grafico
    ),
    titulo=(
        "Comorbidades com maior variação temporal "
        "entre os registros com resposta válida"
    ),
    caminho_saida=(
        CAMINHO_FIGURA_COMORBIDADES
    ),
)

In [ ]:
sintomas_por_ano.to_csv(
    CAMINHO_SINTOMAS_ANO,
    index=False,
    encoding="utf-8-sig",
)

comorbidades_por_ano.to_csv(
    CAMINHO_COMORBIDADES_ANO,
    index=False,
    encoding="utf-8-sig",
)

contadores_clinicos_por_ano.to_csv(
    CAMINHO_CONTADORES_CLINICOS_ANO,
    index=False,
    encoding="utf-8-sig",
)

variacao_sintomas.to_csv(
    CAMINHO_VARIACAO_SINTOMAS,
    index=False,
    encoding="utf-8-sig",
)

variacao_comorbidades.to_csv(
    CAMINHO_VARIACAO_COMORBIDADES,
    index=False,
    encoding="utf-8-sig",
)

anos_clinicos_observados = (
    coorte[
        "ano_internacao"
    ]
    .nunique(
        dropna=True
    )
)

linhas_sintomas_esperadas = (
    len(
        COLUNAS_SINTOMAS
    )
    * anos_clinicos_observados
)

linhas_comorbidades_esperadas = (
    len(
        COLUNAS_COMORBIDADES
    )
    * anos_clinicos_observados
)

totais_sintomas_validos = (
    sintomas_por_ano[
        [
            "quantidade_sim",
            "quantidade_nao",
            "quantidade_ignorado",
            "quantidade_ausente",
            "quantidade_inesperado",
        ]
    ]
    .sum(
        axis=1
    )
    .eq(
        sintomas_por_ano[
            "quantidade_total"
        ]
    )
    .all()
)

totais_comorbidades_validos = (
    comorbidades_por_ano[
        [
            "quantidade_sim",
            "quantidade_nao",
            "quantidade_ignorado",
            "quantidade_ausente",
            "quantidade_inesperado",
        ]
    ]
    .sum(
        axis=1
    )
    .eq(
        comorbidades_por_ano[
            "quantidade_total"
        ]
    )
    .all()
)

percentuais_sintomas_validos = (
    sintomas_por_ano[
        "percentual_sim_validos"
    ]
    .dropna()
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

percentuais_comorbidades_validos = (
    comorbidades_por_ano[
        "percentual_sim_validos"
    ]
    .dropna()
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

verificacoes_clinicas = {
    "sintomas_por_ano_nao_vazio": (
        not sintomas_por_ano.empty
    ),
    "comorbidades_por_ano_nao_vazio": (
        not comorbidades_por_ano.empty
    ),
    "contadores_por_ano_nao_vazio": (
        not contadores_clinicos_por_ano.empty
    ),
    "uma_linha_por_sintoma_e_ano": (
        len(
            sintomas_por_ano
        )
        == linhas_sintomas_esperadas
    ),
    "uma_linha_por_comorbidade_e_ano": (
        len(
            comorbidades_por_ano
        )
        == linhas_comorbidades_esperadas
    ),
    "categorias_sintomas_somam_total": (
        totais_sintomas_validos
    ),
    "categorias_comorbidades_somam_total": (
        totais_comorbidades_validos
    ),
    "percentuais_sintomas_validos": (
        percentuais_sintomas_validos
    ),
    "percentuais_comorbidades_validos": (
        percentuais_comorbidades_validos
    ),
    "contagem_sintomas_dentro_limite": (
        coorte[
            "n_sintomas_confirmados"
        ]
        .between(
            0,
            len(
                COLUNAS_SINTOMAS
            ),
            inclusive="both",
        )
        .all()
    ),
    "contagem_comorbidades_dentro_limite": (
        coorte[
            "n_comorbidades_confirmadas"
        ]
        .between(
            0,
            len(
                COLUNAS_COMORBIDADES
            ),
            inclusive="both",
        )
        .all()
    ),
    "arquivo_sintomas_criado": (
        CAMINHO_SINTOMAS_ANO.exists()
    ),
    "arquivo_comorbidades_criado": (
        CAMINHO_COMORBIDADES_ANO.exists()
    ),
    "arquivo_contadores_criado": (
        CAMINHO_CONTADORES_CLINICOS_ANO.exists()
    ),
    "arquivo_variacao_sintomas_criado": (
        CAMINHO_VARIACAO_SINTOMAS.exists()
    ),
    "arquivo_variacao_comorbidades_criado": (
        CAMINHO_VARIACAO_COMORBIDADES.exists()
    ),
    "figura_sintomas_criada": (
        CAMINHO_FIGURA_SINTOMAS.exists()
    ),
    "figura_comorbidades_criada": (
        CAMINHO_FIGURA_COMORBIDADES.exists()
    ),
}

verificacoes_clinicas_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_clinicas.keys()
        ),
        "resultado": list(
            verificacoes_clinicas.values()
        ),
    }
)

display(
    verificacoes_clinicas_df
)

if not all(
    verificacoes_clinicas.values()
):
    falhas_clinicas = [
        verificacao
        for verificacao, resultado
        in verificacoes_clinicas.items()
        if not resultado
    ]

    raise RuntimeError(
        "A análise de sintomas e comorbidades "
        "terminou com falhas: "
        f"{falhas_clinicas}"
    )

print(
    "Análise de sintomas e comorbidades validada com sucesso."
)

In [ ]:
COLUNAS_VACINACAO_PLANEJADAS = [
    "vacina",
    "vacina_cov",
]

COLUNAS_VACINACAO = [
    coluna
    for coluna in COLUNAS_VACINACAO_PLANEJADAS
    if coluna in coorte.columns
]

COLUNA_ESCOLARIDADE = "cs_escol_n"
COLUNA_UF_RESIDENCIA = "sg_uf"

if not COLUNAS_VACINACAO:
    raise ValueError(
        "Nenhum indicador geral de vacinação "
        "foi encontrado na coorte."
    )

if COLUNA_ESCOLARIDADE not in coorte.columns:
    raise KeyError(
        "A coluna cs_escol_n não está disponível na coorte."
    )

if COLUNA_UF_RESIDENCIA not in coorte.columns:
    raise KeyError(
        "A coluna sg_uf não está disponível na coorte."
    )

CAMINHO_VACINACAO_ANO = (
    C.TABLES_DIR
    / "04_eda_vacinacao_por_ano.csv"
)

CAMINHO_ESCOLARIDADE_ANO = (
    C.TABLES_DIR
    / "04_eda_escolaridade_por_ano.csv"
)

CAMINHO_UF_RESIDENCIA_ANO = (
    C.TABLES_DIR
    / "04_eda_uf_residencia_por_ano.csv"
)

CAMINHO_RESUMO_UF = (
    C.TABLES_DIR
    / "04_eda_resumo_uf_residencia.csv"
)

CAMINHO_FIGURA_VACINACAO = (
    C.FIGURES_DIR
    / "04_eda_vacinacao_por_ano.png"
)

CAMINHO_FIGURA_ESCOLARIDADE = (
    C.FIGURES_DIR
    / "04_eda_escolaridade_preenchimento_por_ano.png"
)

CAMINHO_FIGURA_UF = (
    C.FIGURES_DIR
    / "04_eda_principais_uf_residencia.png"
)

print(
    "Indicadores de vacinação encontrados:",
    COLUNAS_VACINACAO,
)

print(
    "Tabela de vacinação:",
    CAMINHO_VACINACAO_ANO,
)

print(
    "Tabela de escolaridade:",
    CAMINHO_ESCOLARIDADE_ANO,
)

print(
    "Tabela de UF de residência:",
    CAMINHO_UF_RESIDENCIA_ANO,
)

In [ ]:
vacinacao_por_ano = (
    criar_tabela_clinica_por_ano(
        dataframe=coorte,
        colunas=COLUNAS_VACINACAO,
        grupo_clinico="vacinacao",
        nomes_amigaveis=(
            nomes_amigaveis_dicionario
        ),
    )
)

vacinacao_por_ano = (
    vacinacao_por_ano.sort_values(
        [
            "coluna",
            "ano_internacao",
        ]
    )
    .reset_index(drop=True)
)

display(
    vacinacao_por_ano
)

In [ ]:
escolaridade_normalizada = (
    coorte[
        COLUNA_ESCOLARIDADE
    ]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True,
    )
    .replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "NONE": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        }
    )
)

escolaridade_categoria = (
    escolaridade_normalizada
    .fillna(
        "ausente"
    )
)

escolaridade_por_ano = (
    pd.DataFrame(
        {
            "ano_internacao": coorte[
                "ano_internacao"
            ],
            "categoria_escolaridade": (
                escolaridade_categoria
            ),
        }
    )
    .groupby(
        [
            "ano_internacao",
            "categoria_escolaridade",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "quantidade"
    )
    .reset_index()
)

denominador_escolaridade_ano = (
    escolaridade_por_ano.groupby(
        "ano_internacao"
    )[
        "quantidade"
    ]
    .transform("sum")
)

escolaridade_por_ano[
    "percentual_ano"
] = (
    escolaridade_por_ano[
        "quantidade"
    ]
    .div(
        denominador_escolaridade_ano
    )
    .mul(100)
)

escolaridade_por_ano[
    "flag_ausente"
] = (
    escolaridade_por_ano[
        "categoria_escolaridade"
    ]
    .eq("ausente")
)

escolaridade_por_ano = (
    escolaridade_por_ano
    .sort_values(
        [
            "ano_internacao",
            "categoria_escolaridade",
        ]
    )
    .reset_index(drop=True)
)

display(
    escolaridade_por_ano.head(40)
)

In [ ]:
escolaridade_preenchimento_ano = (
    pd.DataFrame(
        {
            "ano_internacao": coorte[
                "ano_internacao"
            ],
            "escolaridade_preenchida": (
                escolaridade_normalizada
                .notna()
                .astype("int8")
            ),
        }
    )
    .groupby(
        "ano_internacao",
        dropna=False,
    )
    .agg(
        quantidade_registros=(
            "escolaridade_preenchida",
            "size",
        ),
        quantidade_preenchida=(
            "escolaridade_preenchida",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "ano_internacao"
    )
    .reset_index(drop=True)
)

escolaridade_preenchimento_ano[
    "quantidade_ausente"
] = (
    escolaridade_preenchimento_ano[
        "quantidade_registros"
    ]
    .sub(
        escolaridade_preenchimento_ano[
            "quantidade_preenchida"
        ]
    )
)

escolaridade_preenchimento_ano[
    "percentual_preenchido"
] = (
    escolaridade_preenchimento_ano[
        "quantidade_preenchida"
    ]
    .div(
        escolaridade_preenchimento_ano[
            "quantidade_registros"
        ]
    )
    .mul(100)
)

escolaridade_preenchimento_ano[
    "percentual_ausente"
] = (
    100
    - escolaridade_preenchimento_ano[
        "percentual_preenchido"
    ]
)

display(
    escolaridade_preenchimento_ano
)

In [ ]:
uf_residencia_normalizada = (
    coorte[
        COLUNA_UF_RESIDENCIA
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace(
        {
            "": pd.NA,
            "NAN": pd.NA,
            "NONE": pd.NA,
            "<NA>": pd.NA,
        }
    )
)

SIGLAS_UF_VALIDAS = {
    "AC",
    "AL",
    "AP",
    "AM",
    "BA",
    "CE",
    "DF",
    "ES",
    "GO",
    "MA",
    "MT",
    "MS",
    "MG",
    "PA",
    "PB",
    "PR",
    "PE",
    "PI",
    "RJ",
    "RN",
    "RS",
    "RO",
    "RR",
    "SC",
    "SP",
    "SE",
    "TO",
}

coorte[
    "uf_residencia_categoria"
] = (
    uf_residencia_normalizada
    .where(
        uf_residencia_normalizada.isin(
            SIGLAS_UF_VALIDAS
        ),
        "Ausente ou inesperada",
    )
)

uf_residencia_por_ano = (
    coorte.groupby(
        [
            "ano_internacao",
            "uf_residencia_categoria",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "quantidade"
    )
    .reset_index()
)

denominador_uf_ano = (
    uf_residencia_por_ano.groupby(
        "ano_internacao"
    )[
        "quantidade"
    ]
    .transform("sum")
)

uf_residencia_por_ano[
    "percentual_ano"
] = (
    uf_residencia_por_ano[
        "quantidade"
    ]
    .div(
        denominador_uf_ano
    )
    .mul(100)
)

uf_residencia_por_ano = (
    uf_residencia_por_ano
    .sort_values(
        [
            "ano_internacao",
            "quantidade",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    uf_residencia_por_ano.head(40)
)

In [ ]:
resumo_uf_residencia = (
    coorte.groupby(
        "uf_residencia_categoria",
        dropna=False,
    )
    .size()
    .rename(
        "quantidade"
    )
    .reset_index()
    .sort_values(
        "quantidade",
        ascending=False,
    )
    .reset_index(drop=True)
)

resumo_uf_residencia[
    "percentual_coorte"
] = (
    resumo_uf_residencia[
        "quantidade"
    ]
    .div(
        len(
            coorte
        )
    )
    .mul(100)
)

resumo_uf_residencia[
    "ordem_frequencia"
] = (
    resumo_uf_residencia.index
    + 1
)

display(
    resumo_uf_residencia
)

In [ ]:
figura_vacinacao, eixo_vacinacao = plt.subplots(
    figsize=(
        11,
        6,
    )
)

for coluna in COLUNAS_VACINACAO:
    dados_variavel = (
        vacinacao_por_ano.loc[
            vacinacao_por_ano[
                "coluna"
            ].eq(
                coluna
            )
        ]
        .sort_values(
            "ano_internacao"
        )
    )

    nome_variavel = (
        nomes_amigaveis_dicionario.get(
            coluna,
            coluna,
        )
    )

    eixo_vacinacao.plot(
        dados_variavel[
            "ano_internacao"
        ].astype(int),
        dados_variavel[
            "percentual_sim_validos"
        ],
        marker="o",
        linewidth=2,
        label=nome_variavel,
    )

anos_vacinacao = (
    vacinacao_por_ano[
        "ano_internacao"
    ]
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
)

eixo_vacinacao.set_title(
    "Registro de vacinação entre respostas válidas por ano de internação"
)

eixo_vacinacao.set_xlabel(
    "Ano da internação"
)

eixo_vacinacao.set_ylabel(
    "Resposta sim entre respostas válidas"
)

eixo_vacinacao.set_xticks(
    anos_vacinacao
)

eixo_vacinacao.set_ylim(
    bottom=0
)

eixo_vacinacao.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

eixo_vacinacao.legend(
    title="Indicador"
)

figura_vacinacao.text(
    0.01,
    0.01,
    (
        "Denominador de cada indicador: respostas sim ou não. "
        "Ausentes, ignorados e códigos inesperados não entram "
        "na proporção."
    ),
    fontsize=8,
)

figura_vacinacao.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1,
    ]
)

figura_vacinacao.savefig(
    CAMINHO_FIGURA_VACINACAO,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_VACINACAO,
)

In [ ]:
figura_escolaridade, eixo_escolaridade = plt.subplots(
    figsize=(
        11,
        6,
    )
)

anos_escolaridade = (
    escolaridade_preenchimento_ano[
        "ano_internacao"
    ]
    .astype(int)
)

percentuais_escolaridade = (
    escolaridade_preenchimento_ano[
        "percentual_preenchido"
    ]
)

eixo_escolaridade.plot(
    anos_escolaridade,
    percentuais_escolaridade,
    color="tab:green",
    marker="o",
    linewidth=2,
)

eixo_escolaridade.set_title(
    "Preenchimento da escolaridade por ano de internação"
)

eixo_escolaridade.set_xlabel(
    "Ano da internação"
)

eixo_escolaridade.set_ylabel(
    "Registros com escolaridade preenchida"
)

eixo_escolaridade.set_xticks(
    anos_escolaridade
)

eixo_escolaridade.set_ylim(
    0,
    100,
)

eixo_escolaridade.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

for ano, percentual in zip(
    anos_escolaridade,
    percentuais_escolaridade,
):
    eixo_escolaridade.annotate(
        f"{percentual:.1f}%",
        (
            ano,
            percentual,
        ),
        textcoords="offset points",
        xytext=(
            0,
            8,
        ),
        ha="center",
        fontsize=8,
    )

figura_escolaridade.text(
    0.01,
    0.01,
    (
        "Preenchimento técnico: valor não nulo. "
        "Categorias ignoradas permanecem como categorias registradas."
    ),
    fontsize=8,
)

figura_escolaridade.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1,
    ]
)

figura_escolaridade.savefig(
    CAMINHO_FIGURA_ESCOLARIDADE,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_ESCOLARIDADE,
)

In [ ]:
QUANTIDADE_UF_GRAFICO = 10

ufs_validas_resumo = (
    resumo_uf_residencia.loc[
        resumo_uf_residencia[
            "uf_residencia_categoria"
        ].ne(
            "Ausente ou inesperada"
        )
    ]
    .head(
        QUANTIDADE_UF_GRAFICO
    )
    .copy()
)

if ufs_validas_resumo.empty:
    raise ValueError(
        "Nenhuma UF válida foi encontrada para o gráfico."
    )

ufs_grafico = (
    ufs_validas_resumo[
        "uf_residencia_categoria"
    ]
    .tolist()
)

dados_uf_grafico = (
    uf_residencia_por_ano.loc[
        uf_residencia_por_ano[
            "uf_residencia_categoria"
        ].isin(
            ufs_grafico
        )
    ]
    .copy()
)

tabela_uf_grafico = (
    dados_uf_grafico.pivot(
        index="ano_internacao",
        columns="uf_residencia_categoria",
        values="percentual_ano",
    )
    .reindex(
        columns=ufs_grafico,
        fill_value=0,
    )
    .fillna(0)
)

figura_uf, eixo_uf = plt.subplots(
    figsize=(
        13,
        7,
    )
)

tabela_uf_grafico.plot(
    kind="line",
    marker="o",
    linewidth=2,
    ax=eixo_uf,
)

eixo_uf.set_title(
    "Participação das UFs de residência mais frequentes por ano"
)

eixo_uf.set_xlabel(
    "Ano da internação"
)

eixo_uf.set_ylabel(
    "Percentual dos registros do ano"
)

eixo_uf.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

eixo_uf.legend(
    title="UF de residência",
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)

figura_uf.text(
    0.01,
    0.01,
    (
        "Foram exibidas as dez UFs com maior volume global "
        "na coorte carregada."
    ),
    fontsize=8,
)

figura_uf.tight_layout(
    rect=[
        0,
        0.04,
        0.86,
        1,
    ]
)

figura_uf.savefig(
    CAMINHO_FIGURA_UF,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_UF,
)

In [ ]:
vacinacao_por_ano.to_csv(
    CAMINHO_VACINACAO_ANO,
    index=False,
    encoding="utf-8-sig",
)

escolaridade_por_ano.to_csv(
    CAMINHO_ESCOLARIDADE_ANO,
    index=False,
    encoding="utf-8-sig",
)

uf_residencia_por_ano.to_csv(
    CAMINHO_UF_RESIDENCIA_ANO,
    index=False,
    encoding="utf-8-sig",
)

resumo_uf_residencia.to_csv(
    CAMINHO_RESUMO_UF,
    index=False,
    encoding="utf-8-sig",
)

total_escolaridade = int(
    escolaridade_por_ano[
        "quantidade"
    ]
    .sum()
)

total_uf = int(
    uf_residencia_por_ano[
        "quantidade"
    ]
    .sum()
)

anos_contexto_observados = int(
    coorte[
        "ano_internacao"
    ]
    .nunique(
        dropna=True
    )
)

linhas_vacinacao_esperadas = (
    len(
        COLUNAS_VACINACAO
    )
    * anos_contexto_observados
)

categorias_vacinacao_somam_total = (
    vacinacao_por_ano[
        [
            "quantidade_sim",
            "quantidade_nao",
            "quantidade_ignorado",
            "quantidade_ausente",
            "quantidade_inesperado",
        ]
    ]
    .sum(
        axis=1
    )
    .eq(
        vacinacao_por_ano[
            "quantidade_total"
        ]
    )
    .all()
)

percentuais_vacinacao_validos = (
    vacinacao_por_ano[
        "percentual_sim_validos"
    ]
    .dropna()
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

percentuais_escolaridade_validos = (
    escolaridade_por_ano[
        "percentual_ano"
    ]
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

percentuais_uf_validos = (
    uf_residencia_por_ano[
        "percentual_ano"
    ]
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

verificacoes_contexto = {
    "vacinacao_por_ano_nao_vazia": (
        not vacinacao_por_ano.empty
    ),
    "escolaridade_por_ano_nao_vazia": (
        not escolaridade_por_ano.empty
    ),
    "uf_por_ano_nao_vazia": (
        not uf_residencia_por_ano.empty
    ),
    "uma_linha_por_vacina_e_ano": (
        len(
            vacinacao_por_ano
        )
        == linhas_vacinacao_esperadas
    ),
    "categorias_vacinacao_somam_total": (
        categorias_vacinacao_somam_total
    ),
    "total_escolaridade_igual_coorte": (
        total_escolaridade
        == len(
            coorte
        )
    ),
    "total_uf_igual_coorte": (
        total_uf
        == len(
            coorte
        )
    ),
    "percentuais_vacinacao_validos": (
        percentuais_vacinacao_validos
    ),
    "percentuais_escolaridade_validos": (
        percentuais_escolaridade_validos
    ),
    "percentuais_uf_validos": (
        percentuais_uf_validos
    ),
    "arquivo_vacinacao_criado": (
        CAMINHO_VACINACAO_ANO.exists()
    ),
    "arquivo_escolaridade_criado": (
        CAMINHO_ESCOLARIDADE_ANO.exists()
    ),
    "arquivo_uf_criado": (
        CAMINHO_UF_RESIDENCIA_ANO.exists()
    ),
    "arquivo_resumo_uf_criado": (
        CAMINHO_RESUMO_UF.exists()
    ),
    "figura_vacinacao_criada": (
        CAMINHO_FIGURA_VACINACAO.exists()
    ),
    "figura_escolaridade_criada": (
        CAMINHO_FIGURA_ESCOLARIDADE.exists()
    ),
    "figura_uf_criada": (
        CAMINHO_FIGURA_UF.exists()
    ),
}

verificacoes_contexto_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_contexto.keys()
        ),
        "resultado": list(
            verificacoes_contexto.values()
        ),
    }
)

display(
    verificacoes_contexto_df
)

if not all(
    verificacoes_contexto.values()
):
    falhas_contexto = [
        verificacao
        for verificacao, resultado
        in verificacoes_contexto.items()
        if not resultado
    ]

    raise RuntimeError(
        "A análise de vacinação, escolaridade e geografia "
        "terminou com falhas: "
        f"{falhas_contexto}"
    )

print(
    "Análise de vacinação, escolaridade e geografia "
    "validada com sucesso."
)

In [ ]:
ANO_INICIAL_TRIAGEM_TEMPORAL = 2019
ANO_FINAL_TRIAGEM_TEMPORAL = 2025

ANOS_TRIAGEM_TEMPORAL = list(
    range(
        ANO_INICIAL_TRIAGEM_TEMPORAL,
        ANO_FINAL_TRIAGEM_TEMPORAL + 1,
    )
)

CAMINHO_RUPTURAS_TEMPORAIS = (
    C.TABLES_DIR
    / "04_eda_rupturas_temporais.csv"
)

CAMINHO_RESUMO_RUPTURAS = (
    C.TABLES_DIR
    / "04_eda_resumo_rupturas_temporais.csv"
)

CAMINHO_FIGURA_RUPTURAS = (
    C.FIGURES_DIR
    / "04_eda_rupturas_temporais.png"
)

print(
    "Período utilizado na triagem temporal:",
    ANO_INICIAL_TRIAGEM_TEMPORAL,
    "a",
    ANO_FINAL_TRIAGEM_TEMPORAL,
)

print(
    "Tabela de rupturas:",
    CAMINHO_RUPTURAS_TEMPORAIS,
)

print(
    "Resumo de rupturas:",
    CAMINHO_RESUMO_RUPTURAS,
)

In [ ]:
def resumir_amplitude_temporal(
    dataframe: pd.DataFrame,
    coluna_indicador: str,
    coluna_valor: str,
    tipo_indicador: str,
    grupo_indicador: str,
    unidade: str,
    limite_destaque: float,
) -> pd.DataFrame:
    """Calcula mínimo, máximo e amplitude do indicador entre 2019 e 2025."""

    COLUNAS_OBRIGATORIAS = [
        "ano_internacao",
        coluna_indicador,
        coluna_valor,
    ]

    colunas_ausentes = [
        coluna
        for coluna in COLUNAS_OBRIGATORIAS
        if coluna not in dataframe.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            "A tabela não contém as colunas necessárias: "
            f"{colunas_ausentes}"
        )

    dados_periodo = (
        dataframe.loc[
            dataframe[
                "ano_internacao"
            ].isin(
                ANOS_TRIAGEM_TEMPORAL
            ),
            COLUNAS_OBRIGATORIAS,
        ]
        .copy()
    )

    dados_periodo[
        coluna_valor
    ] = pd.to_numeric(
        dados_periodo[
            coluna_valor
        ],
        errors="coerce",
    )

    dados_periodo = (
        dados_periodo.dropna(
            subset=[
                coluna_indicador,
                coluna_valor,
            ]
        )
    )

    if dados_periodo.empty:
        return pd.DataFrame(
            columns=[
                "tipo_indicador",
                "grupo_indicador",
                "indicador",
                "unidade",
                "ano_minimo",
                "valor_minimo",
                "ano_maximo",
                "valor_maximo",
                "amplitude",
                "limite_destaque",
                "flag_destaque",
                "quantidade_anos",
            ]
        )

    registros_resumo = []

    indicadores = (
        dados_periodo[
            coluna_indicador
        ]
        .drop_duplicates()
        .tolist()
    )

    for indicador in indicadores:
        dados_indicador = (
            dados_periodo.loc[
                dados_periodo[
                    coluna_indicador
                ].eq(
                    indicador
                )
            ]
            .sort_values(
                "ano_internacao"
            )
            .copy()
        )

        if dados_indicador.empty:
            continue

        indice_minimo = (
            dados_indicador[
                coluna_valor
            ]
            .idxmin()
        )

        indice_maximo = (
            dados_indicador[
                coluna_valor
            ]
            .idxmax()
        )

        valor_minimo = float(
            dados_indicador.loc[
                indice_minimo,
                coluna_valor,
            ]
        )

        valor_maximo = float(
            dados_indicador.loc[
                indice_maximo,
                coluna_valor,
            ]
        )

        ano_minimo = int(
            dados_indicador.loc[
                indice_minimo,
                "ano_internacao",
            ]
        )

        ano_maximo = int(
            dados_indicador.loc[
                indice_maximo,
                "ano_internacao",
            ]
        )

        amplitude = (
            valor_maximo
            - valor_minimo
        )

        registros_resumo.append(
            {
                "tipo_indicador": tipo_indicador,
                "grupo_indicador": grupo_indicador,
                "indicador": str(
                    indicador
                ),
                "unidade": unidade,
                "ano_minimo": ano_minimo,
                "valor_minimo": valor_minimo,
                "ano_maximo": ano_maximo,
                "valor_maximo": valor_maximo,
                "amplitude": amplitude,
                "limite_destaque": (
                    limite_destaque
                ),
                "flag_destaque": (
                    amplitude
                    >= limite_destaque
                ),
                "quantidade_anos": int(
                    dados_indicador[
                        "ano_internacao"
                    ]
                    .nunique()
                ),
            }
        )

    resultado = pd.DataFrame(
        registros_resumo
    )

    resultado = (
        resultado.sort_values(
            [
                "flag_destaque",
                "amplitude",
                "indicador",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return resultado

In [ ]:
volume_triagem = (
    resumo_temporal.loc[
        resumo_temporal[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "quantidade_registros",
        ],
    ]
    .copy()
)

total_volume_triagem = int(
    volume_triagem[
        "quantidade_registros"
    ]
    .sum()
)

if total_volume_triagem == 0:
    raise ValueError(
        "O período de triagem temporal não possui registros."
    )

volume_triagem[
    "percentual_periodo"
] = (
    volume_triagem[
        "quantidade_registros"
    ]
    .div(
        total_volume_triagem
    )
    .mul(100)
)

volume_triagem[
    "indicador"
] = (
    "Participação no volume do período"
)

desfecho_triagem = (
    alvo_por_ano.loc[
        alvo_por_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "percentual_obito_srag",
        ],
    ]
    .copy()
)

desfecho_triagem[
    "indicador"
] = (
    "Óbito por SRAG"
)

idade_triagem = (
    idade_por_ano.loc[
        idade_por_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "idade_mediana",
        ],
    ]
    .copy()
)

idade_triagem[
    "indicador"
] = (
    "Idade mediana"
)

display(
    volume_triagem
)

display(
    desfecho_triagem
)

display(
    idade_triagem
)

In [ ]:
ruptura_volume = (
    resumir_amplitude_temporal(
        dataframe=volume_triagem,
        coluna_indicador="indicador",
        coluna_valor="percentual_periodo",
        tipo_indicador="composicao",
        grupo_indicador="volume",
        unidade="pontos_percentuais",
        limite_destaque=5.0,
    )
)

ruptura_desfecho = (
    resumir_amplitude_temporal(
        dataframe=desfecho_triagem,
        coluna_indicador="indicador",
        coluna_valor=(
            "percentual_obito_srag"
        ),
        tipo_indicador="desfecho",
        grupo_indicador="desfecho",
        unidade="pontos_percentuais",
        limite_destaque=5.0,
    )
)

ruptura_idade = (
    resumir_amplitude_temporal(
        dataframe=idade_triagem,
        coluna_indicador="indicador",
        coluna_valor="idade_mediana",
        tipo_indicador="demografia",
        grupo_indicador="idade",
        unidade="anos",
        limite_destaque=5.0,
    )
)

display(
    ruptura_volume
)

display(
    ruptura_desfecho
)

display(
    ruptura_idade
)

In [ ]:
preenchimento_triagem = (
    preenchimento_por_variavel_ano.loc[
        preenchimento_por_variavel_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "coluna",
            "nome_amigavel",
            "grupo",
            "percentual_preenchido",
        ],
    ]
    .copy()
)

preenchimento_triagem[
    "indicador"
] = (
    preenchimento_triagem[
        "nome_amigavel"
    ]
    .where(
        preenchimento_triagem[
            "nome_amigavel"
        ]
        .astype("string")
        .str.strip()
        .ne(""),
        preenchimento_triagem[
            "coluna"
        ],
    )
)

ruptura_preenchimento = (
    resumir_amplitude_temporal(
        dataframe=preenchimento_triagem,
        coluna_indicador="indicador",
        coluna_valor=(
            "percentual_preenchido"
        ),
        tipo_indicador="qualidade_dados",
        grupo_indicador="preenchimento",
        unidade="pontos_percentuais",
        limite_destaque=20.0,
    )
)

display(
    ruptura_preenchimento.head(30)
)

In [ ]:
sintomas_triagem = (
    sintomas_por_ano.loc[
        sintomas_por_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "coluna",
            "nome_amigavel",
            "percentual_sim_validos",
        ],
    ]
    .copy()
)

sintomas_triagem[
    "indicador"
] = (
    sintomas_triagem[
        "nome_amigavel"
    ]
    .where(
        sintomas_triagem[
            "nome_amigavel"
        ]
        .astype("string")
        .str.strip()
        .ne(""),
        sintomas_triagem[
            "coluna"
        ],
    )
)

comorbidades_triagem = (
    comorbidades_por_ano.loc[
        comorbidades_por_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "coluna",
            "nome_amigavel",
            "percentual_sim_validos",
        ],
    ]
    .copy()
)

comorbidades_triagem[
    "indicador"
] = (
    comorbidades_triagem[
        "nome_amigavel"
    ]
    .where(
        comorbidades_triagem[
            "nome_amigavel"
        ]
        .astype("string")
        .str.strip()
        .ne(""),
        comorbidades_triagem[
            "coluna"
        ],
    )
)

ruptura_sintomas = (
    resumir_amplitude_temporal(
        dataframe=sintomas_triagem,
        coluna_indicador="indicador",
        coluna_valor=(
            "percentual_sim_validos"
        ),
        tipo_indicador="clinico",
        grupo_indicador="sintomas",
        unidade="pontos_percentuais",
        limite_destaque=10.0,
    )
)

ruptura_comorbidades = (
    resumir_amplitude_temporal(
        dataframe=comorbidades_triagem,
        coluna_indicador="indicador",
        coluna_valor=(
            "percentual_sim_validos"
        ),
        tipo_indicador="clinico",
        grupo_indicador="comorbidades",
        unidade="pontos_percentuais",
        limite_destaque=10.0,
    )
)

display(
    ruptura_sintomas
)

display(
    ruptura_comorbidades
)

In [ ]:
vacinacao_triagem = (
    vacinacao_por_ano.loc[
        vacinacao_por_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "coluna",
            "nome_amigavel",
            "percentual_sim_validos",
        ],
    ]
    .copy()
)

vacinacao_triagem[
    "indicador"
] = (
    vacinacao_triagem[
        "nome_amigavel"
    ]
    .where(
        vacinacao_triagem[
            "nome_amigavel"
        ]
        .astype("string")
        .str.strip()
        .ne(""),
        vacinacao_triagem[
            "coluna"
        ],
    )
)

escolaridade_triagem = (
    escolaridade_preenchimento_ano.loc[
        escolaridade_preenchimento_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        ),
        [
            "ano_internacao",
            "percentual_preenchido",
        ],
    ]
    .copy()
)

escolaridade_triagem[
    "indicador"
] = (
    "Preenchimento da escolaridade"
)

ruptura_vacinacao = (
    resumir_amplitude_temporal(
        dataframe=vacinacao_triagem,
        coluna_indicador="indicador",
        coluna_valor=(
            "percentual_sim_validos"
        ),
        tipo_indicador="contexto",
        grupo_indicador="vacinacao",
        unidade="pontos_percentuais",
        limite_destaque=10.0,
    )
)

ruptura_escolaridade = (
    resumir_amplitude_temporal(
        dataframe=escolaridade_triagem,
        coluna_indicador="indicador",
        coluna_valor=(
            "percentual_preenchido"
        ),
        tipo_indicador="qualidade_dados",
        grupo_indicador="escolaridade",
        unidade="pontos_percentuais",
        limite_destaque=20.0,
    )
)

display(
    ruptura_vacinacao
)

display(
    ruptura_escolaridade
)

In [ ]:
ufs_triagem = (
    resumo_uf_residencia.loc[
        resumo_uf_residencia[
            "uf_residencia_categoria"
        ].ne(
            "Ausente ou inesperada"
        )
    ]
    .head(10)[
        "uf_residencia_categoria"
    ]
    .tolist()
)

uf_triagem = (
    uf_residencia_por_ano.loc[
        uf_residencia_por_ano[
            "ano_internacao"
        ].isin(
            ANOS_TRIAGEM_TEMPORAL
        )
        & uf_residencia_por_ano[
            "uf_residencia_categoria"
        ].isin(
            ufs_triagem
        ),
        [
            "ano_internacao",
            "uf_residencia_categoria",
            "percentual_ano",
        ],
    ]
    .copy()
)

uf_triagem = (
    pd.MultiIndex.from_product(
        [
            ANOS_TRIAGEM_TEMPORAL,
            ufs_triagem,
        ],
        names=[
            "ano_internacao",
            "uf_residencia_categoria",
        ],
    )
    .to_frame(
        index=False
    )
    .merge(
        uf_triagem,
        on=[
            "ano_internacao",
            "uf_residencia_categoria",
        ],
        how="left",
        validate="one_to_one",
    )
)

uf_triagem[
    "percentual_ano"
] = (
    uf_triagem[
        "percentual_ano"
    ]
    .fillna(0)
)

uf_triagem[
    "indicador"
] = (
    "UF "
    + uf_triagem[
        "uf_residencia_categoria"
    ]
    .astype("string")
)

ruptura_uf = (
    resumir_amplitude_temporal(
        dataframe=uf_triagem,
        coluna_indicador="indicador",
        coluna_valor="percentual_ano",
        tipo_indicador="territorio",
        grupo_indicador="uf_residencia",
        unidade="pontos_percentuais",
        limite_destaque=5.0,
    )
)

display(
    ruptura_uf
)

In [ ]:
TABELAS_RUPTURAS = [
    ruptura_volume,
    ruptura_desfecho,
    ruptura_idade,
    ruptura_preenchimento,
    ruptura_sintomas,
    ruptura_comorbidades,
    ruptura_vacinacao,
    ruptura_escolaridade,
    ruptura_uf,
]

rupturas_temporais = (
    pd.concat(
        TABELAS_RUPTURAS,
        ignore_index=True,
    )
)

rupturas_temporais[
    "periodo_avaliado"
] = (
    f"{ANO_INICIAL_TRIAGEM_TEMPORAL}-"
    f"{ANO_FINAL_TRIAGEM_TEMPORAL}"
)

rupturas_temporais[
    "ordem_destaque"
] = (
    rupturas_temporais[
        "amplitude"
    ]
    .div(
        rupturas_temporais[
            "limite_destaque"
        ]
    )
)

COLUNAS_RUPTURAS_TEMPORAIS = [
    "tipo_indicador",
    "grupo_indicador",
    "indicador",
    "unidade",
    "periodo_avaliado",
    "ano_minimo",
    "valor_minimo",
    "ano_maximo",
    "valor_maximo",
    "amplitude",
    "limite_destaque",
    "flag_destaque",
    "ordem_destaque",
    "quantidade_anos",
]

rupturas_temporais = (
    rupturas_temporais[
        COLUNAS_RUPTURAS_TEMPORAIS
    ]
    .sort_values(
        [
            "flag_destaque",
            "ordem_destaque",
            "grupo_indicador",
            "indicador",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    rupturas_temporais.head(40)
)

In [ ]:
resumo_rupturas_temporais = (
    rupturas_temporais.groupby(
        [
            "tipo_indicador",
            "grupo_indicador",
        ],
        dropna=False,
    )
    .agg(
        quantidade_indicadores=(
            "indicador",
            "size",
        ),
        quantidade_destaques=(
            "flag_destaque",
            "sum",
        ),
        maior_amplitude=(
            "amplitude",
            "max",
        ),
        amplitude_mediana=(
            "amplitude",
            "median",
        ),
    )
    .reset_index()
)

resumo_rupturas_temporais[
    "percentual_indicadores_destacados"
] = (
    resumo_rupturas_temporais[
        "quantidade_destaques"
    ]
    .div(
        resumo_rupturas_temporais[
            "quantidade_indicadores"
        ]
    )
    .mul(100)
)

resumo_rupturas_temporais = (
    resumo_rupturas_temporais
    .sort_values(
        [
            "quantidade_destaques",
            "maior_amplitude",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    resumo_rupturas_temporais
)

In [ ]:
QUANTIDADE_DESTAQUES_GRAFICO = 20

rupturas_para_grafico = (
    rupturas_temporais.loc[
        rupturas_temporais[
            "flag_destaque"
        ]
    ]
    .head(
        QUANTIDADE_DESTAQUES_GRAFICO
    )
    .copy()
)

if rupturas_para_grafico.empty:
    print(
        "Nenhum indicador ultrapassou os limites "
        "exploratórios definidos."
    )
else:
    rupturas_para_grafico[
        "rotulo_grafico"
    ] = (
        rupturas_para_grafico[
            "grupo_indicador"
        ]
        .astype("string")
        + " | "
        + rupturas_para_grafico[
            "indicador"
        ]
        .astype("string")
    )

    rupturas_para_grafico = (
        rupturas_para_grafico
        .sort_values(
            "ordem_destaque",
            ascending=True,
        )
        .reset_index(drop=True)
    )

    figura_rupturas, eixo_rupturas = plt.subplots(
        figsize=(
            13,
            9,
        )
    )

    barras_rupturas = eixo_rupturas.barh(
        rupturas_para_grafico[
            "rotulo_grafico"
        ],
        rupturas_para_grafico[
            "ordem_destaque"
        ],
        color="tab:orange",
    )

    eixo_rupturas.axvline(
        x=1,
        color="tab:red",
        linestyle="--",
        linewidth=1.5,
        label="Limite exploratório",
    )

    eixo_rupturas.set_title(
        "Indicadores com maior mudança temporal entre 2019 e 2025"
    )

    eixo_rupturas.set_xlabel(
        "Amplitude observada dividida pelo limite exploratório"
    )

    eixo_rupturas.set_ylabel(
        "Indicador"
    )

    eixo_rupturas.legend()

    for barra, valor in zip(
        barras_rupturas,
        rupturas_para_grafico[
            "ordem_destaque"
        ],
    ):
        eixo_rupturas.text(
            barra.get_width(),
            barra.get_y()
            + barra.get_height()
            / 2,
            f"{valor:.1f}x",
            va="center",
            ha="left",
            fontsize=8,
        )

    figura_rupturas.text(
        0.01,
        0.01,
        (
            "Triagem exploratória. Valores acima de 1 ultrapassam "
            "o limite definido para o tipo de indicador. "
            "A triagem não determina exclusão automática."
        ),
        fontsize=8,
    )

    figura_rupturas.tight_layout(
        rect=[
            0,
            0.04,
            1,
            1,
        ]
    )

    figura_rupturas.savefig(
        CAMINHO_FIGURA_RUPTURAS,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Figura salva em:",
        CAMINHO_FIGURA_RUPTURAS,
    )

In [ ]:
rupturas_temporais.to_csv(
    CAMINHO_RUPTURAS_TEMPORAIS,
    index=False,
    encoding="utf-8-sig",
)

resumo_rupturas_temporais.to_csv(
    CAMINHO_RESUMO_RUPTURAS,
    index=False,
    encoding="utf-8-sig",
)

grupos_esperados_rupturas = {
    "volume",
    "desfecho",
    "idade",
    "preenchimento",
    "sintomas",
    "comorbidades",
    "vacinacao",
    "escolaridade",
    "uf_residencia",
}

grupos_observados_rupturas = set(
    rupturas_temporais[
        "grupo_indicador"
    ]
    .dropna()
    .astype("string")
    .tolist()
)

amplitudes_validas = (
    rupturas_temporais[
        "amplitude"
    ]
    .dropna()
    .ge(0)
    .all()
)

anos_minimos_validos = (
    rupturas_temporais[
        "ano_minimo"
    ]
    .dropna()
    .between(
        ANO_INICIAL_TRIAGEM_TEMPORAL,
        ANO_FINAL_TRIAGEM_TEMPORAL,
        inclusive="both",
    )
    .all()
)

anos_maximos_validos = (
    rupturas_temporais[
        "ano_maximo"
    ]
    .dropna()
    .between(
        ANO_INICIAL_TRIAGEM_TEMPORAL,
        ANO_FINAL_TRIAGEM_TEMPORAL,
        inclusive="both",
    )
    .all()
)

figura_rupturas_necessaria = (
    rupturas_temporais[
        "flag_destaque"
    ]
    .any()
)

if figura_rupturas_necessaria:
    figura_rupturas_valida = (
        CAMINHO_FIGURA_RUPTURAS.exists()
    )
else:
    figura_rupturas_valida = True

verificacoes_rupturas = {
    "tabela_rupturas_nao_vazia": (
        not rupturas_temporais.empty
    ),
    "resumo_rupturas_nao_vazio": (
        not resumo_rupturas_temporais.empty
    ),
    "todos_grupos_esperados_presentes": (
        grupos_esperados_rupturas
        .issubset(
            grupos_observados_rupturas
        )
    ),
    "amplitudes_nao_negativas": (
        amplitudes_validas
    ),
    "anos_minimos_dentro_periodo": (
        anos_minimos_validos
    ),
    "anos_maximos_dentro_periodo": (
        anos_maximos_validos
    ),
    "arquivo_rupturas_criado": (
        CAMINHO_RUPTURAS_TEMPORAIS.exists()
    ),
    "arquivo_resumo_criado": (
        CAMINHO_RESUMO_RUPTURAS.exists()
    ),
    "figura_rupturas_valida": (
        figura_rupturas_valida
    ),
}

verificacoes_rupturas_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_rupturas.keys()
        ),
        "resultado": list(
            verificacoes_rupturas.values()
        ),
    }
)

display(
    verificacoes_rupturas_df
)

if not all(
    verificacoes_rupturas.values()
):
    falhas_rupturas = [
        verificacao
        for verificacao, resultado
        in verificacoes_rupturas.items()
        if not resultado
    ]

    raise RuntimeError(
        "A consolidação temporal terminou com falhas: "
        f"{falhas_rupturas}"
    )

print(
    "Consolidação das mudanças temporais validada com sucesso."
)

In [ ]:

import gc
import time

t0_base_desfecho = time.perf_counter()

COLUNAS_FIXAS_DESFECHO = [
    "ano_internacao",
    "alvo_obito_srag",
    "faixa_etaria",
    "sexo_categoria",
    "n_sintomas_confirmados",
    "n_sintomas_avaliados",
    "n_comorbidades_confirmadas",
    "n_comorbidades_avaliadas",
    "flag_multimorbidade",
]

COLUNAS_ANALISE_DESFECHO = list(
    dict.fromkeys(
        COLUNAS_FIXAS_DESFECHO
        + list(COLUNAS_SINTOMAS)
        + list(COLUNAS_COMORBIDADES)
    )
)

colunas_desfecho_ausentes = [
    coluna
    for coluna in COLUNAS_ANALISE_DESFECHO
    if coluna not in coorte.columns
]

if colunas_desfecho_ausentes:
    raise KeyError(
        "A coorte não contém todas as colunas necessárias "
        "para a análise de desfecho. "
        f"Colunas ausentes: {colunas_desfecho_ausentes}"
    )

memoria_coorte_mb = (
    coorte.memory_usage(
        index=True,
        deep=True,
    )
    .sum()
    / 1_048_576
)

if "particao" not in coorte.columns:
    raise KeyError(
        "A coluna 'particao' não existe na coorte carregada. "
        "Confira a leitura da partição-mestra na célula 2."
    )

mascara_base_desfecho = (
    coorte["particao"]
    .eq("train_ref")
    .to_numpy(dtype=bool)
)

if not mascara_base_desfecho.any():
    raise ValueError(
        "Não há registros do bloco train_ref na coorte."
    )

base_analise_desfecho = (
    coorte.loc[
        mascara_base_desfecho,
        COLUNAS_ANALISE_DESFECHO,
    ]
    .reset_index(drop=True)
)

origem_analise_desfecho = "particao_treino"

del mascara_base_desfecho

if base_analise_desfecho.empty:
    raise ValueError(
        "A base selecionada para a análise "
        "do desfecho está vazia."
    )

alvo_valido = (
    base_analise_desfecho["alvo_obito_srag"]
    .isin([0, 1])
    .all()
)

if not alvo_valido:
    valores_invalidos_alvo = sorted(
        base_analise_desfecho.loc[
            ~base_analise_desfecho["alvo_obito_srag"].isin([0, 1]),
            "alvo_obito_srag",
        ]
        .drop_duplicates()
        .astype("string")
        .tolist()
    )

    raise ValueError(
        "A base da análise contém valores inválidos "
        "em 'alvo_obito_srag': "
        f"{valores_invalidos_alvo}"
    )

colunas_textuais_desfecho = (
    base_analise_desfecho
    .select_dtypes(
        include=[
            "object",
            "string",
        ]
    )
    .columns
    .tolist()
)

for coluna in colunas_textuais_desfecho:
    quantidade_categorias = (
        base_analise_desfecho[coluna]
        .nunique(dropna=False)
    )

    if quantidade_categorias <= 500:
        base_analise_desfecho[coluna] = (
            base_analise_desfecho[coluna]
            .astype("category")
        )

memoria_base_desfecho_mb = (
    base_analise_desfecho
    .memory_usage(
        index=True,
        deep=True,
    )
    .sum()
    / 1_048_576
)

gc.collect()

print(
    "Origem da análise do desfecho:",
    origem_analise_desfecho,
)

print(
    "Registros utilizados:",
    len(base_analise_desfecho),
)

print(
    "Colunas utilizadas:",
    len(COLUNAS_ANALISE_DESFECHO),
)

print(
    "Óbitos por SRAG:",
    int(
        base_analise_desfecho["alvo_obito_srag"]
        .sum()
    ),
)

print(
    "Memória da coorte completa:",
    f"{memoria_coorte_mb:,.2f} MB",
)

print(
    "Memória da base enxuta de desfecho:",
    f"{memoria_base_desfecho_mb:,.2f} MB",
)

print(
    "Redução de memória:",
    f"{(
        1
        - memoria_base_desfecho_mb
        / memoria_coorte_mb
    ) * 100:,.2f}%",
)

print(
    "Tempo de preparação:",
    f"{time.perf_counter() - t0_base_desfecho:,.2f} s",
)

In [ ]:
CAMINHO_OBITO_FAIXA_ETARIA = (
    C.TABLES_DIR
    / "04_eda_obito_faixa_etaria.csv"
)

CAMINHO_OBITO_NUMERO_COMORBIDADES = (
    C.TABLES_DIR
    / "04_eda_obito_numero_comorbidades.csv"
)

CAMINHO_OBITO_SINTOMAS = (
    C.TABLES_DIR
    / "04_eda_obito_sintomas.csv"
)

CAMINHO_OBITO_COMORBIDADES = (
    C.TABLES_DIR
    / "04_eda_obito_comorbidades.csv"
)

CAMINHO_AUSENCIA_DESFECHO = (
    C.TABLES_DIR
    / "04_eda_ausencia_por_desfecho.csv"
)

CAMINHO_RESUMO_DESFECHO = (
    C.TABLES_DIR
    / "04_eda_resumo_analise_desfecho.csv"
)

CAMINHO_FIGURA_OBITO_FAIXA_ETARIA = (
    C.FIGURES_DIR
    / "04_eda_obito_faixa_etaria.png"
)

CAMINHO_FIGURA_OBITO_COMORBIDADES = (
    C.FIGURES_DIR
    / "04_eda_obito_numero_comorbidades.png"
)

CAMINHO_FIGURA_OR_SINTOMAS = (
    C.FIGURES_DIR
    / "04_eda_odds_ratio_sintomas.png"
)

CAMINHO_FIGURA_OR_COMORBIDADES = (
    C.FIGURES_DIR
    / "04_eda_odds_ratio_comorbidades.png"
)

CAMINHO_FIGURA_AUSENCIA_DESFECHO = (
    C.FIGURES_DIR
    / "04_eda_ausencia_por_desfecho.png"
)

print(
    "Tabela de óbito por faixa etária:",
    CAMINHO_OBITO_FAIXA_ETARIA,
)

print(
    "Tabela de óbito por comorbidades:",
    CAMINHO_OBITO_NUMERO_COMORBIDADES,
)

print(
    "Tabela de óbito por sintomas:",
    CAMINHO_OBITO_SINTOMAS,
)

In [ ]:
def calcular_ic_wilson(
    quantidade_eventos: int,
    quantidade_total: int,
    nivel_confianca: float = 0.95,
) -> tuple:
    """Calcula o intervalo de confiança de Wilson para uma proporção."""

    if quantidade_total <= 0:
        return (
            np.nan,
            np.nan,
        )

    if nivel_confianca != 0.95:
        raise ValueError(
            "Esta implementação utiliza o valor crítico "
            "correspondente ao nível de confiança de 95%."
        )

    valor_z = 1.959963984540054

    proporcao = (
        quantidade_eventos
        / quantidade_total
    )

    z_quadrado = np.square(
        valor_z
    )

    denominador = (
        1
        + z_quadrado
        / quantidade_total
    )

    centro = (
        proporcao
        + z_quadrado
        / (
            2
            * quantidade_total
        )
    )

    componente_variancia = (
        proporcao
        * (
            1
            - proporcao
        )
        / quantidade_total
        + z_quadrado
        / (
            4
            * np.square(
                quantidade_total
            )
        )
    )

    margem = (
        valor_z
        * np.sqrt(
            componente_variancia
        )
    )

    limite_inferior = (
        (
            centro
            - margem
        )
        / denominador
    )

    limite_superior = (
        (
            centro
            + margem
        )
        / denominador
    )

    return (
        limite_inferior
        * 100,
        limite_superior
        * 100,
    )


def calcular_odds_ratio_bruta(
    obito_exposto: int,
    cura_exposto: int,
    obito_referencia: int,
    cura_referencia: int,
) -> tuple:
    """Calcula a razão de chances bruta e o IC de 95%."""

    celulas = np.array(
        [
            obito_exposto,
            cura_exposto,
            obito_referencia,
            cura_referencia,
        ],
        dtype="float64",
    )

    if np.any(
        celulas < 0
    ):
        raise ValueError(
            "As contagens da tabela dois por dois "
            "não podem ser negativas."
        )

    if np.any(
        celulas == 0
    ):
        celulas = (
            celulas
            + 0.5
        )

    obito_exposto_corrigido = (
        celulas[0]
    )

    cura_exposto_corrigido = (
        celulas[1]
    )

    obito_referencia_corrigido = (
        celulas[2]
    )

    cura_referencia_corrigido = (
        celulas[3]
    )

    odds_ratio = (
        obito_exposto_corrigido
        * cura_referencia_corrigido
        / (
            cura_exposto_corrigido
            * obito_referencia_corrigido
        )
    )

    erro_padrao_log = np.sqrt(
        1
        / obito_exposto_corrigido
        + 1
        / cura_exposto_corrigido
        + 1
        / obito_referencia_corrigido
        + 1
        / cura_referencia_corrigido
    )

    valor_z = 1.959963984540054

    limite_inferior = np.exp(
        np.log(
            odds_ratio
        )
        - valor_z
        * erro_padrao_log
    )

    limite_superior = np.exp(
        np.log(
            odds_ratio
        )
        + valor_z
        * erro_padrao_log
    )

    return (
        float(
            odds_ratio
        ),
        float(
            limite_inferior
        ),
        float(
            limite_superior
        ),
    )


def criar_tabela_obito_categorica(
    dataframe: pd.DataFrame,
    coluna_categoria: str,
    nome_categoria_saida: str,
) -> pd.DataFrame:
    """
    Calcula quantidade, proporção de óbito e intervalo de Wilson
    para cada categoria de uma variável.
    """

    if coluna_categoria not in dataframe.columns:
        raise KeyError(
            "A coluna não está disponível na base: "
            f"{coluna_categoria}"
        )

    tabela = (
        dataframe.groupby(
            coluna_categoria,
            dropna=False,
        )
        .agg(
            quantidade_total=(
                "alvo_obito_srag",
                "size",
            ),
            quantidade_obito_srag=(
                "alvo_obito_srag",
                "sum",
            ),
        )
        .reset_index()
        .rename(
            columns={
                coluna_categoria: (
                    nome_categoria_saida
                ),
            }
        )
    )

    tabela[
        "quantidade_cura"
    ] = (
        tabela[
            "quantidade_total"
        ]
        .sub(
            tabela[
                "quantidade_obito_srag"
            ]
        )
    )

    tabela[
        "percentual_obito_srag"
    ] = (
        tabela[
            "quantidade_obito_srag"
        ]
        .div(
            tabela[
                "quantidade_total"
        ]
        )
        .mul(100)
    )

    intervalos = [
        calcular_ic_wilson(
            quantidade_eventos=int(
                linha[
                    "quantidade_obito_srag"
                ]
            ),
            quantidade_total=int(
                linha[
                    "quantidade_total"
                ]
            ),
        )
        for _, linha in tabela.iterrows()
    ]

    tabela[
        "ic95_inferior"
    ] = [
        intervalo[0]
        for intervalo in intervalos
    ]

    tabela[
        "ic95_superior"
    ] = [
        intervalo[1]
        for intervalo in intervalos
    ]

    tabela = (
        tabela.sort_values(
            "percentual_obito_srag",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return tabela

In [ ]:
if "faixa_etaria" not in base_analise_desfecho.columns:
    raise KeyError(
        "A coluna faixa_etaria não está disponível. "
        "Execute primeiro a seção demográfica."
    )

obito_faixa_etaria = (
    criar_tabela_obito_categorica(
        dataframe=base_analise_desfecho,
        coluna_categoria="faixa_etaria",
        nome_categoria_saida="faixa_etaria",
    )
)

ORDEM_FAIXA_OBITO = (
    ROTULOS_FAIXA_ETARIA
    + [
        "Idade ausente ou inválida",
    ]
)

ordem_faixa_obito = {
    categoria: posicao
    for posicao, categoria in enumerate(
        ORDEM_FAIXA_OBITO
    )
}

obito_faixa_etaria[
    "ordem_faixa"
] = (
    obito_faixa_etaria[
        "faixa_etaria"
    ]
    .astype("string")
    .map(
        ordem_faixa_obito
    )
    .fillna(
        len(
            ordem_faixa_obito
        )
    )
)

obito_faixa_etaria = (
    obito_faixa_etaria
    .sort_values(
        "ordem_faixa"
    )
    .drop(
        columns=[
            "ordem_faixa",
        ]
    )
    .reset_index(drop=True)
)

display(
    obito_faixa_etaria
)

In [ ]:
COLUNAS_CONTAGEM_COMORBIDADES = [
    "n_comorbidades_confirmadas",
    "n_comorbidades_avaliadas",
]

colunas_contagem_comorbidades_ausentes = [
    coluna
    for coluna in COLUNAS_CONTAGEM_COMORBIDADES
    if coluna not in base_analise_desfecho.columns
]

if colunas_contagem_comorbidades_ausentes:
    raise KeyError(
        "As colunas de contagem de comorbidades "
        "não estão disponíveis: "
        f"{colunas_contagem_comorbidades_ausentes}"
    )

numero_comorbidades = (
    base_analise_desfecho[
        "n_comorbidades_confirmadas"
    ]
    .astype("Int64")
)

condicoes_comorbidades = [
    numero_comorbidades.eq(0),
    numero_comorbidades.eq(1),
    numero_comorbidades.eq(2),
    numero_comorbidades.ge(3),
]

rotulos_comorbidades = [
    "Nenhuma comorbidade confirmada",
    "Uma comorbidade confirmada",
    "Duas comorbidades confirmadas",
    "Três ou mais comorbidades confirmadas",
]

base_analise_desfecho[
    "grupo_numero_comorbidades"
] = pd.Series(
    np.select(
        condicoes_comorbidades,
        rotulos_comorbidades,
        default=(
            "Número de comorbidades indisponível"
        ),
    ),
    index=base_analise_desfecho.index,
    dtype="string",
)

obito_numero_comorbidades = (
    criar_tabela_obito_categorica(
        dataframe=base_analise_desfecho,
        coluna_categoria=(
            "grupo_numero_comorbidades"
        ),
        nome_categoria_saida=(
            "grupo_numero_comorbidades"
        ),
    )
)

ORDEM_NUMERO_COMORBIDADES = (
    rotulos_comorbidades
    + [
        "Número de comorbidades indisponível",
    ]
)

ordem_numero_comorbidades = {
    categoria: posicao
    for posicao, categoria in enumerate(
        ORDEM_NUMERO_COMORBIDADES
    )
}

obito_numero_comorbidades[
    "ordem_comorbidades"
] = (
    obito_numero_comorbidades[
        "grupo_numero_comorbidades"
    ]
    .map(
        ordem_numero_comorbidades
    )
    .fillna(
        len(
            ordem_numero_comorbidades
        )
    )
)

obito_numero_comorbidades = (
    obito_numero_comorbidades
    .sort_values(
        "ordem_comorbidades"
    )
    .drop(
        columns=[
            "ordem_comorbidades",
        ]
    )
    .reset_index(drop=True)
)

display(
    obito_numero_comorbidades
)

In [ ]:
def criar_tabela_obito_indicadores(
    dataframe: pd.DataFrame,
    colunas: list,
    grupo_indicador: str,
    nomes_amigaveis: dict,
) -> pd.DataFrame:
    """
    Calcula proporções de óbito e razão de chances bruta
    para respostas sim versus não.
    """

    registros = []

    for coluna in colunas:
        if coluna not in dataframe.columns:
            continue

        resposta = (
            normalizar_resposta_clinica(
                dataframe[
                    coluna
                ]
            )
        )

        dados_variavel = pd.DataFrame(
            {
                "resposta": resposta,
                "alvo_obito_srag": dataframe[
                    "alvo_obito_srag"
                ],
            }
        )

        dados_validos = (
            dados_variavel.loc[
                dados_variavel[
                    "resposta"
                ].isin(
                    [
                        "sim",
                        "nao",
                    ]
                )
            ]
            .copy()
        )

        quantidade_sim = int(
            dados_validos[
                "resposta"
            ]
            .eq("sim")
            .sum()
        )

        quantidade_nao = int(
            dados_validos[
                "resposta"
            ]
            .eq("nao")
            .sum()
        )

        obito_sim = int(
            dados_validos.loc[
                dados_validos[
                    "resposta"
                ].eq("sim"),
                "alvo_obito_srag",
            ]
            .sum()
        )

        obito_nao = int(
            dados_validos.loc[
                dados_validos[
                    "resposta"
                ].eq("nao"),
                "alvo_obito_srag",
            ]
            .sum()
        )

        cura_sim = (
            quantidade_sim
            - obito_sim
        )

        cura_nao = (
            quantidade_nao
            - obito_nao
        )

        percentual_obito_sim = (
            obito_sim
            / quantidade_sim
            * 100
            if quantidade_sim > 0
            else np.nan
        )

        percentual_obito_nao = (
            obito_nao
            / quantidade_nao
            * 100
            if quantidade_nao > 0
            else np.nan
        )

        ic_sim = calcular_ic_wilson(
            quantidade_eventos=obito_sim,
            quantidade_total=quantidade_sim,
        )

        ic_nao = calcular_ic_wilson(
            quantidade_eventos=obito_nao,
            quantidade_total=quantidade_nao,
        )

        if (
            quantidade_sim > 0
            and quantidade_nao > 0
        ):
            resultado_or = (
                calcular_odds_ratio_bruta(
                    obito_exposto=obito_sim,
                    cura_exposto=cura_sim,
                    obito_referencia=obito_nao,
                    cura_referencia=cura_nao,
                )
            )
        else:
            resultado_or = (
                np.nan,
                np.nan,
                np.nan,
            )

        quantidade_ignorado = int(
            dados_variavel[
                "resposta"
            ]
            .eq("ignorado")
            .sum()
        )

        quantidade_ausente = int(
            dados_variavel[
                "resposta"
            ]
            .eq("ausente")
            .sum()
        )

        quantidade_inesperado = int(
            dados_variavel[
                "resposta"
            ]
            .eq("inesperado")
            .sum()
        )

        registros.append(
            {
                "grupo_indicador": grupo_indicador,
                "coluna": coluna,
                "nome_amigavel": (
                    nomes_amigaveis.get(
                        coluna,
                        coluna,
                    )
                ),
                "quantidade_sim": quantidade_sim,
                "obito_sim": obito_sim,
                "cura_sim": cura_sim,
                "percentual_obito_sim": (
                    percentual_obito_sim
                ),
                "ic95_obito_sim_inferior": (
                    ic_sim[0]
                ),
                "ic95_obito_sim_superior": (
                    ic_sim[1]
                ),
                "quantidade_nao": quantidade_nao,
                "obito_nao": obito_nao,
                "cura_nao": cura_nao,
                "percentual_obito_nao": (
                    percentual_obito_nao
                ),
                "ic95_obito_nao_inferior": (
                    ic_nao[0]
                ),
                "ic95_obito_nao_superior": (
                    ic_nao[1]
                ),
                "odds_ratio_bruta": (
                    resultado_or[0]
                ),
                "ic95_or_inferior": (
                    resultado_or[1]
                ),
                "ic95_or_superior": (
                    resultado_or[2]
                ),
                "quantidade_ignorado": (
                    quantidade_ignorado
                ),
                "quantidade_ausente": (
                    quantidade_ausente
                ),
                "quantidade_inesperado": (
                    quantidade_inesperado
                ),
            }
        )

    resultado = pd.DataFrame(
        registros
    )

    resultado = (
        resultado.sort_values(
            [
                "odds_ratio_bruta",
                "coluna",
            ],
            ascending=[
                False,
                True,
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    return resultado

In [ ]:
obito_sintomas = (
    criar_tabela_obito_indicadores(
        dataframe=base_analise_desfecho,
        colunas=COLUNAS_SINTOMAS,
        grupo_indicador="sintomas",
        nomes_amigaveis=(
            nomes_amigaveis_dicionario
        ),
    )
)

obito_comorbidades = (
    criar_tabela_obito_indicadores(
        dataframe=base_analise_desfecho,
        colunas=COLUNAS_COMORBIDADES,
        grupo_indicador="comorbidades",
        nomes_amigaveis=(
            nomes_amigaveis_dicionario
        ),
    )
)

if obito_sintomas.empty:
    raise ValueError(
        "A tabela de óbito por sintomas está vazia."
    )

if obito_comorbidades.empty:
    raise ValueError(
        "A tabela de óbito por comorbidades está vazia."
    )

display(
    obito_sintomas
)

display(
    obito_comorbidades
)

In [ ]:
COLUNAS_AUSENCIA_DESFECHO = list(
    dict.fromkeys(
        COLUNAS_SINTOMAS
        + COLUNAS_COMORBIDADES
        + COLUNAS_VACINACAO
        + [
            "cs_escol_n",
            "sg_uf",
            "nu_idade_n",
            "tp_idade",
        ]
    )
)

COLUNAS_AUSENCIA_DESFECHO = [
    coluna
    for coluna in COLUNAS_AUSENCIA_DESFECHO
    if coluna in base_analise_desfecho.columns
]

registros_ausencia_desfecho = []

for coluna in COLUNAS_AUSENCIA_DESFECHO:
    serie_ausente = (
        base_analise_desfecho[
            coluna
        ]
        .isna()
    )

    mascara_cura = (
        base_analise_desfecho[
            "alvo_obito_srag"
        ]
        .eq(0)
    )

    mascara_obito = (
        base_analise_desfecho[
            "alvo_obito_srag"
        ]
        .eq(1)
    )

    quantidade_cura = int(
        mascara_cura.sum()
    )

    quantidade_obito = int(
        mascara_obito.sum()
    )

    ausente_cura = int(
        (
            serie_ausente
            & mascara_cura
        )
        .sum()
    )

    ausente_obito = int(
        (
            serie_ausente
            & mascara_obito
        )
        .sum()
    )

    percentual_ausente_cura = (
        ausente_cura
        / quantidade_cura
        * 100
        if quantidade_cura > 0
        else np.nan
    )

    percentual_ausente_obito = (
        ausente_obito
        / quantidade_obito
        * 100
        if quantidade_obito > 0
        else np.nan
    )

    registros_ausencia_desfecho.append(
        {
            "coluna": coluna,
            "nome_amigavel": (
                nomes_amigaveis_dicionario.get(
                    coluna,
                    coluna,
                )
            ),
            "quantidade_cura": quantidade_cura,
            "ausente_cura": ausente_cura,
            "percentual_ausente_cura": (
                percentual_ausente_cura
            ),
            "quantidade_obito": quantidade_obito,
            "ausente_obito": ausente_obito,
            "percentual_ausente_obito": (
                percentual_ausente_obito
            ),
            "diferenca_pp_obito_menos_cura": (
                percentual_ausente_obito
                - percentual_ausente_cura
            ),
        }
    )

ausencia_por_desfecho = pd.DataFrame(
    registros_ausencia_desfecho
)

ausencia_por_desfecho[
    "diferenca_absoluta_pp"
] = (
    ausencia_por_desfecho[
        "diferenca_pp_obito_menos_cura"
    ]
    .abs()
)

ausencia_por_desfecho = (
    ausencia_por_desfecho
    .sort_values(
        [
            "diferenca_absoluta_pp",
            "coluna",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    ausencia_por_desfecho.head(30)
)

In [ ]:
dados_faixa_grafico = (
    obito_faixa_etaria.loc[
        obito_faixa_etaria[
            "quantidade_total"
        ].gt(0)
    ]
    .copy()
)

categorias_faixa = (
    dados_faixa_grafico[
        "faixa_etaria"
    ]
    .astype("string")
    .tolist()
)

percentuais_faixa = (
    dados_faixa_grafico[
        "percentual_obito_srag"
    ]
    .astype(float)
    .to_numpy()
)

erros_inferiores_faixa = (
    dados_faixa_grafico[
        "percentual_obito_srag"
    ]
    .sub(
        dados_faixa_grafico[
            "ic95_inferior"
        ]
    )
    .astype(float)
    .to_numpy()
)

erros_superiores_faixa = (
    dados_faixa_grafico[
        "ic95_superior"
    ]
    .sub(
        dados_faixa_grafico[
            "percentual_obito_srag"
        ]
    )
    .astype(float)
    .to_numpy()
)

posicoes_faixa = np.arange(
    len(
        dados_faixa_grafico
    )
)

figura_obito_faixa, eixo_obito_faixa = plt.subplots(
    figsize=(
        12,
        7,
    )
)

eixo_obito_faixa.bar(
    posicoes_faixa,
    percentuais_faixa,
    color="tab:red",
)

eixo_obito_faixa.errorbar(
    posicoes_faixa,
    percentuais_faixa,
    yerr=[
        erros_inferiores_faixa,
        erros_superiores_faixa,
    ],
    fmt="none",
    ecolor="black",
    capsize=4,
)

eixo_obito_faixa.set_title(
    "Proporção observada de óbito por SRAG por faixa etária"
)

eixo_obito_faixa.set_xlabel(
    "Faixa etária"
)

eixo_obito_faixa.set_ylabel(
    "Óbito por SRAG"
)

eixo_obito_faixa.set_xticks(
    posicoes_faixa
)

eixo_obito_faixa.set_xticklabels(
    categorias_faixa,
    rotation=35,
    ha="right",
)

eixo_obito_faixa.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

figura_obito_faixa.text(
    0.01,
    0.01,
    (
        "Barras de erro: intervalo de confiança de Wilson de 95%. "
        f"Base: {origem_analise_desfecho}."
    ),
    fontsize=8,
)

figura_obito_faixa.tight_layout(
    rect=[
        0,
        0.05,
        1,
        1,
    ]
)

figura_obito_faixa.savefig(
    CAMINHO_FIGURA_OBITO_FAIXA_ETARIA,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_OBITO_FAIXA_ETARIA,
)

In [ ]:
dados_comorbidades_grafico = (
    obito_numero_comorbidades.loc[
        obito_numero_comorbidades[
            "quantidade_total"
        ].gt(0)
    ]
    .copy()
)

categorias_comorbidades = (
    dados_comorbidades_grafico[
        "grupo_numero_comorbidades"
    ]
    .astype("string")
    .tolist()
)

percentuais_comorbidades = (
    dados_comorbidades_grafico[
        "percentual_obito_srag"
    ]
    .astype(float)
    .to_numpy()
)

erros_inferiores_comorbidades = (
    dados_comorbidades_grafico[
        "percentual_obito_srag"
    ]
    .sub(
        dados_comorbidades_grafico[
            "ic95_inferior"
        ]
    )
    .astype(float)
    .to_numpy()
)

erros_superiores_comorbidades = (
    dados_comorbidades_grafico[
        "ic95_superior"
    ]
    .sub(
        dados_comorbidades_grafico[
            "percentual_obito_srag"
        ]
    )
    .astype(float)
    .to_numpy()
)

posicoes_comorbidades = np.arange(
    len(
        dados_comorbidades_grafico
    )
)

figura_obito_comorbidades, eixo_obito_comorbidades = plt.subplots(
    figsize=(
        11,
        7,
    )
)

eixo_obito_comorbidades.bar(
    posicoes_comorbidades,
    percentuais_comorbidades,
    color="tab:orange",
)

eixo_obito_comorbidades.errorbar(
    posicoes_comorbidades,
    percentuais_comorbidades,
    yerr=[
        erros_inferiores_comorbidades,
        erros_superiores_comorbidades,
    ],
    fmt="none",
    ecolor="black",
    capsize=4,
)

eixo_obito_comorbidades.set_title(
    "Proporção observada de óbito por número de comorbidades"
)

eixo_obito_comorbidades.set_xlabel(
    "Número de comorbidades confirmadas"
)

eixo_obito_comorbidades.set_ylabel(
    "Óbito por SRAG"
)

eixo_obito_comorbidades.set_xticks(
    posicoes_comorbidades
)

eixo_obito_comorbidades.set_xticklabels(
    categorias_comorbidades,
    rotation=25,
    ha="right",
)

eixo_obito_comorbidades.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

figura_obito_comorbidades.text(
    0.01,
    0.01,
    (
        "Barras de erro: intervalo de confiança de Wilson de 95%. "
        f"Base: {origem_analise_desfecho}."
    ),
    fontsize=8,
)

figura_obito_comorbidades.tight_layout(
    rect=[
        0,
        0.05,
        1,
        1,
    ]
)

figura_obito_comorbidades.savefig(
    CAMINHO_FIGURA_OBITO_COMORBIDADES,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_OBITO_COMORBIDADES,
)

In [ ]:
def criar_grafico_odds_ratio(
    tabela: pd.DataFrame,
    titulo: str,
    caminho_saida: Path,
) -> None:
    """
    Cria um gráfico de razão de chances bruta com IC de 95%.
    """

    dados_grafico = (
        tabela.dropna(
            subset=[
                "odds_ratio_bruta",
                "ic95_or_inferior",
                "ic95_or_superior",
            ]
        )
        .copy()
    )

    dados_grafico = (
        dados_grafico.loc[
            dados_grafico[
                "odds_ratio_bruta"
            ].gt(0)
            & dados_grafico[
                "ic95_or_inferior"
            ].gt(0)
            & dados_grafico[
                "ic95_or_superior"
            ].gt(0)
        ]
        .sort_values(
            "odds_ratio_bruta"
        )
        .reset_index(drop=True)
    )

    if dados_grafico.empty:
        raise ValueError(
            "Não existem razões de chances válidas "
            "para gerar o gráfico."
        )

    posicoes = np.arange(
        len(
            dados_grafico
        )
    )

    valores_or = (
        dados_grafico[
            "odds_ratio_bruta"
        ]
        .astype(float)
        .to_numpy()
    )

    erro_inferior = (
        dados_grafico[
            "odds_ratio_bruta"
        ]
        .sub(
            dados_grafico[
                "ic95_or_inferior"
            ]
        )
        .astype(float)
        .to_numpy()
    )

    erro_superior = (
        dados_grafico[
            "ic95_or_superior"
        ]
        .sub(
            dados_grafico[
                "odds_ratio_bruta"
            ]
        )
        .astype(float)
        .to_numpy()
    )

    rotulos = (
        dados_grafico[
            "nome_amigavel"
        ]
        .astype("string")
        .tolist()
    )

    figura, eixo = plt.subplots(
        figsize=(
            12,
            8,
        )
    )

    eixo.errorbar(
        valores_or,
        posicoes,
        xerr=[
            erro_inferior,
            erro_superior,
        ],
        fmt="o",
        color="tab:blue",
        ecolor="black",
        capsize=4,
    )

    eixo.axvline(
        x=1,
        color="tab:red",
        linestyle="--",
        linewidth=1.5,
    )

    eixo.set_xscale(
        "log"
    )

    eixo.set_yticks(
        posicoes
    )

    eixo.set_yticklabels(
        rotulos
    )

    eixo.set_title(
        titulo
    )

    eixo.set_xlabel(
        "Razão de chances bruta, escala logarítmica"
    )

    eixo.set_ylabel(
        "Indicador"
    )

    figura.text(
        0.01,
        0.01,
        (
            "Comparação: resposta sim versus resposta não. "
            "Intervalo de confiança de 95%. "
            "Associação bruta, não ajustada e não causal."
        ),
        fontsize=8,
    )

    figura.tight_layout(
        rect=[
            0,
            0.05,
            1,
            1,
        ]
    )

    figura.savefig(
        caminho_saida,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Figura salva em:",
        caminho_saida,
    )

In [ ]:
criar_grafico_odds_ratio(
    tabela=obito_sintomas,
    titulo=(
        "Associação bruta entre sintomas registrados "
        "e óbito por SRAG"
    ),
    caminho_saida=(
        CAMINHO_FIGURA_OR_SINTOMAS
    ),
)

criar_grafico_odds_ratio(
    tabela=obito_comorbidades,
    titulo=(
        "Associação bruta entre comorbidades registradas "
        "e óbito por SRAG"
    ),
    caminho_saida=(
        CAMINHO_FIGURA_OR_COMORBIDADES
    ),
)

In [ ]:
QUANTIDADE_VARIAVEIS_AUSENCIA_GRAFICO = 15

ausencia_grafico = (
    ausencia_por_desfecho.head(
        QUANTIDADE_VARIAVEIS_AUSENCIA_GRAFICO
    )
    .sort_values(
        "diferenca_pp_obito_menos_cura"
    )
    .copy()
)

figura_ausencia, eixo_ausencia = plt.subplots(
    figsize=(
        12,
        8,
    )
)

cores_ausencia = [
    (
        "tab:red"
        if diferenca > 0
        else "tab:blue"
    )
    for diferenca in ausencia_grafico[
        "diferenca_pp_obito_menos_cura"
    ]
]

barras_ausencia = eixo_ausencia.barh(
    ausencia_grafico[
        "nome_amigavel"
    ],
    ausencia_grafico[
        "diferenca_pp_obito_menos_cura"
    ],
    color=cores_ausencia,
)

eixo_ausencia.axvline(
    x=0,
    color="black",
    linewidth=1,
)

eixo_ausencia.set_title(
    "Diferença de ausência entre óbitos e curas"
)

eixo_ausencia.set_xlabel(
    "Ausência nos óbitos menos ausência nas curas, em pontos percentuais"
)

eixo_ausencia.set_ylabel(
    "Variável"
)

for barra, diferenca in zip(
    barras_ausencia,
    ausencia_grafico[
        "diferenca_pp_obito_menos_cura"
    ],
):
    alinhamento = (
        "left"
        if diferenca >= 0
        else "right"
    )

    eixo_ausencia.text(
        diferenca,
        barra.get_y()
        + barra.get_height()
        / 2,
        f"{diferenca:.1f} p.p.",
        va="center",
        ha=alinhamento,
        fontsize=8,
    )

figura_ausencia.text(
    0.01,
    0.01,
    (
        "Valores positivos indicam maior ausência entre óbitos. "
        "A diferença não identifica o mecanismo da ausência."
    ),
    fontsize=8,
)

figura_ausencia.tight_layout(
    rect=[
        0,
        0.05,
        1,
        1,
    ]
)

figura_ausencia.savefig(
    CAMINHO_FIGURA_AUSENCIA_DESFECHO,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Figura salva em:",
    CAMINHO_FIGURA_AUSENCIA_DESFECHO,
)

In [ ]:
quantidade_base_desfecho = len(
    base_analise_desfecho
)

quantidade_obitos_desfecho = int(
    base_analise_desfecho[
        "alvo_obito_srag"
    ]
    .sum()
)

quantidade_curas_desfecho = (
    quantidade_base_desfecho
    - quantidade_obitos_desfecho
)

percentual_obito_global = (
    quantidade_obitos_desfecho
    / quantidade_base_desfecho
    * 100
)

ic_global_obito = calcular_ic_wilson(
    quantidade_eventos=(
        quantidade_obitos_desfecho
    ),
    quantidade_total=(
        quantidade_base_desfecho
    ),
)

resumo_analise_desfecho = pd.DataFrame(
    {
        "metrica": [
            "origem_analise",
            "quantidade_registros",
            "quantidade_curas",
            "quantidade_obitos_srag",
            "percentual_obito_srag",
            "ic95_obito_inferior",
            "ic95_obito_superior",
            "quantidade_sintomas_analisados",
            "quantidade_comorbidades_analisadas",
            "quantidade_variaveis_ausencia",
        ],
        "valor": [
            origem_analise_desfecho,
            quantidade_base_desfecho,
            quantidade_curas_desfecho,
            quantidade_obitos_desfecho,
            percentual_obito_global,
            ic_global_obito[0],
            ic_global_obito[1],
            len(
                obito_sintomas
            ),
            len(
                obito_comorbidades
            ),
            len(
                ausencia_por_desfecho
            ),
        ],
    }
)

display(
    resumo_analise_desfecho
)

In [ ]:
obito_faixa_etaria.to_csv(
    CAMINHO_OBITO_FAIXA_ETARIA,
    index=False,
    encoding="utf-8-sig",
)

obito_numero_comorbidades.to_csv(
    CAMINHO_OBITO_NUMERO_COMORBIDADES,
    index=False,
    encoding="utf-8-sig",
)

obito_sintomas.to_csv(
    CAMINHO_OBITO_SINTOMAS,
    index=False,
    encoding="utf-8-sig",
)

obito_comorbidades.to_csv(
    CAMINHO_OBITO_COMORBIDADES,
    index=False,
    encoding="utf-8-sig",
)

ausencia_por_desfecho.to_csv(
    CAMINHO_AUSENCIA_DESFECHO,
    index=False,
    encoding="utf-8-sig",
)

resumo_analise_desfecho.to_csv(
    CAMINHO_RESUMO_DESFECHO,
    index=False,
    encoding="utf-8-sig",
)

percentuais_faixa_validos = (
    obito_faixa_etaria[
        "percentual_obito_srag"
    ]
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

percentuais_comorbidades_validos = (
    obito_numero_comorbidades[
        "percentual_obito_srag"
    ]
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

odds_ratios_sintomas_validas = (
    obito_sintomas[
        "odds_ratio_bruta"
    ]
    .dropna()
    .gt(0)
    .all()
)

odds_ratios_comorbidades_validas = (
    obito_comorbidades[
        "odds_ratio_bruta"
    ]
    .dropna()
    .gt(0)
    .all()
)

verificacoes_desfecho = {
    "base_desfecho_nao_vazia": (
        not base_analise_desfecho.empty
    ),
    "cura_presente": (
        quantidade_curas_desfecho > 0
    ),
    "obito_presente": (
        quantidade_obitos_desfecho > 0
    ),
    "tabela_faixa_etaria_nao_vazia": (
        not obito_faixa_etaria.empty
    ),
    "tabela_numero_comorbidades_nao_vazia": (
        not obito_numero_comorbidades.empty
    ),
    "tabela_sintomas_nao_vazia": (
        not obito_sintomas.empty
    ),
    "tabela_comorbidades_nao_vazia": (
        not obito_comorbidades.empty
    ),
    "tabela_ausencia_nao_vazia": (
        not ausencia_por_desfecho.empty
    ),
    "percentuais_faixa_validos": (
        percentuais_faixa_validos
    ),
    "percentuais_comorbidades_validos": (
        percentuais_comorbidades_validos
    ),
    "odds_ratios_sintomas_positivas": (
        odds_ratios_sintomas_validas
    ),
    "odds_ratios_comorbidades_positivas": (
        odds_ratios_comorbidades_validas
    ),
    "arquivo_faixa_etaria_criado": (
        CAMINHO_OBITO_FAIXA_ETARIA.exists()
    ),
    "arquivo_numero_comorbidades_criado": (
        CAMINHO_OBITO_NUMERO_COMORBIDADES.exists()
    ),
    "arquivo_sintomas_criado": (
        CAMINHO_OBITO_SINTOMAS.exists()
    ),
    "arquivo_comorbidades_criado": (
        CAMINHO_OBITO_COMORBIDADES.exists()
    ),
    "arquivo_ausencia_criado": (
        CAMINHO_AUSENCIA_DESFECHO.exists()
    ),
    "arquivo_resumo_criado": (
        CAMINHO_RESUMO_DESFECHO.exists()
    ),
    "figura_faixa_etaria_criada": (
        CAMINHO_FIGURA_OBITO_FAIXA_ETARIA.exists()
    ),
    "figura_numero_comorbidades_criada": (
        CAMINHO_FIGURA_OBITO_COMORBIDADES.exists()
    ),
    "figura_or_sintomas_criada": (
        CAMINHO_FIGURA_OR_SINTOMAS.exists()
    ),
    "figura_or_comorbidades_criada": (
        CAMINHO_FIGURA_OR_COMORBIDADES.exists()
    ),
    "figura_ausencia_criada": (
        CAMINHO_FIGURA_AUSENCIA_DESFECHO.exists()
    ),
}

verificacoes_desfecho_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_desfecho.keys()
        ),
        "resultado": list(
            verificacoes_desfecho.values()
        ),
    }
)

display(
    verificacoes_desfecho_df
)

if not all(
    verificacoes_desfecho.values()
):
    falhas_desfecho = [
        verificacao
        for verificacao, resultado
        in verificacoes_desfecho.items()
        if not resultado
    ]

    raise RuntimeError(
        "A análise do desfecho terminou com falhas: "
        f"{falhas_desfecho}"
    )

print(
    "Análise de óbito validada com sucesso."
)

In [ ]:
import seaborn as sns


def preparar_matriz_clinica_heatmap(
    tabela_clinica: pd.DataFrame,
    coluna_valor: str,
) -> tuple:
    """Transforma a tabela clínica longa em uma matriz variável por ano."""

    COLUNAS_NECESSARIAS = [
        "ano_internacao",
        "coluna",
        "nome_amigavel",
        coluna_valor,
    ]

    colunas_ausentes = [
        coluna
        for coluna in COLUNAS_NECESSARIAS
        if coluna not in tabela_clinica.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            "A tabela clínica não contém as colunas necessárias: "
            f"{colunas_ausentes}"
        )

    dados_heatmap = (
        tabela_clinica[
            COLUNAS_NECESSARIAS
        ]
        .copy()
    )

    dados_heatmap[
        coluna_valor
    ] = pd.to_numeric(
        dados_heatmap[
            coluna_valor
        ],
        errors="coerce",
    )

    matriz_heatmap = (
        dados_heatmap.pivot(
            index="coluna",
            columns="ano_internacao",
            values=coluna_valor,
        )
        .sort_index(
            axis=1
        )
    )

    nomes_amigaveis_heatmap = (
        dados_heatmap[
            [
                "coluna",
                "nome_amigavel",
            ]
        ]
        .drop_duplicates(
            subset=[
                "coluna",
            ]
        )
        .set_index(
            "coluna"
        )[
            "nome_amigavel"
        ]
        .astype("string")
    )

    rotulos_linhas = []

    for coluna in matriz_heatmap.index:
        nome_amigavel = nomes_amigaveis_heatmap.get(
            coluna,
            coluna,
        )

        if pd.isna(
            nome_amigavel
        ):
            nome_amigavel = coluna

        nome_amigavel = str(
            nome_amigavel
        ).strip()

        if not nome_amigavel:
            nome_amigavel = coluna

        if nome_amigavel.lower() == coluna.lower():
            rotulo = nome_amigavel
        else:
            rotulo = (
                f"{nome_amigavel} ({coluna})"
            )

        rotulos_linhas.append(
            rotulo
        )

    matriz_heatmap.index = (
        rotulos_linhas
    )

    matriz_heatmap.columns = [
        int(
            ano
        )
        for ano in matriz_heatmap.columns
    ]

    return (
        matriz_heatmap,
        rotulos_linhas,
    )


def exibir_heatmap_percentual(
    matriz: pd.DataFrame,
    titulo: str,
    rotulo_escala: str,
    mapa_cores: str,
) -> None:
    """
    Exibe um heatmap percentual diretamente no notebook.
    """

    if matriz.empty:
        raise ValueError(
            "A matriz recebida para o heatmap está vazia."
        )

    valores_validos = (
        matriz.stack(
            future_stack=True
        )
        .dropna()
    )

    if valores_validos.empty:
        raise ValueError(
            "A matriz do heatmap não contém valores numéricos."
        )

    if not valores_validos.between(
        0,
        100,
        inclusive="both",
    ).all():
        raise ValueError(
            "O heatmap percentual contém valores "
            "fora do intervalo de zero a cem."
        )

    altura_figura = max(
        5,
        len(
            matriz
        )
        * 0.48,
    )

    figura, eixo = plt.subplots(
        figsize=(
            12,
            altura_figura,
        )
    )

    sns.heatmap(
        matriz,
        annot=True,
        fmt=".1f",
        cmap=mapa_cores,
        vmin=0,
        vmax=100,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={
            "label": rotulo_escala,
        },
        annot_kws={
            "fontsize": 8,
        },
        ax=eixo,
    )

    eixo.set_title(
        titulo
    )

    eixo.set_xlabel(
        "Ano da internação"
    )

    eixo.set_ylabel(
        "Indicador"
    )

    eixo.set_xticklabels(
        eixo.get_xticklabels(),
        rotation=0,
    )

    eixo.set_yticklabels(
        eixo.get_yticklabels(),
        rotation=0,
    )

    figura.tight_layout()

    plt.show()

    plt.close(
        figura
    )

In [ ]:
matriz_sintomas_presenca, rotulos_sintomas = (
    preparar_matriz_clinica_heatmap(
        tabela_clinica=sintomas_por_ano,
        coluna_valor="percentual_sim_validos",
    )
)

exibir_heatmap_percentual(
    matriz=matriz_sintomas_presenca,
    titulo=(
        "Presença registrada dos sintomas por ano de internação"
    ),
    rotulo_escala=(
        "Resposta sim entre respostas válidas (%)"
    ),
    mapa_cores="YlOrRd",
)

In [ ]:
matriz_comorbidades_presenca, rotulos_comorbidades = (
    preparar_matriz_clinica_heatmap(
        tabela_clinica=comorbidades_por_ano,
        coluna_valor="percentual_sim_validos",
    )
)

exibir_heatmap_percentual(
    matriz=matriz_comorbidades_presenca,
    titulo=(
        "Presença registrada das comorbidades por ano de internação"
    ),
    rotulo_escala=(
        "Resposta sim entre respostas válidas (%)"
    ),
    mapa_cores="YlGnBu",
)

In [ ]:
matriz_sintomas_sem_resposta, rotulos_sintomas_sem_resposta = (
    preparar_matriz_clinica_heatmap(
        tabela_clinica=sintomas_por_ano,
        coluna_valor=(
            "percentual_sem_resposta_valida"
        ),
    )
)

matriz_comorbidades_sem_resposta, rotulos_comorbidades_sem_resposta = (
    preparar_matriz_clinica_heatmap(
        tabela_clinica=comorbidades_por_ano,
        coluna_valor=(
            "percentual_sem_resposta_valida"
        ),
    )
)

exibir_heatmap_percentual(
    matriz=matriz_sintomas_sem_resposta,
    titulo=(
        "Sintomas sem resposta válida por ano de internação"
    ),
    rotulo_escala=(
        "Ausente, ignorado ou inesperado (%)"
    ),
    mapa_cores="Purples",
)

exibir_heatmap_percentual(
    matriz=matriz_comorbidades_sem_resposta,
    titulo=(
        "Comorbidades sem resposta válida por ano de internação"
    ),
    rotulo_escala=(
        "Ausente, ignorado ou inesperado (%)"
    ),
    mapa_cores="Purples",
)

In [ ]:
COLUNAS_CONTADORES_GRAFICO = [
    "ano_internacao",
    "sintomas_confirmados_media",
    "sintomas_avaliados_media",
    "comorbidades_confirmadas_media",
    "comorbidades_avaliadas_media",
    "percentual_multimorbidade",
]

colunas_contadores_ausentes = [
    coluna
    for coluna in COLUNAS_CONTADORES_GRAFICO
    if coluna not in contadores_clinicos_por_ano.columns
]

if colunas_contadores_ausentes:
    raise KeyError(
        "A tabela de contadores clínicos não contém: "
        f"{colunas_contadores_ausentes}"
    )

dados_contadores_grafico = (
    contadores_clinicos_por_ano[
        COLUNAS_CONTADORES_GRAFICO
    ]
    .sort_values(
        "ano_internacao"
    )
    .copy()
)

anos_contadores = (
    dados_contadores_grafico[
        "ano_internacao"
    ]
    .astype(int)
    .to_numpy()
)

figura_contadores, eixos_contadores = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(
        12,
        11,
    ),
    sharex=True,
)

eixo_sintomas = (
    eixos_contadores[0]
)

eixo_sintomas.plot(
    anos_contadores,
    dados_contadores_grafico[
        "sintomas_confirmados_media"
    ],
    marker="o",
    linewidth=2,
    label="Sintomas confirmados",
)

eixo_sintomas.plot(
    anos_contadores,
    dados_contadores_grafico[
        "sintomas_avaliados_media"
    ],
    marker="o",
    linewidth=2,
    label="Sintomas avaliados",
)

eixo_sintomas.set_title(
    "Média de sintomas confirmados e avaliados por registro"
)

eixo_sintomas.set_ylabel(
    "Quantidade média"
)

eixo_sintomas.legend()

eixo_sintomas.grid(
    axis="y",
    alpha=0.25,
)

eixo_comorbidades = (
    eixos_contadores[1]
)

eixo_comorbidades.plot(
    anos_contadores,
    dados_contadores_grafico[
        "comorbidades_confirmadas_media"
    ],
    marker="o",
    linewidth=2,
    color="tab:blue",
    label="Comorbidades confirmadas",
)

eixo_comorbidades.plot(
    anos_contadores,
    dados_contadores_grafico[
        "comorbidades_avaliadas_media"
    ],
    marker="o",
    linewidth=2,
    color="tab:green",
    label="Comorbidades avaliadas",
)

eixo_comorbidades.set_title(
    "Média de comorbidades confirmadas e avaliadas por registro"
)

eixo_comorbidades.set_xlabel(
    "Ano da internação"
)

eixo_comorbidades.set_ylabel(
    "Quantidade média"
)

eixo_comorbidades.set_xticks(
    anos_contadores
)

eixo_comorbidades.legend(
    loc="upper left"
)

eixo_comorbidades.grid(
    axis="y",
    alpha=0.25,
)

eixo_multimorbidade = (
    eixo_comorbidades.twinx()
)

eixo_multimorbidade.plot(
    anos_contadores,
    dados_contadores_grafico[
        "percentual_multimorbidade"
    ],
    marker="s",
    linewidth=2,
    linestyle="--",
    color="tab:red",
    label="Multimorbidade",
)

eixo_multimorbidade.set_ylabel(
    "Registros com duas ou mais comorbidades"
)

eixo_multimorbidade.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

linhas_comorbidades, rotulos_comorbidades_grafico = (
    eixo_comorbidades.get_legend_handles_labels()
)

linhas_multimorbidade, rotulos_multimorbidade = (
    eixo_multimorbidade.get_legend_handles_labels()
)

eixo_comorbidades.legend(
    linhas_comorbidades
    + linhas_multimorbidade,
    rotulos_comorbidades_grafico
    + rotulos_multimorbidade,
    loc="upper left",
)

figura_contadores.suptitle(
    "Evolução anual da quantidade de indicadores clínicos registrados",
    fontsize=14,
)

figura_contadores.text(
    0.01,
    0.01,
    (
        "Os contadores distinguem condições confirmadas de campos "
        "efetivamente avaliados. Maior número avaliado pode refletir "
        "mudança de preenchimento, e não somente mudança clínica."
    ),
    fontsize=8,
)

figura_contadores.tight_layout(
    rect=[
        0,
        0.04,
        1,
        0.97,
    ]
)

plt.show()

plt.close(
    figura_contadores
)

In [ ]:
def criar_tabela_obito_categoria_ano(
    dataframe: pd.DataFrame,
    coluna_categoria: str,
    nome_categoria_saida: str,
) -> pd.DataFrame:
    """Calcula quantidade, proporção de óbito e intervalo de confiança de Wilson por categoria e ano de internação."""

    COLUNAS_NECESSARIAS = [
        "ano_internacao",
        "alvo_obito_srag",
        coluna_categoria,
    ]

    colunas_ausentes = [
        coluna
        for coluna in COLUNAS_NECESSARIAS
        if coluna not in dataframe.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            "A base não contém as colunas necessárias: "
            f"{colunas_ausentes}"
        )

    tabela = (
        dataframe.groupby(
            [
                "ano_internacao",
                coluna_categoria,
            ],
            dropna=False,
        )
        .agg(
            quantidade_total=(
                "alvo_obito_srag",
                "size",
            ),
            quantidade_obito_srag=(
                "alvo_obito_srag",
                "sum",
            ),
        )
        .reset_index()
        .rename(
            columns={
                coluna_categoria: (
                    nome_categoria_saida
                ),
            }
        )
    )

    tabela[
        "quantidade_cura"
    ] = (
        tabela[
            "quantidade_total"
        ]
        .sub(
            tabela[
                "quantidade_obito_srag"
            ]
        )
    )

    tabela[
        "percentual_obito_srag"
    ] = (
        tabela[
            "quantidade_obito_srag"
        ]
        .div(
            tabela[
                "quantidade_total"
            ]
            .replace(
                0,
                np.nan,
            )
        )
        .mul(100)
    )

    intervalos = []

    for _, linha in tabela.iterrows():
        intervalo = calcular_ic_wilson(
            quantidade_eventos=int(
                linha[
                    "quantidade_obito_srag"
                ]
            ),
            quantidade_total=int(
                linha[
                    "quantidade_total"
                ]
            ),
        )

        intervalos.append(
            intervalo
        )

    tabela[
        "ic95_inferior"
    ] = [
        intervalo[0]
        for intervalo in intervalos
    ]

    tabela[
        "ic95_superior"
    ] = [
        intervalo[1]
        for intervalo in intervalos
    ]

    tabela[
        "ano_internacao"
    ] = (
        tabela[
            "ano_internacao"
        ]
        .astype("Int64")
    )

    tabela[
        "quantidade_total"
    ] = (
        tabela[
            "quantidade_total"
        ]
        .astype("int64")
    )

    tabela[
        "quantidade_obito_srag"
    ] = (
        tabela[
            "quantidade_obito_srag"
        ]
        .astype("int64")
    )

    tabela[
        "quantidade_cura"
    ] = (
        tabela[
            "quantidade_cura"
        ]
        .astype("int64")
    )

    percentuais_validos = (
        tabela[
            "percentual_obito_srag"
        ]
        .dropna()
        .between(
            0,
            100,
            inclusive="both",
        )
        .all()
    )

    if not percentuais_validos:
        raise RuntimeError(
            "Foram produzidos percentuais de óbito "
            "fora do intervalo de 0% a 100%."
        )

    total_tabela = int(
        tabela[
            "quantidade_total"
        ]
        .sum()
    )

    if total_tabela != len(
        dataframe
    ):
        raise RuntimeError(
            "A soma dos denominadores da tabela por categoria "
            "e ano não corresponde ao total da base analisada."
        )

    tabela = (
        tabela.sort_values(
            [
                "ano_internacao",
                nome_categoria_saida,
            ]
        )
        .reset_index(drop=True)
    )

    return tabela

In [ ]:
if "faixa_etaria" not in base_analise_desfecho.columns:
    raise KeyError(
        "A coluna faixa_etaria não está disponível. "
        "Execute primeiro a seção demográfica."
    )

obito_faixa_etaria_ano = (
    criar_tabela_obito_categoria_ano(
        dataframe=base_analise_desfecho,
        coluna_categoria="faixa_etaria",
        nome_categoria_saida="faixa_etaria",
    )
)

ordem_faixas_disponiveis = [
    faixa
    for faixa in ORDEM_FAIXA_OBITO
    if faixa in set(
        obito_faixa_etaria_ano[
            "faixa_etaria"
        ]
        .astype("string")
        .tolist()
    )
]

matriz_obito_faixa_etaria_ano = (
    obito_faixa_etaria_ano.pivot(
        index="faixa_etaria",
        columns="ano_internacao",
        values="percentual_obito_srag",
    )
    .reindex(
        ordem_faixas_disponiveis
    )
    .sort_index(
        axis=1
    )
)

matriz_n_faixa_etaria_ano = (
    obito_faixa_etaria_ano.pivot(
        index="faixa_etaria",
        columns="ano_internacao",
        values="quantidade_total",
    )
    .reindex(
        ordem_faixas_disponiveis
    )
    .reindex(
        columns=(
            matriz_obito_faixa_etaria_ano.columns
        )
    )
)

if matriz_obito_faixa_etaria_ano.empty:
    raise ValueError(
        "A matriz de óbito por faixa etária e ano está vazia."
    )

valores_obito_faixa = (
    matriz_obito_faixa_etaria_ano
    .stack(
        future_stack=True
    )
    .dropna()
)

if valores_obito_faixa.empty:
    raise ValueError(
        "A matriz não contém percentuais de óbito válidos."
    )

if not valores_obito_faixa.between(
    0,
    100,
    inclusive="both",
).all():
    raise RuntimeError(
        "A matriz contém percentuais fora "
        "do intervalo de 0% a 100%."
    )

if valores_obito_faixa.eq(0).all():
    raise RuntimeError(
        "Todos os percentuais de óbito ficaram iguais a zero. "
        "Verifique o cálculo de percentual_obito_srag."
    )

valor_maximo_observado = float(
    valores_obito_faixa.max()
)

limite_superior_escala = max(
    10.0,
    np.ceil(
        valor_maximo_observado
        / 5
    )
    * 5,
)

anotacoes_heatmap = (
    matriz_obito_faixa_etaria_ano
    .copy()
    .astype("object")
)

for faixa in matriz_obito_faixa_etaria_ano.index:
    for ano in matriz_obito_faixa_etaria_ano.columns:
        percentual = (
            matriz_obito_faixa_etaria_ano.loc[
                faixa,
                ano,
            ]
        )

        quantidade = (
            matriz_n_faixa_etaria_ano.loc[
                faixa,
                ano,
            ]
        )

        if pd.isna(
            percentual
        ):
            anotacao = ""
        else:
            percentual_formatado = (
                f"{float(percentual):.1f}%"
            )

            if pd.isna(
                quantidade
            ):
                anotacao = percentual_formatado
            else:
                quantidade_formatada = (
                    f"{int(quantidade):,}"
                    .replace(
                        ",",
                        ".",
                    )
                )

                anotacao = (
                    f"{percentual_formatado}\n"
                    f"n={quantidade_formatada}"
                )

        anotacoes_heatmap.loc[
            faixa,
            ano,
        ] = anotacao

matriz_obito_faixa_etaria_ano_plot = pd.DataFrame(
    matriz_obito_faixa_etaria_ano.to_numpy(
        dtype=np.float64,
        na_value=np.nan,
    ),
    index=matriz_obito_faixa_etaria_ano.index.copy(),
    columns=matriz_obito_faixa_etaria_ano.columns.copy(),
)

anotacoes_heatmap_plot = pd.DataFrame(
    anotacoes_heatmap.to_numpy(
        dtype=object,
        copy=True,
    ),
    index=anotacoes_heatmap.index.copy(),
    columns=anotacoes_heatmap.columns.copy(),
)

if (
    matriz_obito_faixa_etaria_ano_plot.shape
    != anotacoes_heatmap_plot.shape
):
    raise RuntimeError(
        "A matriz numérica e a matriz de anotações "
        "possuem dimensões diferentes."
    )

if not matriz_obito_faixa_etaria_ano_plot.index.equals(
    anotacoes_heatmap_plot.index
):
    raise RuntimeError(
        "A matriz numérica e a matriz de anotações "
        "possuem índices diferentes."
    )

if not matriz_obito_faixa_etaria_ano_plot.columns.equals(
    anotacoes_heatmap_plot.columns
):
    raise RuntimeError(
        "A matriz numérica e a matriz de anotações "
        "possuem colunas diferentes."
    )

valores_finitos_heatmap = (
    matriz_obito_faixa_etaria_ano_plot
    .to_numpy(
        dtype=np.float64,
        copy=False,
    )
)

mascara_valores_validos_heatmap = np.isfinite(
    valores_finitos_heatmap
)

if not mascara_valores_validos_heatmap.any():
    raise ValueError(
        "A matriz do heatmap não contém nenhum "
        "percentual numérico válido."
    )

valores_validos_heatmap = (
    valores_finitos_heatmap[
        mascara_valores_validos_heatmap
    ]
)

if not (
    (
        valores_validos_heatmap
        >= 0
    )
    & (
        valores_validos_heatmap
        <= 100
    )
).all():
    raise RuntimeError(
        "A matriz do heatmap contém percentuais "
        "fora do intervalo de 0% a 100%."
    )

mascara_heatmap = (
    matriz_obito_faixa_etaria_ano_plot
    .isna()
)

figura_obito_faixa_ano, eixo_obito_faixa_ano = plt.subplots(
    figsize=(
        14,
        8,
    )
)

sns.heatmap(
    matriz_obito_faixa_etaria_ano_plot,
    mask=mascara_heatmap,
    annot=anotacoes_heatmap_plot,
    fmt="",
    cmap="Reds",
    vmin=0,
    vmax=limite_superior_escala,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Óbito por SRAG (%)",
    },
    annot_kws={
        "fontsize": 8,
    },
    ax=eixo_obito_faixa_ano,
)

eixo_obito_faixa_ano.set_title(
    "Óbito por SRAG segundo faixa etária e ano de internação"
)

eixo_obito_faixa_ano.set_xlabel(
    "Ano da internação"
)

eixo_obito_faixa_ano.set_ylabel(
    "Faixa etária"
)

eixo_obito_faixa_ano.set_xticklabels(
    eixo_obito_faixa_ano.get_xticklabels(),
    rotation=0,
)

eixo_obito_faixa_ano.set_yticklabels(
    eixo_obito_faixa_ano.get_yticklabels(),
    rotation=0,
)

figura_obito_faixa_ano.text(
    0.01,
    0.01,
    (
        "Cada célula apresenta o percentual de óbito e o "
        "denominador da respectiva faixa etária e ano."
    ),
    fontsize=8,
)

figura_obito_faixa_ano.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1,
    ]
)

plt.show()

plt.close(
    figura_obito_faixa_ano
)

display(
    obito_faixa_etaria_ano.head(30)
)

In [ ]:
if (
    "grupo_numero_comorbidades"
    not in base_analise_desfecho.columns
):
    raise KeyError(
        "A coluna grupo_numero_comorbidades não está disponível. "
        "Execute primeiro a análise de óbito por comorbidades."
    )

obito_numero_comorbidades_ano = (
    criar_tabela_obito_categoria_ano(
        dataframe=base_analise_desfecho,
        coluna_categoria=(
            "grupo_numero_comorbidades"
        ),
        nome_categoria_saida=(
            "grupo_numero_comorbidades"
        ),
    )
)

grupos_comorbidades_disponiveis = [
    grupo
    for grupo in ORDEM_NUMERO_COMORBIDADES
    if grupo in set(
        obito_numero_comorbidades_ano[
            "grupo_numero_comorbidades"
        ]
        .astype("string")
        .tolist()
    )
]

figura_obito_comorbidades_ano, eixo_obito_comorbidades_ano = plt.subplots(
    figsize=(
        12,
        7,
    )
)

for grupo in grupos_comorbidades_disponiveis:
    dados_grupo = (
        obito_numero_comorbidades_ano.loc[
            obito_numero_comorbidades_ano[
                "grupo_numero_comorbidades"
            ]
            .astype("string")
            .eq(
                grupo
            )
        ]
        .sort_values(
            "ano_internacao"
        )
    )

    eixo_obito_comorbidades_ano.plot(
        dados_grupo[
            "ano_internacao"
        ]
        .astype(int),
        dados_grupo[
            "percentual_obito_srag"
        ],
        marker="o",
        linewidth=2,
        label=grupo,
    )

anos_comorbidades_obito = (
    obito_numero_comorbidades_ano[
        "ano_internacao"
    ]
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
)

eixo_obito_comorbidades_ano.set_title(
    "Óbito por SRAG segundo número de comorbidades e ano"
)

eixo_obito_comorbidades_ano.set_xlabel(
    "Ano da internação"
)

eixo_obito_comorbidades_ano.set_ylabel(
    "Óbito por SRAG"
)

eixo_obito_comorbidades_ano.set_xticks(
    anos_comorbidades_obito
)

eixo_obito_comorbidades_ano.set_ylim(
    bottom=0
)

eixo_obito_comorbidades_ano.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

eixo_obito_comorbidades_ano.legend(
    title="Comorbidades confirmadas",
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)

eixo_obito_comorbidades_ano.grid(
    axis="y",
    alpha=0.25,
)

figura_obito_comorbidades_ano.text(
    0.01,
    0.01,
    (
        "As linhas mostram a proporção observada de óbito em cada "
        "grupo. As associações são brutas e descritivas."
    ),
    fontsize=8,
)

figura_obito_comorbidades_ano.tight_layout(
    rect=[
        0,
        0.04,
        0.8,
        1,
    ]
)

plt.show()

plt.close(
    figura_obito_comorbidades_ano
)

display(
    obito_numero_comorbidades_ano.head(20)
)

In [ ]:
def exibir_letalidade_sim_nao(
    tabela: pd.DataFrame,
    titulo: str,
    quantidade_indicadores: int,
) -> pd.DataFrame:
    """Exibe a letalidade observada para respostas sim e não."""

    COLUNAS_NECESSARIAS = [
        "coluna",
        "nome_amigavel",
        "quantidade_sim",
        "percentual_obito_sim",
        "quantidade_nao",
        "percentual_obito_nao",
    ]

    colunas_ausentes = [
        coluna
        for coluna in COLUNAS_NECESSARIAS
        if coluna not in tabela.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            "A tabela não contém as colunas necessárias: "
            f"{colunas_ausentes}"
        )

    dados_grafico = (
        tabela[
            COLUNAS_NECESSARIAS
        ]
        .dropna(
            subset=[
                "percentual_obito_sim",
                "percentual_obito_nao",
            ]
        )
        .copy()
    )

    dados_grafico[
        "diferenca_absoluta_pp"
    ] = (
        dados_grafico[
            "percentual_obito_sim"
        ]
        .sub(
            dados_grafico[
                "percentual_obito_nao"
            ]
        )
        .abs()
    )

    dados_grafico = (
        dados_grafico.sort_values(
            "diferenca_absoluta_pp",
            ascending=False,
        )
        .head(
            quantidade_indicadores
        )
        .sort_values(
            "diferenca_absoluta_pp",
            ascending=True,
        )
        .reset_index(drop=True)
    )

    if dados_grafico.empty:
        raise ValueError(
            "Não existem indicadores válidos para o gráfico."
        )

    dados_grafico[
        "rotulo"
    ] = (
        dados_grafico[
            "nome_amigavel"
        ]
        .astype("string")
        .str.strip()
    )

    mascara_rotulo_vazio = (
        dados_grafico[
            "rotulo"
        ]
        .eq("")
        | dados_grafico[
            "rotulo"
        ]
        .isna()
    )

    dados_grafico.loc[
        mascara_rotulo_vazio,
        "rotulo",
    ] = dados_grafico.loc[
        mascara_rotulo_vazio,
        "coluna",
    ]

    posicoes = np.arange(
        len(
            dados_grafico
        )
    )

    altura_barra = 0.36

    figura, eixo = plt.subplots(
        figsize=(
            12,
            8,
        )
    )

    barras_sim = eixo.barh(
        posicoes
        + altura_barra
        / 2,
        dados_grafico[
            "percentual_obito_sim"
        ],
        height=altura_barra,
        color="tab:red",
        label="Resposta sim",
    )

    barras_nao = eixo.barh(
        posicoes
        - altura_barra
        / 2,
        dados_grafico[
            "percentual_obito_nao"
        ],
        height=altura_barra,
        color="tab:blue",
        label="Resposta não",
    )

    eixo.set_yticks(
        posicoes
    )

    eixo.set_yticklabels(
        dados_grafico[
            "rotulo"
        ]
    )

    eixo.set_title(
        titulo
    )

    eixo.set_xlabel(
        "Óbito por SRAG"
    )

    eixo.set_ylabel(
        "Indicador"
    )

    eixo.xaxis.set_major_formatter(
        plt.FuncFormatter(
            lambda valor, posicao: (
                f"{valor:.0f}%"
            )
        )
    )

    eixo.legend()

    eixo.grid(
        axis="x",
        alpha=0.25,
    )

    for barras in [
        barras_sim,
        barras_nao,
    ]:
        for barra in barras:
            valor = (
                barra.get_width()
            )

            eixo.text(
                valor,
                barra.get_y()
                + barra.get_height()
                / 2,
                f"{valor:.1f}%",
                va="center",
                ha="left",
                fontsize=8,
            )

    figura.text(
        0.01,
        0.01,
        (
            "Denominadores específicos de cada resposta estão "
            "disponíveis na tabela exibida abaixo do gráfico."
        ),
        fontsize=8,
    )

    figura.tight_layout(
        rect=[
            0,
            0.04,
            1,
            1,
        ]
    )

    plt.show()

    plt.close(
        figura
    )

    return dados_grafico

In [ ]:
letalidade_sintomas_grafico = (
    exibir_letalidade_sim_nao(
        tabela=obito_sintomas,
        titulo=(
            "Óbito por SRAG segundo presença ou ausência "
            "dos principais sintomas"
        ),
        quantidade_indicadores=10,
    )
)

display(
    letalidade_sintomas_grafico[
        [
            "coluna",
            "rotulo",
            "quantidade_sim",
            "percentual_obito_sim",
            "quantidade_nao",
            "percentual_obito_nao",
            "diferenca_absoluta_pp",
        ]
    ]
)

In [ ]:
letalidade_comorbidades_grafico = (
    exibir_letalidade_sim_nao(
        tabela=obito_comorbidades,
        titulo=(
            "Óbito por SRAG segundo presença ou ausência "
            "das principais comorbidades"
        ),
        quantidade_indicadores=10,
    )
)

display(
    letalidade_comorbidades_grafico[
        [
            "coluna",
            "rotulo",
            "quantidade_sim",
            "percentual_obito_sim",
            "quantidade_nao",
            "percentual_obito_nao",
            "diferenca_absoluta_pp",
        ]
    ]
)

In [ ]:
if "ORDEM_SEXO" not in globals():
    raise NameError(
        "A ordem centralizada das categorias de sexo não foi criada. "
        "Execute primeiro a seção demográfica."
    )

if not ORDEM_SEXO:
    raise ValueError(
        "A ordem centralizada das categorias de sexo está vazia."
    )

PALETA_SEXO = [
    "tab:blue",
    "tab:orange",
    "tab:gray",
    "tab:red",
    "tab:purple",
    "tab:brown",
]

if len(ORDEM_SEXO) > len(PALETA_SEXO):
    raise ValueError(
        "A quantidade de categorias de sexo excede a paleta disponível: "
        f"{len(ORDEM_SEXO)} categorias para {len(PALETA_SEXO)} cores."
    )

CORES_SEXO = {
    categoria: PALETA_SEXO[posicao]
    for posicao, categoria in enumerate(ORDEM_SEXO)
}

COLUNAS_SEXO_NECESSARIAS = [
    "sexo_categoria",
]

colunas_sexo_ausentes_coorte = [
    coluna
    for coluna in COLUNAS_SEXO_NECESSARIAS
    if coluna not in coorte.columns
]

if colunas_sexo_ausentes_coorte:
    raise KeyError(
        "A coorte não contém as colunas necessárias "
        "para os gráficos de sexo: "
        f"{colunas_sexo_ausentes_coorte}"
    )

colunas_sexo_ausentes_desfecho = [
    coluna
    for coluna in COLUNAS_SEXO_NECESSARIAS
    if coluna not in base_analise_desfecho.columns
]

if colunas_sexo_ausentes_desfecho:
    raise KeyError(
        "A base da análise do desfecho não contém "
        "as colunas necessárias para os gráficos de sexo: "
        f"{colunas_sexo_ausentes_desfecho}"
    )

categorias_observadas_coorte = set(
    coorte[
        "sexo_categoria"
    ]
    .dropna()
    .astype("string")
    .tolist()
)

categorias_observadas_desfecho = set(
    base_analise_desfecho[
        "sexo_categoria"
    ]
    .dropna()
    .astype("string")
    .tolist()
)

categorias_sexo_presentes_coorte = [
    categoria
    for categoria in ORDEM_SEXO
    if categoria in categorias_observadas_coorte
]

categorias_sexo_presentes_desfecho = [
    categoria
    for categoria in ORDEM_SEXO
    if categoria in categorias_observadas_desfecho
]

categorias_nao_previstas_coorte = sorted(
    categoria
    for categoria in categorias_observadas_coorte
    if categoria not in ORDEM_SEXO
)

categorias_nao_previstas_desfecho = sorted(
    categoria
    for categoria in categorias_observadas_desfecho
    if categoria not in ORDEM_SEXO
)

if categorias_nao_previstas_coorte:
    raise ValueError(
        "Foram encontradas categorias de sexo não previstas "
        "na coorte: "
        f"{categorias_nao_previstas_coorte}"
    )

if categorias_nao_previstas_desfecho:
    raise ValueError(
        "Foram encontradas categorias de sexo não previstas "
        "na base da análise do desfecho: "
        f"{categorias_nao_previstas_desfecho}"
    )

if not categorias_sexo_presentes_coorte:
    raise ValueError(
        "Nenhuma categoria de sexo prevista foi encontrada "
        "na coorte."
    )

if not categorias_sexo_presentes_desfecho:
    raise ValueError(
        "Nenhuma categoria de sexo prevista foi encontrada "
        "na base da análise do desfecho."
    )

print(
    "Categorias presentes na coorte:",
    categorias_sexo_presentes_coorte,
)

print(
    "Categorias presentes na análise do desfecho:",
    categorias_sexo_presentes_desfecho,
)

In [ ]:
resumo_sexo_observado = (
    coorte.groupby(
        "sexo_categoria",
        dropna=False,
    )
    .agg(
        quantidade_registros=(
            "sexo_categoria",
            "size",
        ),
    )
    .reset_index()
)

resumo_sexo_observado[
    "percentual_coorte"
] = (
    resumo_sexo_observado[
        "quantidade_registros"
    ]
    .div(
        len(
            coorte
        )
    )
    .mul(100)
)

resumo_sexo_observado = (
    resumo_sexo_observado.sort_values(
        "quantidade_registros",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    resumo_sexo_observado
)

In [ ]:
sexo_grafico = (
    sexo_por_ano.pivot(
        index="ano_internacao",
        columns="sexo_categoria",
        values="percentual_ano",
    )
    .reindex(
        columns=(
            categorias_sexo_presentes_coorte
        )
    )
    .fillna(0)
    .sort_index()
)

cores_composicao_sexo = [
    CORES_SEXO[
        categoria
    ]
    for categoria in sexo_grafico.columns
]

figura_sexo_ano, eixo_sexo_ano = plt.subplots(
    figsize=(
        12,
        7,
    )
)

sexo_grafico.plot(
    kind="bar",
    stacked=True,
    width=0.85,
    color=cores_composicao_sexo,
    ax=eixo_sexo_ano,
)

eixo_sexo_ano.set_title(
    "Composição do sexo registrado por ano de internação"
)

eixo_sexo_ano.set_xlabel(
    "Ano da internação"
)

eixo_sexo_ano.set_ylabel(
    "Percentual dos registros do ano"
)

eixo_sexo_ano.set_ylim(
    0,
    100,
)

eixo_sexo_ano.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

eixo_sexo_ano.set_xticklabels(
    eixo_sexo_ano.get_xticklabels(),
    rotation=0,
)

eixo_sexo_ano.legend(
    title="Sexo registrado",
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)

eixo_sexo_ano.grid(
    axis="y",
    alpha=0.20,
)

figura_sexo_ano.text(
    0.01,
    0.01,
    (
        "Cada barra representa 100% dos registros elegíveis "
        "do respectivo ano. Categorias com participação muito "
        "pequena podem não ser visualmente perceptíveis."
    ),
    fontsize=8,
)

figura_sexo_ano.tight_layout(
    rect=[
        0,
        0.04,
        0.84,
        1,
    ]
)

plt.show()

plt.close(
    figura_sexo_ano
)

display(
    sexo_por_ano.loc[
        sexo_por_ano[
            "sexo_categoria"
        ]
        .isin(
            categorias_sexo_presentes_coorte
        )
    ]
    .sort_values(
        [
            "ano_internacao",
            "sexo_categoria",
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
obito_por_sexo = (
    criar_tabela_obito_categorica(
        dataframe=base_analise_desfecho,
        coluna_categoria="sexo_categoria",
        nome_categoria_saida="sexo_categoria",
    )
)

ordem_sexo_obito = {
    categoria: posicao
    for posicao, categoria in enumerate(
        ORDEM_SEXO
    )
}

obito_por_sexo[
    "ordem_sexo"
] = (
    obito_por_sexo[
        "sexo_categoria"
    ]
    .astype("string")
    .map(
        ordem_sexo_obito
    )
    .fillna(
        len(
            ordem_sexo_obito
        )
    )
)

obito_por_sexo = (
    obito_por_sexo.loc[
        obito_por_sexo[
            "sexo_categoria"
        ]
        .astype("string")
        .isin(
            categorias_sexo_presentes_desfecho
        )
    ]
    .sort_values(
        "ordem_sexo"
    )
    .drop(
        columns=[
            "ordem_sexo",
        ]
    )
    .reset_index(drop=True)
)

dados_obito_sexo_grafico = (
    obito_por_sexo.loc[
        obito_por_sexo[
            "quantidade_total"
        ].gt(0)
    ]
    .copy()
)

posicoes_sexo = np.arange(
    len(
        dados_obito_sexo_grafico
    )
)

percentuais_obito_sexo = (
    dados_obito_sexo_grafico[
        "percentual_obito_srag"
    ]
    .astype(float)
    .to_numpy()
)

erros_inferiores_sexo = (
    dados_obito_sexo_grafico[
        "percentual_obito_srag"
    ]
    .sub(
        dados_obito_sexo_grafico[
            "ic95_inferior"
        ]
    )
    .clip(
        lower=0
    )
    .astype(float)
    .to_numpy()
)

erros_superiores_sexo = (
    dados_obito_sexo_grafico[
        "ic95_superior"
    ]
    .sub(
        dados_obito_sexo_grafico[
            "percentual_obito_srag"
        ]
    )
    .clip(
        lower=0
    )
    .astype(float)
    .to_numpy()
)

cores_obito_sexo = [
    CORES_SEXO[
        categoria
    ]
    for categoria in dados_obito_sexo_grafico[
        "sexo_categoria"
    ]
]

figura_obito_sexo, eixo_obito_sexo = plt.subplots(
    figsize=(
        11,
        7,
    )
)

barras_obito_sexo = eixo_obito_sexo.bar(
    posicoes_sexo,
    percentuais_obito_sexo,
    color=cores_obito_sexo,
    width=0.72,
)

eixo_obito_sexo.errorbar(
    posicoes_sexo,
    percentuais_obito_sexo,
    yerr=[
        erros_inferiores_sexo,
        erros_superiores_sexo,
    ],
    fmt="none",
    ecolor="black",
    capsize=4,
)

eixo_obito_sexo.set_xticks(
    posicoes_sexo
)

eixo_obito_sexo.set_xticklabels(
    dados_obito_sexo_grafico[
        "sexo_categoria"
    ]
)

eixo_obito_sexo.set_title(
    "Óbito por SRAG segundo sexo registrado"
)

eixo_obito_sexo.set_xlabel(
    "Sexo registrado"
)

eixo_obito_sexo.set_ylabel(
    "Óbito por SRAG"
)

eixo_obito_sexo.set_ylim(
    bottom=0
)

eixo_obito_sexo.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

eixo_obito_sexo.grid(
    axis="y",
    alpha=0.20,
)

for barra, percentual, quantidade in zip(
    barras_obito_sexo,
    percentuais_obito_sexo,
    dados_obito_sexo_grafico[
        "quantidade_total"
    ],
):
    quantidade_formatada = (
        f"{int(quantidade):,}"
        .replace(
            ",",
            ".",
        )
    )

    eixo_obito_sexo.text(
        barra.get_x()
        + barra.get_width()
        / 2,
        barra.get_height(),
        (
            f"{percentual:.1f}%\n"
            f"n={quantidade_formatada}"
        ),
        ha="center",
        va="bottom",
        fontsize=9,
    )

figura_obito_sexo.text(
    0.01,
    0.01,
    (
        "Barras de erro: intervalo de confiança de Wilson de 95%. "
        "As categorias mantêm as mesmas cores utilizadas "
        "nos demais gráficos."
    ),
    fontsize=8,
)

figura_obito_sexo.tight_layout(
    rect=[
        0,
        0.05,
        1,
        1,
    ]
)

plt.show()

plt.close(
    figura_obito_sexo
)

display(
    obito_por_sexo
)

In [ ]:
COLUNAS_OBITO_SEXO_ANO_NECESSARIAS = [
    "ano_internacao",
    "sexo_categoria",
    "alvo_obito_srag",
]

colunas_obito_sexo_ano_ausentes = [
    coluna
    for coluna in COLUNAS_OBITO_SEXO_ANO_NECESSARIAS
    if coluna not in base_analise_desfecho.columns
]

if colunas_obito_sexo_ano_ausentes:
    raise KeyError(
        "A base da análise do desfecho não contém "
        "as colunas necessárias para calcular óbito "
        "por sexo e ano: "
        f"{colunas_obito_sexo_ano_ausentes}"
    )

if "categorias_sexo_presentes_desfecho" not in globals():
    raise NameError(
        "A lista categorias_sexo_presentes_desfecho "
        "não foi criada. Execute primeiro a célula de "
        "configuração das categorias de sexo."
    )

if not categorias_sexo_presentes_desfecho:
    raise ValueError(
        "Nenhuma categoria de sexo está disponível "
        "para a análise anual do desfecho."
    )

obito_sexo_ano = (
    criar_tabela_obito_categoria_ano(
        dataframe=base_analise_desfecho,
        coluna_categoria="sexo_categoria",
        nome_categoria_saida="sexo_categoria",
    )
)

obito_sexo_ano = (
    obito_sexo_ano.loc[
        obito_sexo_ano[
            "sexo_categoria"
        ]
        .astype("string")
        .isin(
            categorias_sexo_presentes_desfecho
        )
    ]
    .copy()
)

ordem_sexo_ano = {
    categoria: posicao
    for posicao, categoria in enumerate(
        ORDEM_SEXO
    )
}

obito_sexo_ano[
    "ordem_sexo"
] = (
    obito_sexo_ano[
        "sexo_categoria"
    ]
    .astype("string")
    .map(
        ordem_sexo_ano
    )
    .fillna(
        len(
            ordem_sexo_ano
        )
    )
)

obito_sexo_ano = (
    obito_sexo_ano
    .sort_values(
        [
            "ano_internacao",
            "ordem_sexo",
        ]
    )
    .drop(
        columns=[
            "ordem_sexo",
        ]
    )
    .reset_index(drop=True)
)

if obito_sexo_ano.empty:
    raise ValueError(
        "A tabela de óbito por sexo e ano está vazia."
    )

percentuais_obito_sexo_ano_validos = (
    obito_sexo_ano[
        "percentual_obito_srag"
    ]
    .dropna()
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

if not percentuais_obito_sexo_ano_validos:
    raise RuntimeError(
        "A tabela de óbito por sexo e ano contém "
        "percentuais fora do intervalo de 0% a 100%."
    )

categorias_obito_sexo_ano = [
    categoria
    for categoria in ORDEM_SEXO
    if categoria in set(
        obito_sexo_ano[
            "sexo_categoria"
        ]
        .astype("string")
        .tolist()
    )
]

if not categorias_obito_sexo_ano:
    raise ValueError(
        "Nenhuma categoria de sexo prevista foi encontrada "
        "na tabela anual do desfecho."
    )

figura_obito_sexo_ano, eixo_obito_sexo_ano = plt.subplots(
    figsize=(
        12,
        7,
    )
)

for categoria in categorias_obito_sexo_ano:
    dados_categoria = (
        obito_sexo_ano.loc[
            obito_sexo_ano[
                "sexo_categoria"
            ]
            .astype("string")
            .eq(
                categoria
            )
        ]
        .sort_values(
            "ano_internacao"
        )
        .copy()
    )

    if dados_categoria.empty:
        continue

    eixo_obito_sexo_ano.plot(
        dados_categoria[
            "ano_internacao"
        ]
        .astype(int),
        dados_categoria[
            "percentual_obito_srag"
        ],
        marker="o",
        linewidth=2,
        color=CORES_SEXO[
            categoria
        ],
        label=categoria,
    )

anos_obito_sexo = (
    obito_sexo_ano[
        "ano_internacao"
    ]
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
)

eixo_obito_sexo_ano.set_title(
    "Óbito por SRAG segundo sexo registrado e ano de internação"
)

eixo_obito_sexo_ano.set_xlabel(
    "Ano da internação"
)

eixo_obito_sexo_ano.set_ylabel(
    "Óbito por SRAG"
)

eixo_obito_sexo_ano.set_xticks(
    anos_obito_sexo
)

eixo_obito_sexo_ano.set_ylim(
    bottom=0
)

eixo_obito_sexo_ano.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda valor, posicao: (
            f"{valor:.0f}%"
        )
    )
)

eixo_obito_sexo_ano.legend(
    title="Sexo registrado",
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)

eixo_obito_sexo_ano.grid(
    axis="y",
    alpha=0.20,
)

figura_obito_sexo_ano.text(
    0.01,
    0.01,
    (
        "Cada linha apresenta a proporção observada de óbito "
        "dentro da respectiva categoria e ano. "
        "A comparação é bruta e descritiva."
    ),
    fontsize=8,
)

figura_obito_sexo_ano.tight_layout(
    rect=[
        0,
        0.04,
        0.84,
        1,
    ]
)

plt.show()

plt.close(
    figura_obito_sexo_ano
)

display(
    obito_sexo_ano
)

In [ ]:
COLUNAS_SERIE_MENSAL_NECESSARIAS = [
    "dt_interna_convertida",
    "ano_internacao",
    "alvo_obito_srag",
]

colunas_serie_mensal_ausentes = [
    coluna
    for coluna in COLUNAS_SERIE_MENSAL_NECESSARIAS
    if coluna not in coorte.columns
]

if colunas_serie_mensal_ausentes:
    raise KeyError(
        "A coorte não contém as colunas necessárias "
        "para a análise mensal: "
        f"{colunas_serie_mensal_ausentes}"
    )


data_internacao_mensal_arrow = pd.to_datetime(
    coorte[
        "dt_interna_convertida"
    ],
    errors="coerce",
)

data_internacao_mensal = pd.Series(
    data_internacao_mensal_arrow.to_numpy(
        dtype="datetime64[ns]",
        na_value=np.datetime64("NaT","ns"),
        copy=False,
    ),
    index=coorte.index,
    name="dt_interna_convertida",
    dtype="datetime64[ns]",
)

del data_internacao_mensal_arrow

quantidade_datas_mensais_ausentes = int(
    data_internacao_mensal
    .isna()
    .sum()
)

if quantidade_datas_mensais_ausentes > 0:
    raise ValueError(
        "Existem datas de internação ausentes ou inválidas "
        "na coorte analítica: "
        f"{quantidade_datas_mensais_ausentes}"
    )

if not coorte[
    "alvo_obito_srag"
].isin(
    [
        0,
        1,
    ]
).all():
    raise ValueError(
        "A coluna alvo_obito_srag contém valores "
        "diferentes de zero e um."
    )

ano_data_internacao = (
    data_internacao_mensal
    .dt.year
    .astype("Int64")
)

ano_coorte = (
    pd.to_numeric(
        coorte[
            "ano_internacao"
        ],
        errors="coerce",
    )
    .astype("Int64")
)

mascara_ano_divergente = (
    ano_data_internacao.ne(
        ano_coorte
    )
)

quantidade_anos_divergentes = int(
    mascara_ano_divergente
    .fillna(True)
    .sum()
)

if quantidade_anos_divergentes > 0:
    raise RuntimeError(
        "O ano extraído de dt_interna_convertida diverge "
        "de ano_internacao em "
        f"{quantidade_anos_divergentes} registros."
    )

coorte[
    "mes_internacao"
] = (
    data_internacao_mensal
    .dt.month
    .astype("Int8")
)

coorte[
    "ano_mes_internacao"
] = (
    data_internacao_mensal
    .dt.to_period("M")
    .dt.to_timestamp()
)

if not coorte[
    "mes_internacao"
].between(
    1,
    12,
    inclusive="both",
).all():
    raise RuntimeError(
        "Foram identificados meses fora do intervalo "
        "de janeiro a dezembro."
    )

data_minima_internacao = (
    data_internacao_mensal.min()
)

data_maxima_internacao = (
    data_internacao_mensal.max()
)

primeiro_mes_observado = (
    data_minima_internacao
    .to_period("M")
    .to_timestamp()
)

ultimo_mes_observado = (
    data_maxima_internacao
    .to_period("M")
    .to_timestamp()
)

ultimo_dia_mes_maximo = (
    ultimo_mes_observado
    + pd.offsets.MonthEnd(0)
)

ultimo_mes_completo = (
    data_maxima_internacao.normalize()
    >= ultimo_dia_mes_maximo.normalize()
)

calendario_mensal = pd.DataFrame(
    {
        "ano_mes_internacao": pd.date_range(
            start=primeiro_mes_observado,
            end=ultimo_mes_observado,
            freq="MS",
        ),
    }
)

calendario_mensal[
    "ano_internacao"
] = (
    calendario_mensal[
        "ano_mes_internacao"
    ]
    .dt.year
    .astype("Int64")
)

calendario_mensal[
    "mes_internacao"
] = (
    calendario_mensal[
        "ano_mes_internacao"
    ]
    .dt.month
    .astype("Int8")
)

calendario_mensal[
    "mes_observado"
] = True

calendario_mensal[
    "mes_completo"
] = True

if not ultimo_mes_completo:
    calendario_mensal.loc[
        calendario_mensal[
            "ano_mes_internacao"
        ].eq(
            ultimo_mes_observado
        ),
        "mes_completo",
    ] = False

contagem_alvo_mensal = (
    coorte.groupby(
        [
            "ano_mes_internacao",
            "alvo_obito_srag",
        ],
        dropna=False,
    )
    .size()
    .unstack(
        fill_value=0
    )
)

contagem_alvo_mensal = (
    contagem_alvo_mensal.reindex(
        columns=[
            0,
            1,
        ],
        fill_value=0,
    )
    .rename(
        columns={
            0: "quantidade_cura",
            1: "quantidade_obito_srag",
        }
    )
    .reset_index()
)

serie_mensal = (
    calendario_mensal.merge(
        contagem_alvo_mensal,
        on="ano_mes_internacao",
        how="left",
        validate="one_to_one",
    )
)

COLUNAS_CONTAGEM_MENSAL = [
    "quantidade_cura",
    "quantidade_obito_srag",
]

for coluna in COLUNAS_CONTAGEM_MENSAL:
    serie_mensal[
        coluna
    ] = (
        pd.to_numeric(
            serie_mensal[
                coluna
            ],
            errors="coerce",
        )
        .fillna(0)
        .astype("int64")
    )

serie_mensal[
    "quantidade_registros"
] = (
    serie_mensal[
        "quantidade_cura"
    ]
    .add(
        serie_mensal[
            "quantidade_obito_srag"
        ]
    )
    .astype("int64")
)

serie_mensal[
    "percentual_cura"
] = (
    serie_mensal[
        "quantidade_cura"
    ]
    .div(
        serie_mensal[
            "quantidade_registros"
        ]
        .replace(
            0,
            np.nan,
        )
    )
    .mul(100)
)

serie_mensal[
    "percentual_obito_srag"
] = (
    serie_mensal[
        "quantidade_obito_srag"
    ]
    .div(
        serie_mensal[
            "quantidade_registros"
        ]
        .replace(
            0,
            np.nan,
        )
    )
    .mul(100)
)

serie_mensal[
    "rotulo_ano_mes"
] = (
    serie_mensal[
        "ano_mes_internacao"
    ]
    .dt.strftime(
        "%Y-%m"
    )
)

serie_mensal = (
    serie_mensal[
        [
            "ano_mes_internacao",
            "rotulo_ano_mes",
            "ano_internacao",
            "mes_internacao",
            "mes_observado",
            "mes_completo",
            "quantidade_registros",
            "quantidade_cura",
            "quantidade_obito_srag",
            "percentual_cura",
            "percentual_obito_srag",
        ]
    ]
    .sort_values(
        "ano_mes_internacao"
    )
    .reset_index(drop=True)
)

total_registros_serie_mensal = int(
    serie_mensal[
        "quantidade_registros"
    ]
    .sum()
)

if total_registros_serie_mensal != len(
    coorte
):
    raise RuntimeError(
        "A soma mensal dos registros não corresponde "
        "ao total da coorte."
    )

total_obitos_serie_mensal = int(
    serie_mensal[
        "quantidade_obito_srag"
    ]
    .sum()
)

total_obitos_coorte = int(
    coorte[
        "alvo_obito_srag"
    ]
    .sum()
)

if total_obitos_serie_mensal != total_obitos_coorte:
    raise RuntimeError(
        "A soma mensal dos óbitos não corresponde "
        "ao total de óbitos da coorte."
    )

percentuais_mensais_validos = (
    serie_mensal[
        "percentual_obito_srag"
    ]
    .dropna()
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

if not percentuais_mensais_validos:
    raise RuntimeError(
        "A série mensal contém percentuais de óbito "
        "fora do intervalo de 0% a 100%."
    )

meses_sem_registros = (
    serie_mensal.loc[
        serie_mensal[
            "quantidade_registros"
        ].eq(0),
        "rotulo_ano_mes",
    ]
    .tolist()
)

print(
    "Primeiro mês observado:",
    primeiro_mes_observado.strftime(
        "%Y-%m"
    ),
)

print(
    "Último mês observado:",
    ultimo_mes_observado.strftime(
        "%Y-%m"
    ),
)

print(
    "Data máxima de internação:",
    data_maxima_internacao.strftime(
        "%Y-%m-%d"
    ),
)

print(
    "Último mês está completo:",
    bool(
        ultimo_mes_completo
    ),
)

print(
    "Quantidade de meses no calendário:",
    len(
        serie_mensal
    ),
)

print(
    "Meses sem registros:",
    len(
        meses_sem_registros
    ),
)

if meses_sem_registros:
    print(
        "Lista de meses sem registros:",
        meses_sem_registros,
    )

display(
    serie_mensal
)

In [ ]:
from matplotlib.ticker import FuncFormatter

JANELA_MEDIA_MOVEL_MESES = 3

if "serie_mensal" not in globals():
    raise NameError(
        "A tabela serie_mensal não foi criada. "
        "Execute primeiro a célula de construção da série mensal."
    )

COLUNAS_SERIE_GRAFICO_NECESSARIAS = [
    "ano_mes_internacao",
    "mes_completo",
    "quantidade_registros",
    "quantidade_obito_srag",
    "percentual_obito_srag",
]

colunas_serie_grafico_ausentes = [
    coluna
    for coluna in COLUNAS_SERIE_GRAFICO_NECESSARIAS
    if coluna not in serie_mensal.columns
]

if colunas_serie_grafico_ausentes:
    raise KeyError(
        "A série mensal não contém as colunas necessárias: "
        f"{colunas_serie_grafico_ausentes}"
    )

serie_mensal_grafico = (
    serie_mensal[
        COLUNAS_SERIE_GRAFICO_NECESSARIAS
    ]
    .copy()
    .sort_values(
        "ano_mes_internacao"
    )
    .reset_index(drop=True)
)

serie_mensal_grafico[
    "ano_mes_internacao"
] = pd.to_datetime(
    serie_mensal_grafico[
        "ano_mes_internacao"
    ],
    errors="coerce",
)

if serie_mensal_grafico[
    "ano_mes_internacao"
].isna().any():
    raise ValueError(
        "A série mensal contém referências mensais inválidas."
    )

serie_mensal_grafico[
    "quantidade_registros"
] = pd.to_numeric(
    serie_mensal_grafico[
        "quantidade_registros"
    ],
    errors="coerce",
)

serie_mensal_grafico[
    "quantidade_obito_srag"
] = pd.to_numeric(
    serie_mensal_grafico[
        "quantidade_obito_srag"
    ],
    errors="coerce",
)

if serie_mensal_grafico[
    [
        "quantidade_registros",
        "quantidade_obito_srag",
    ]
].isna().any().any():
    raise ValueError(
        "A série mensal contém contagens não numéricas."
    )

serie_mensal_grafico[
    "volume_media_movel_3_meses"
] = (
    serie_mensal_grafico[
        "quantidade_registros"
    ]
    .rolling(
        window=JANELA_MEDIA_MOVEL_MESES,
        min_periods=1,
    )
    .mean()
)

obitos_moveis_3_meses = (
    serie_mensal_grafico[
        "quantidade_obito_srag"
    ]
    .rolling(
        window=JANELA_MEDIA_MOVEL_MESES,
        min_periods=1,
    )
    .sum()
)

registros_moveis_3_meses = (
    serie_mensal_grafico[
        "quantidade_registros"
    ]
    .rolling(
        window=JANELA_MEDIA_MOVEL_MESES,
        min_periods=1,
    )
    .sum()
)

serie_mensal_grafico[
    "obito_media_movel_3_meses"
] = (
    obitos_moveis_3_meses
    .div(
        registros_moveis_3_meses
        .replace(
            0,
            np.nan,
        )
    )
    .mul(100)
)

percentuais_moveis_validos = (
    serie_mensal_grafico[
        "obito_media_movel_3_meses"
    ]
    .dropna()
    .between(
        0,
        100,
        inclusive="both",
    )
    .all()
)

if not percentuais_moveis_validos:
    raise RuntimeError(
        "A média móvel do óbito contém valores "
        "fora do intervalo de 0% a 100%."
    )

meses_incompletos = (
    serie_mensal_grafico.loc[
        serie_mensal_grafico[
            "mes_completo"
        ].eq(False),
        "ano_mes_internacao",
    ]
    .tolist()
)

figura_serie_mensal, eixos_serie_mensal = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(
        15,
        11,
    ),
    sharex=True,
)

eixo_volume_mensal = eixos_serie_mensal[0]

eixo_volume_mensal.plot(
    serie_mensal_grafico[
        "ano_mes_internacao"
    ],
    serie_mensal_grafico[
        "quantidade_registros"
    ],
    color="tab:blue",
    linewidth=1,
    marker="o",
    markersize=2.5,
    alpha=0.35,
    label="Volume mensal",
)

eixo_volume_mensal.plot(
    serie_mensal_grafico[
        "ano_mes_internacao"
    ],
    serie_mensal_grafico[
        "volume_media_movel_3_meses"
    ],
    color="tab:blue",
    linewidth=2.5,
    label="Média móvel de 3 meses",
)

eixo_volume_mensal.set_title(
    "Volume mensal de internações elegíveis"
)

eixo_volume_mensal.set_ylabel(
    "Quantidade de registros"
)

eixo_volume_mensal.ticklabel_format(
    style="plain",
    axis="y",
)

eixo_volume_mensal.grid(
    axis="y",
    alpha=0.20,
)

eixo_volume_mensal.legend(
    loc="upper right"
)

eixo_obito_mensal = eixos_serie_mensal[1]

eixo_obito_mensal.plot(
    serie_mensal_grafico[
        "ano_mes_internacao"
    ],
    serie_mensal_grafico[
        "percentual_obito_srag"
    ],
    color="tab:red",
    linewidth=1,
    marker="o",
    markersize=2.5,
    alpha=0.35,
    label="Óbito mensal",
)

eixo_obito_mensal.plot(
    serie_mensal_grafico[
        "ano_mes_internacao"
    ],
    serie_mensal_grafico[
        "obito_media_movel_3_meses"
    ],
    color="tab:red",
    linewidth=2.5,
    label="Média móvel de 3 meses",
)

eixo_obito_mensal.set_title(
    "Proporção mensal observada de óbito por SRAG"
)

eixo_obito_mensal.set_xlabel(
    "Mês da internação"
)

eixo_obito_mensal.set_ylabel(
    "Óbito por SRAG"
)

eixo_obito_mensal.set_ylim(
    bottom=0
)

eixo_obito_mensal.yaxis.set_major_formatter(
    FuncFormatter(
        lambda valor, posicao: f"{valor:.0f}%"
    )
)

eixo_obito_mensal.grid(
    axis="y",
    alpha=0.20,
)

eixo_obito_mensal.legend(
    loc="upper right"
)

for mes_incompleto in meses_incompletos:
    inicio_mes_incompleto = pd.Timestamp(
        mes_incompleto
    )

    fim_mes_incompleto = (
        inicio_mes_incompleto
        + pd.offsets.MonthEnd(1)
    )

    eixo_volume_mensal.axvspan(
        inicio_mes_incompleto,
        fim_mes_incompleto,
        color="tab:orange",
        alpha=0.15,
    )

    eixo_obito_mensal.axvspan(
        inicio_mes_incompleto,
        fim_mes_incompleto,
        color="tab:orange",
        alpha=0.15,
    )

figura_serie_mensal.suptitle(
    "Evolução mensal do volume e do desfecho",
    fontsize=15,
)

figura_serie_mensal.text(
    0.01,
    0.01,
    (
        "As linhas finas representam os valores mensais. "
        "As linhas espessas representam médias móveis de três meses. "
        "Meses incompletos são destacados em laranja."
    ),
    fontsize=8,
)

figura_serie_mensal.tight_layout(
    rect=(
        0.0,
        0.04,
        1.0,
        0.97,
    )
)

plt.show()

plt.close(
    figura_serie_mensal
)

display(
    serie_mensal_grafico[
        [
            "ano_mes_internacao",
            "mes_completo",
            "quantidade_registros",
            "volume_media_movel_3_meses",
            "percentual_obito_srag",
            "obito_media_movel_3_meses",
        ]
    ]
)

In [ ]:
ORDEM_MESES = [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
]

ROTULOS_MESES = [
    "Jan",
    "Fev",
    "Mar",
    "Abr",
    "Mai",
    "Jun",
    "Jul",
    "Ago",
    "Set",
    "Out",
    "Nov",
    "Dez",
]

COLUNAS_MATRIZES_MENSAIS_NECESSARIAS = [
    "ano_internacao",
    "mes_internacao",
    "mes_completo",
    "quantidade_registros",
    "quantidade_obito_srag",
    "percentual_obito_srag",
]

colunas_matrizes_mensais_ausentes = [
    coluna
    for coluna in COLUNAS_MATRIZES_MENSAIS_NECESSARIAS
    if coluna not in serie_mensal.columns
]

if colunas_matrizes_mensais_ausentes:
    raise KeyError(
        "A série mensal não contém as colunas necessárias: "
        f"{colunas_matrizes_mensais_ausentes}"
    )

dados_matrizes_mensais = (
    serie_mensal[
        COLUNAS_MATRIZES_MENSAIS_NECESSARIAS
    ]
    .copy()
)

dados_matrizes_mensais[
    "ano_internacao"
] = pd.to_numeric(
    dados_matrizes_mensais[
        "ano_internacao"
    ],
    errors="coerce",
).astype("Int64")

dados_matrizes_mensais[
    "mes_internacao"
] = pd.to_numeric(
    dados_matrizes_mensais[
        "mes_internacao"
    ],
    errors="coerce",
).astype("Int64")

if dados_matrizes_mensais[
    [
        "ano_internacao",
        "mes_internacao",
    ]
].isna().any().any():
    raise ValueError(
        "A série mensal contém ano ou mês inválido."
    )

if not dados_matrizes_mensais[
    "mes_internacao"
].between(
    1,
    12,
    inclusive="both",
).all():
    raise RuntimeError(
        "A série mensal contém mês fora do intervalo "
        "de janeiro a dezembro."
    )

matriz_volume_ano_mes = (
    dados_matrizes_mensais.pivot(
        index="ano_internacao",
        columns="mes_internacao",
        values="quantidade_registros",
    )
    .reindex(
        columns=ORDEM_MESES
    )
    .sort_index()
)

matriz_obito_ano_mes = (
    dados_matrizes_mensais.pivot(
        index="ano_internacao",
        columns="mes_internacao",
        values="percentual_obito_srag",
    )
    .reindex(
        columns=ORDEM_MESES
    )
    .sort_index()
)

matriz_denominador_ano_mes = (
    dados_matrizes_mensais.pivot(
        index="ano_internacao",
        columns="mes_internacao",
        values="quantidade_registros",
    )
    .reindex(
        columns=ORDEM_MESES
    )
    .sort_index()
)

matriz_mes_completo_ano_mes = (
    dados_matrizes_mensais.pivot(
        index="ano_internacao",
        columns="mes_internacao",
        values="mes_completo",
    )
    .reindex(
        columns=ORDEM_MESES
    )
    .sort_index()
)

matriz_volume_ano_mes.columns = (
    ROTULOS_MESES
)

matriz_obito_ano_mes.columns = (
    ROTULOS_MESES
)

matriz_denominador_ano_mes.columns = (
    ROTULOS_MESES
)

matriz_mes_completo_ano_mes.columns = (
    ROTULOS_MESES
)

if matriz_volume_ano_mes.empty:
    raise ValueError(
        "A matriz mensal de volume está vazia."
    )

if matriz_obito_ano_mes.empty:
    raise ValueError(
        "A matriz mensal de óbito está vazia."
    )

valores_obito_mensais = (
    matriz_obito_ano_mes
    .stack()
    .dropna()
)

if valores_obito_mensais.empty:
    raise ValueError(
        "A matriz mensal de óbito não possui "
        "percentuais válidos."
    )

percentuais_obito_validos = (
    valores_obito_mensais.ge(0)
    & valores_obito_mensais.le(100)
).all()

if not bool(percentuais_obito_validos):
    raise RuntimeError(
        "A matriz mensal de óbito contém percentuais "
        "fora do intervalo de 0% a 100%."
    )

print(
    "Dimensão da matriz de volume:",
    matriz_volume_ano_mes.shape,
)

print(
    "Dimensão da matriz de óbito:",
    matriz_obito_ano_mes.shape,
)

display(
    matriz_volume_ano_mes
)

display(
    matriz_obito_ano_mes
)

In [ ]:
anotacoes_volume_ano_mes = (
    matriz_volume_ano_mes
    .copy()
    .astype("object")
)

for ano in matriz_volume_ano_mes.index:
    for mes in matriz_volume_ano_mes.columns:
        quantidade = (
            matriz_volume_ano_mes.loc[
                ano,
                mes,
            ]
        )

        if np.asarray(pd.isna(quantidade)).any():
            anotacao = ""
        else:
            else:
                quantidade_inteira = int(
                np.asarray(quantidade).item()
                )

                if quantidade_inteira >= 1_000_000:
                anotacao = (
                    f"{quantidade_inteira / 1_000_000:.1f} mi"
                )
                elif quantidade_inteira >= 1_000:
                anotacao = (
                    f"{quantidade_inteira / 1_000:.0f} mil"
                )
                else:
                anotacao = str(quantidade_inteira)

            if quantidade_inteira >= 1_000_000:
            anotacao = (
                f"{quantidade_inteira / 1_000_000:.1f} mi"
            )
            elif quantidade_inteira >= 1_000:
            anotacao = (
                f"{quantidade_inteira / 1_000:.0f} mil"
            )
            else:
            anotacao = str(
                quantidade_inteira
            )

        anotacoes_volume_ano_mes.loc[
            ano,
            mes,
        ] = anotacao

figura_heatmap_volume, eixo_heatmap_volume = plt.subplots(
    figsize=(
        15,
        7,
    )
)

sns.heatmap(
    matriz_volume_ano_mes,
    annot=anotacoes_volume_ano_mes,
    fmt="",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Quantidade de registros",
    },
    annot_kws={
        "fontsize": 8,
    },
    ax=eixo_heatmap_volume,
)

eixo_heatmap_volume.set_title(
    "Volume de internações segundo ano e mês"
)

eixo_heatmap_volume.set_xlabel(
    "Mês da internação"
)

eixo_heatmap_volume.set_ylabel(
    "Ano da internação"
)

eixo_heatmap_volume.set_xticklabels(
    eixo_heatmap_volume.get_xticklabels(),
    rotation=0,
)

eixo_heatmap_volume.set_yticklabels(
    eixo_heatmap_volume.get_yticklabels(),
    rotation=0,
)

figura_heatmap_volume.text(
    0.01,
    0.01,
    (
        "A intensidade da cor representa o volume absoluto. "
        "Células vazias correspondem a meses fora da cobertura "
        "temporal da coorte."
    ),
    fontsize=8,
)

figura_heatmap_volume.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1,
    ]
)

plt.show()

plt.close(
    figura_heatmap_volume
)

In [ ]:
valor_maximo_obito_mensal = float(
    valores_obito_mensais.max()
)

limite_superior_escala_obito = max(
    10.0,
    float(
        np.ceil(
            valor_maximo_obito_mensal
            / 5
        )
        * 5
    ),
)

anotacoes_obito_ano_mes = (
    matriz_obito_ano_mes
    .copy()
    .astype("object")
)

for ano in matriz_obito_ano_mes.index:
    for mes in matriz_obito_ano_mes.columns:
        percentual = (
            matriz_obito_ano_mes.loc[
                ano,
                mes,
            ]
        )

        denominador = (
            matriz_denominador_ano_mes.loc[
                ano,
                mes,
            ]
        )

        mes_completo = (
            matriz_mes_completo_ano_mes.loc[
                ano,
                mes,
            ]
        )

        if pd.isna(
            percentual
        ):
            anotacao = ""
        else:
            percentual_formatado = (
                f"{float(percentual):.1f}%"
            )

            if pd.isna(
                denominador
            ):
                anotacao = percentual_formatado
            else:
                denominador_formatado = (
                    f"{int(denominador):,}"
                    .replace(
                        ",",
                        ".",
                    )
                )

                anotacao = (
                    f"{percentual_formatado}\n"
                    f"n={denominador_formatado}"
                )

                if (
                    not pd.isna(
                        mes_completo
                    )
                    and not bool(
                        mes_completo
                    )
                ):
                    anotacao = (
                        f"{anotacao}\n"
                        "parcial"
                    )

        anotacoes_obito_ano_mes.loc[
            ano,
            mes,
        ] = anotacao

figura_heatmap_obito, eixo_heatmap_obito = plt.subplots(
    figsize=(
        16,
        8,
    )
)

sns.heatmap(
    matriz_obito_ano_mes,
    annot=anotacoes_obito_ano_mes,
    fmt="",
    cmap="Reds",
    vmin=0,
    vmax=limite_superior_escala_obito,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Óbito por SRAG (%)",
    },
    annot_kws={
        "fontsize": 7,
    },
    ax=eixo_heatmap_obito,
)

eixo_heatmap_obito.set_title(
    "Óbito por SRAG segundo ano e mês da internação"
)

eixo_heatmap_obito.set_xlabel(
    "Mês da internação"
)

eixo_heatmap_obito.set_ylabel(
    "Ano da internação"
)

eixo_heatmap_obito.set_xticklabels(
    eixo_heatmap_obito.get_xticklabels(),
    rotation=0,
)

eixo_heatmap_obito.set_yticklabels(
    eixo_heatmap_obito.get_yticklabels(),
    rotation=0,
)

figura_heatmap_obito.text(
    0.01,
    0.01,
    (
        "Cada célula apresenta o percentual observado de óbito "
        "e o denominador mensal. O último mês incompleto é "
        "identificado como parcial."
    ),
    fontsize=8,
)

figura_heatmap_obito.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1,
    ]
)

plt.show()

plt.close(
    figura_heatmap_obito
)

In [ ]:
memoria_antes_liberacao_gb = memoria_processo_gb()
if "coorte" in globals():
    del coorte
gc.collect()
try:
    import pyarrow as pa
    pa.default_memory_pool().release_unused()
except Exception:
    pass
print(
    "Memória do kernel antes e depois de liberar a coorte:",
    f"{memoria_antes_liberacao_gb:.2f} GB ->",
    f"{memoria_processo_gb():.2f} GB",
)

from itertools import combinations

import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.stats import chi2_contingency, f_oneway, kruskal
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import KNeighborsClassifier

from src.dados import load_supervised_artifact
from src.features import construir_preprocessador_clustering

NS_SEMENTE = 42
NS_TAMANHO_AMOSTRA = 80_000
NS_TAMANHO_SUBAMOSTRA = 10_000
NS_COMPONENTES_SVD = 40
NS_VARIANCIA_ALVO = 0.90
NS_K_KMEANS = range(2, 11)
NS_K_HCA = range(2, 11)
NS_K_MINIMO_DESCRICAO = 3
NS_LIMITE_CLUSTER_PEQUENO = 1.0
NS_CANDIDATOS_POR_METODO = 3
NS_REPETICOES_ESTABILIDADE = 5
NS_N_INIT = 10
NS_MAX_ITER = 200
NS_TAMANHO_LOTE = 2_048
NS_N_INIT_FINAL = 30
NS_MAX_ITER_FINAL = 300

NS_PASTA_FINAL = C.OUTPUTS_DIR / "final"
for ns_pasta in (C.TABLES_DIR, C.FIGURES_DIR, NS_PASTA_FINAL):
    ns_pasta.mkdir(parents=True, exist_ok=True)


def ns_renumerar_por_tamanho(rotulos):
    """Numera os grupos de 0 em diante, do maior para o menor, para o número ter sempre o mesmo sentido."""
    tamanhos = pd.Series(rotulos).value_counts().sort_index()
    ordem = tamanhos.sort_values(ascending=False, kind="stable").index
    mapa = {antigo: novo for novo, antigo in enumerate(ordem)}
    return pd.Series(rotulos).map(mapa).to_numpy(dtype=int)


def ns_candidatos_kmeans(diagnostico):
    """Candidatos a k: maior silhueta, cotovelo e os melhores por silhueta, entre os sem grupo muito pequeno."""
    elegiveis = diagnostico.loc[diagnostico["menor_cluster_pct"] >= NS_LIMITE_CLUSTER_PEQUENO]
    if elegiveis.empty:
        elegiveis = diagnostico.copy()
    por_silhueta = elegiveis.sort_values(["silhueta", "k"], ascending=[False, True])
    k_silhueta = int(por_silhueta["k"].iloc[0])
    ordenado = elegiveis.sort_values("k").reset_index(drop=True)
    anterior = ordenado["inercia_por_registro"].shift(1)
    ordenado["reducao_inercia_pct"] = 100 * (anterior - ordenado["inercia_por_registro"]) / anterior
    reducoes = ordenado.dropna(subset=["reducao_inercia_pct"])
    if reducoes.empty:
        k_cotovelo = k_silhueta
    else:
        k_cotovelo = int(reducoes.sort_values(["reducao_inercia_pct", "k"], ascending=[False, True])["k"].iloc[0])
    melhores = por_silhueta["k"].head(NS_CANDIDATOS_POR_METODO).astype(int).tolist()
    return sorted(dict.fromkeys([k_silhueta, k_cotovelo] + melhores))


def ns_maior_salto(tabela, k_minimo):
    """k de maior salto de distância entre fusões, entre os cortes sem grupo muito pequeno."""
    elegiveis = tabela.loc[(tabela["k"] >= k_minimo) & (tabela["menor_cluster_pct"] >= NS_LIMITE_CLUSTER_PEQUENO)]
    if elegiveis.empty:
        elegiveis = tabela.loc[tabela["k"] >= k_minimo]
    return int(elegiveis.sort_values(["salto_distancia", "k"], ascending=[False, True])["k"].iloc[0])


def ns_perfil_grupos(base, rotulos, numericas, categoricas):
    """Uma linha por variável: teste de associação com o grupo, tamanho do efeito e amplitude entre grupos."""
    grupos = np.asarray(rotulos)
    linhas = []
    for coluna in categoricas:
        texto = base[coluna].astype("string").str.strip()
        ausente = (texto.isna() | texto.eq("")).to_numpy(dtype=bool)
        sim = texto.isin(["1", "1.0"]).to_numpy(dtype=bool)
        valores = np.where(ausente, "AUSENTE", texto.fillna("").to_numpy(dtype=object))
        tabela = pd.crosstab(valores, grupos)
        qui2, p_qui2 = chi2_contingency(tabela, correction=False)[:2]
        menor = min(tabela.shape) - 1
        efeito = float(np.sqrt(qui2 / (tabela.to_numpy().sum() * menor))) if menor > 0 else float("nan")
        pct_ausente = pd.Series(ausente).groupby(grupos).mean() * 100
        pct_sim = pd.Series(sim).groupby(grupos).mean() * 100
        linhas.append({
            "variavel": coluna, "tipo": "categorica", "medida": "V_de_Cramer", "efeito": efeito,
            "teste": "qui_quadrado", "estatistica": float(qui2), "p_valor": float(p_qui2),
            "eta2_por_postos": np.nan, "p_kruskal": np.nan,
            "pct_ausente_amostra": float(ausente.mean() * 100),
            "amplitude_pct_ausente": float(pct_ausente.max() - pct_ausente.min()),
            "amplitude_pct_sim": float(pct_sim.max() - pct_sim.min()),
            "media_minima": np.nan, "media_maxima": np.nan,
        })
    for coluna in numericas:
        serie = pd.to_numeric(base[coluna], errors="coerce").reset_index(drop=True)
        por_grupo = [dados.dropna().to_numpy(dtype=float) for _, dados in serie.groupby(grupos)]
        por_grupo = [dados for dados in por_grupo if dados.size]
        n, k = sum(dados.size for dados in por_grupo), len(por_grupo)
        estatistica_f = p_anova = eta2 = estatistica_h = p_kruskal = float("nan")
        if k >= 2 and n > k:
            todos = np.concatenate(por_grupo)
            media_geral = todos.mean()
            soma_total = float(((todos - media_geral) ** 2).sum())
            soma_entre = float(sum(d.size * (d.mean() - media_geral) ** 2 for d in por_grupo))
            if soma_total > 0:
                estatistica_f, p_anova = (float(v) for v in f_oneway(*por_grupo))
                eta2 = soma_entre / soma_total
                estatistica_h, p_kruskal = (float(v) for v in kruskal(*por_grupo))
        eta2_postos = max(0.0, (estatistica_h - k + 1) / (n - k)) if not np.isnan(estatistica_h) else float("nan")
        medias = serie.groupby(grupos).mean()
        linhas.append({
            "variavel": coluna, "tipo": "numerica", "medida": "eta2_ANOVA", "efeito": eta2,
            "teste": "ANOVA_um_fator", "estatistica": estatistica_f, "p_valor": p_anova,
            "eta2_por_postos": eta2_postos, "p_kruskal": p_kruskal,
            "pct_ausente_amostra": float(serie.isna().mean() * 100),
            "amplitude_pct_ausente": np.nan, "amplitude_pct_sim": np.nan,
            "media_minima": float(medias.min()), "media_maxima": float(medias.max()),
        })
    return pd.DataFrame(linhas).sort_values("efeito", ascending=False, ignore_index=True)

In [ ]:
ns_inicio = time.perf_counter()

with open(ROOT_DIR / "data" / "analytical" / "manifesto_supervisionado.json", encoding="utf-8") as ns_arquivo:
    ns_manifesto = json.load(ns_arquivo)

NS_NUMERICAS = list(ns_manifesto["numeric_columns"])
NS_CATEGORICAS = list(ns_manifesto["categorical_columns"])

ns_treino, _ = load_supervised_artifact(
    ROOT_DIR,
    expected_kind="final",
    partitions=("train_ref",),
    columns=["id_registro", "ano_internacao", *NS_NUMERICAS, *NS_CATEGORICAS],
)

if len(ns_treino) < NS_TAMANHO_AMOSTRA:
    raise ValueError(
        f"O bloco train_ref tem {len(ns_treino)} fichas, menos que a amostra pedida ({NS_TAMANHO_AMOSTRA})."
    )

ns_amostra = ns_treino.sample(n=NS_TAMANHO_AMOSTRA, random_state=NS_SEMENTE).reset_index(drop=True)
print("Fichas em train_ref:", len(ns_treino), "| fichas na amostra:", len(ns_amostra))

with open(ROOT_DIR / "data" / "analytical" / "manifesto_particao.json", encoding="utf-8") as ns_arquivo_particao:
    NS_ANOS_TREINO = {int(ano) for ano in json.load(ns_arquivo_particao)["blocos"]["train_ref"]["anos"]}
ns_anos_amostra = {int(ano) for ano in ns_amostra["ano_internacao"].dropna().unique()}
if ns_anos_amostra - NS_ANOS_TREINO or ns_amostra["ano_internacao"].isna().any():
    raise ValueError(
        f"A amostra tem fichas fora dos anos do train_ref: {sorted(ns_anos_amostra - NS_ANOS_TREINO)}."
    )
print("Anos das fichas da amostra:", sorted(ns_anos_amostra), "| anos do bloco train_ref:", sorted(NS_ANOS_TREINO))
del ns_treino
gc.collect()

ns_preprocessador = construir_preprocessador_clustering(NS_NUMERICAS, NS_CATEGORICAS)
ns_matriz = ns_preprocessador.fit_transform(ns_amostra[NS_NUMERICAS + NS_CATEGORICAS])
ns_colunas_onehot = ns_matriz.shape[1]
print("Colunas depois do one-hot:", ns_colunas_onehot, "| valores não nulos:", ns_matriz.nnz)
print("Peso aplicado às numéricas:", round(ns_preprocessador.peso_numericas_, 3), "| variância média de uma categórica:", round(ns_preprocessador.variancia_media_categorica_, 3))

ns_svd = TruncatedSVD(n_components=NS_COMPONENTES_SVD, random_state=NS_SEMENTE)
ns_projecao = ns_svd.fit_transform(ns_matriz)
ns_variancia_acumulada = np.cumsum(ns_svd.explained_variance_ratio_)
ns_alcancou = np.flatnonzero(ns_variancia_acumulada >= NS_VARIANCIA_ALVO)
ns_n_componentes = int(ns_alcancou[0]) + 1 if ns_alcancou.size else NS_COMPONENTES_SVD
ns_projecao = ns_projecao[:, :ns_n_componentes]
del ns_matriz
gc.collect()

ns_gerador = np.random.default_rng(NS_SEMENTE)
ns_indices_sub = np.sort(ns_gerador.choice(len(ns_projecao), size=NS_TAMANHO_SUBAMOSTRA, replace=False))

print(
    "Componentes usados:", ns_n_componentes,
    "| variância acumulada:", round(float(ns_variancia_acumulada[ns_n_componentes - 1]), 3),
    "| variância acumulada com 40:", round(float(ns_variancia_acumulada[-1]), 3),
)
print("Tempo da preparação:", f"{time.perf_counter() - ns_inicio:.1f} s", "| memória:", f"{memoria_processo_gb():.2f} GB")

In [ ]:
ns_inicio = time.perf_counter()
ns_rotulos_por_k = {}
ns_linhas_diagnostico = []

for ns_k in NS_K_KMEANS:
    ns_modelo = MiniBatchKMeans(
        n_clusters=ns_k, n_init=NS_N_INIT, max_iter=NS_MAX_ITER, batch_size=NS_TAMANHO_LOTE,
        random_state=NS_SEMENTE + ns_k,
    )
    ns_rotulos = ns_modelo.fit_predict(ns_projecao)
    ns_rotulos_por_k[ns_k] = ns_rotulos
    ns_tamanhos = pd.Series(ns_rotulos).value_counts()
    ns_linhas_diagnostico.append({
        "metodo": "kmeans",
        "k": ns_k,
        "silhueta": float(silhouette_score(ns_projecao[ns_indices_sub], ns_rotulos[ns_indices_sub])),
        "inercia_por_registro": float(ns_modelo.inertia_ / len(ns_projecao)),
        "menor_cluster_pct": float(100 * ns_tamanhos.min() / len(ns_rotulos)),
    })

ns_diagnostico = pd.DataFrame(ns_linhas_diagnostico)
ns_candidatos = ns_candidatos_kmeans(ns_diagnostico)

ns_linhas_estabilidade = []
for ns_k in ns_candidatos:
    ns_repeticoes = []
    ns_pequeno = False
    for ns_repeticao in range(NS_REPETICOES_ESTABILIDADE):
        ns_modelo = MiniBatchKMeans(
            n_clusters=ns_k, n_init=NS_N_INIT, max_iter=NS_MAX_ITER, batch_size=NS_TAMANHO_LOTE,
            random_state=NS_SEMENTE + ns_k * 100 + ns_repeticao,
        )
        ns_rotulos = ns_modelo.fit_predict(ns_projecao)
        ns_repeticoes.append(ns_rotulos)
        ns_pequeno = ns_pequeno or (100 * pd.Series(ns_rotulos).value_counts().min() / len(ns_rotulos) < NS_LIMITE_CLUSTER_PEQUENO)
    ns_aris = [adjusted_rand_score(a, b) for a, b in combinations(ns_repeticoes, 2)]
    ns_linhas_estabilidade.append({
        "k": ns_k,
        "ari_sementes_medio": float(np.mean(ns_aris)),
        "ari_sementes_minimo": float(np.min(ns_aris)),
        "alguma_repeticao_cluster_pequeno": bool(ns_pequeno),
    })
    del ns_repeticoes
    gc.collect()

ns_estabilidade = pd.DataFrame(ns_linhas_estabilidade).merge(
    ns_diagnostico[["k", "silhueta", "menor_cluster_pct"]], on="k", validate="one_to_one"
)
ns_estabilidade["elegivel"] = ~ns_estabilidade["alguma_repeticao_cluster_pequeno"]
ns_estabilidade["posicao_silhueta"] = ns_estabilidade["silhueta"].rank(method="min", ascending=False)
ns_estabilidade["posicao_estabilidade"] = ns_estabilidade["ari_sementes_medio"].rank(method="min", ascending=False)
ns_estabilidade["posicao_tamanho_minimo"] = ns_estabilidade["menor_cluster_pct"].rank(method="min", ascending=False)
ns_estabilidade["pontuacao_diagnostica"] = (
    ns_estabilidade["posicao_silhueta"] + ns_estabilidade["posicao_estabilidade"] + ns_estabilidade["posicao_tamanho_minimo"]
)
ns_estabilidade = ns_estabilidade.sort_values(
    ["elegivel", "pontuacao_diagnostica", "k"], ascending=[False, True, True], ignore_index=True
)
NS_K_ESCOLHIDO_KMEANS = int(ns_estabilidade["k"].iloc[0])

ns_modelo_final = MiniBatchKMeans(
    n_clusters=NS_K_ESCOLHIDO_KMEANS, n_init=NS_N_INIT_FINAL, max_iter=NS_MAX_ITER_FINAL,
    batch_size=NS_TAMANHO_LOTE, random_state=NS_SEMENTE,
)
ns_rotulos_kmeans = ns_renumerar_por_tamanho(ns_modelo_final.fit_predict(ns_projecao))

ns_tabela_kmeans = ns_diagnostico.merge(
    ns_estabilidade[["k", "ari_sementes_medio", "ari_sementes_minimo", "pontuacao_diagnostica"]], on="k", how="left"
)
ns_tabela_kmeans["candidato"] = ns_tabela_kmeans["k"].isin(ns_candidatos)
ns_tabela_kmeans["salto_distancia"] = np.nan
ns_tabela_kmeans["escolhido"] = ns_tabela_kmeans["k"].eq(NS_K_ESCOLHIDO_KMEANS)

display(ns_tabela_kmeans.round(4))
print("Candidatos a k:", ns_candidatos)
print("k escolhido no K-Means:", NS_K_ESCOLHIDO_KMEANS)
print("Tamanho dos grupos:", pd.Series(ns_rotulos_kmeans).value_counts().sort_index().to_dict())

ns_figura, ns_eixo = plt.subplots(figsize=(7, 4))
ns_eixo.plot(ns_tabela_kmeans["k"], ns_tabela_kmeans["silhueta"], marker="o", label="Silhueta")
ns_candidatas = ns_tabela_kmeans.loc[ns_tabela_kmeans["candidato"]]
ns_eixo.plot(ns_candidatas["k"], ns_candidatas["ari_sementes_medio"], marker="s", label="ARI entre repetições (candidatos)")
ns_eixo.axvline(NS_K_ESCOLHIDO_KMEANS, color="gray", linestyle="--", linewidth=1)
ns_eixo.set_xlabel("Número de grupos (k)")
ns_eixo.set_xticks(list(NS_K_KMEANS))
ns_eixo.legend()
ns_figura.tight_layout()
NS_FIGURA_KMEANS = C.FIGURES_DIR / "04_eda_ns_onehot_kmeans.png"
ns_figura.savefig(NS_FIGURA_KMEANS, dpi=150, bbox_inches="tight")
plt.show()
plt.close(ns_figura)
print("Figura salva em:", NS_FIGURA_KMEANS)
print("Tempo do K-Means:", f"{time.perf_counter() - ns_inicio:.1f} s", "| memória:", f"{memoria_processo_gb():.2f} GB")

In [ ]:
ns_inicio = time.perf_counter()

ns_n_sub = len(ns_indices_sub)
ns_ligacao = linkage(ns_projecao[ns_indices_sub], method="ward")

ns_linhas_hca = []
for ns_k in NS_K_HCA:
    ns_corte = fcluster(ns_ligacao, ns_k, criterion="maxclust")
    ns_linhas_hca.append({
        "metodo": "hca",
        "k": ns_k,
        "silhueta": float(silhouette_score(ns_projecao[ns_indices_sub], ns_corte)),
        "salto_distancia": float(ns_ligacao[ns_n_sub - ns_k, 2] - ns_ligacao[ns_n_sub - ns_k - 1, 2]),
        "menor_cluster_pct": float(100 * pd.Series(ns_corte).value_counts().min() / ns_n_sub),
        "ari_com_kmeans_mesmo_k": float(adjusted_rand_score(ns_rotulos_por_k[ns_k][ns_indices_sub], ns_corte)),
    })
ns_tabela_hca = pd.DataFrame(ns_linhas_hca)

NS_K_HCA_PRINCIPAL = ns_maior_salto(ns_tabela_hca, min(NS_K_HCA))
NS_K_ESCOLHIDO_HCA = NS_K_HCA_PRINCIPAL if NS_K_HCA_PRINCIPAL >= NS_K_MINIMO_DESCRICAO else ns_maior_salto(ns_tabela_hca, NS_K_MINIMO_DESCRICAO)
ns_tabela_hca["escolhido"] = ns_tabela_hca["k"].eq(NS_K_HCA_PRINCIPAL)
ns_tabela_hca["nivel_de_descricao"] = ns_tabela_hca["k"].eq(NS_K_ESCOLHIDO_HCA)
for ns_coluna in ("ari_sementes_medio", "ari_sementes_minimo"):
    ns_tabela_hca[ns_coluna] = np.nan


def ns_rotulos_do_corte(k):
    """Rótulos dos 80 mil registros: os 10 mil do hierárquico e, para os demais, o grupo dos 15 vizinhos mais próximos."""
    rotulos_sub = fcluster(ns_ligacao, k, criterion="maxclust")
    vizinhos = KNeighborsClassifier(n_neighbors=15).fit(ns_projecao[ns_indices_sub], rotulos_sub)
    demais = np.setdiff1d(np.arange(len(ns_projecao)), ns_indices_sub)
    rotulos = np.empty(len(ns_projecao), dtype=int)
    rotulos[ns_indices_sub] = rotulos_sub
    rotulos[demais] = vizinhos.predict(ns_projecao[demais])
    return ns_renumerar_por_tamanho(rotulos)


ns_rotulos_hca_principal = ns_rotulos_do_corte(NS_K_HCA_PRINCIPAL)
ns_rotulos_hca = ns_rotulos_hca_principal if NS_K_ESCOLHIDO_HCA == NS_K_HCA_PRINCIPAL else ns_rotulos_do_corte(NS_K_ESCOLHIDO_HCA)

display(ns_tabela_hca.round(4))
print("k do corte principal do hierárquico (maior salto):", NS_K_HCA_PRINCIPAL)
print("k do nível de descrição (maior salto entre os cortes com pelo menos", NS_K_MINIMO_DESCRICAO, "grupos):", NS_K_ESCOLHIDO_HCA)
print("Concordância com o K-Means no k escolhido do K-Means:",
      round(float(ns_tabela_hca.loc[ns_tabela_hca["k"].eq(NS_K_ESCOLHIDO_KMEANS), "ari_com_kmeans_mesmo_k"].iloc[0]), 4))
print("Tamanho dos grupos do corte principal (80 mil):", pd.Series(ns_rotulos_hca_principal).value_counts().sort_index().to_dict())
print("Tamanho dos grupos do nível de descrição (80 mil):", pd.Series(ns_rotulos_hca).value_counts().sort_index().to_dict())


def ns_altura_do_corte(k):
    return (ns_ligacao[ns_n_sub - k - 1, 2] + ns_ligacao[ns_n_sub - k, 2]) / 2


ns_figura, ns_eixo = plt.subplots(figsize=(10, 4.5))
dendrogram(ns_ligacao, truncate_mode="lastp", p=30, no_labels=True, ax=ns_eixo)
ns_eixo.axhline(ns_altura_do_corte(NS_K_HCA_PRINCIPAL), color="gray", linestyle="--", linewidth=1, label=f"corte principal (k = {NS_K_HCA_PRINCIPAL})")
if NS_K_ESCOLHIDO_HCA != NS_K_HCA_PRINCIPAL:
    ns_eixo.axhline(ns_altura_do_corte(NS_K_ESCOLHIDO_HCA), color="black", linestyle=":", linewidth=1, label=f"nível de descrição (k = {NS_K_ESCOLHIDO_HCA})")
ns_eixo.set_ylabel("Distância de Ward")
ns_eixo.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
ns_figura.tight_layout()
NS_FIGURA_DENDROGRAMA = C.FIGURES_DIR / "04_eda_ns_onehot_dendrograma.png"
ns_figura.savefig(NS_FIGURA_DENDROGRAMA, dpi=150, bbox_inches="tight")
plt.show()
plt.close(ns_figura)
print("Figura salva em:", NS_FIGURA_DENDROGRAMA)
print("Tempo do agrupamento hierárquico:", f"{time.perf_counter() - ns_inicio:.1f} s", "| memória:", f"{memoria_processo_gb():.2f} GB")

In [ ]:
ns_inicio = time.perf_counter()

ns_perfil_kmeans = ns_perfil_grupos(ns_amostra, ns_rotulos_kmeans, NS_NUMERICAS, NS_CATEGORICAS)
ns_perfil_hca = ns_perfil_grupos(ns_amostra, ns_rotulos_hca, NS_NUMERICAS, NS_CATEGORICAS)

print("Perfil dos grupos do K-Means: dez variáveis de maior efeito")
display(ns_perfil_kmeans.head(10).round(3))
print("Perfil dos grupos do agrupamento hierárquico (nível de descrição): dez variáveis de maior efeito")
display(ns_perfil_hca.head(10).round(3))

ns_metricas = pd.concat([ns_tabela_kmeans, ns_tabela_hca], ignore_index=True)
ns_rotulos_tabela = pd.DataFrame({
    "id_registro": ns_amostra["id_registro"].to_numpy(),
    "grupo_kmeans": ns_rotulos_kmeans,
    "grupo_hca": ns_rotulos_hca,
    "grupo_hca_principal": ns_rotulos_hca_principal,
})
ns_resumo = pd.DataFrame({
    "metrica": [
        "fichas_na_amostra", "fichas_na_subamostra", "colunas_apos_onehot", "peso_numericas", "componentes_svd",
        "variancia_acumulada_svd", "k_kmeans", "k_hca_principal", "k_hca_descricao",
    ],
    "valor": [
        len(ns_amostra), len(ns_indices_sub), ns_colunas_onehot, round(ns_preprocessador.peso_numericas_, 4), ns_n_componentes,
        round(float(ns_variancia_acumulada[ns_n_componentes - 1]), 4), NS_K_ESCOLHIDO_KMEANS, NS_K_HCA_PRINCIPAL, NS_K_ESCOLHIDO_HCA,
    ],
})

NS_ARQUIVOS_SAIDA = {
    "metricas": C.TABLES_DIR / "ns_onehot_metricas.csv",
    "resumo": C.TABLES_DIR / "ns_onehot_resumo.csv",
    "rotulos": C.TABLES_DIR / "ns_onehot_rotulos.parquet",
    "perfil_kmeans": NS_PASTA_FINAL / "perfil_clusters_kmeans.csv",
    "perfil_hca": NS_PASTA_FINAL / "perfil_clusters_hca.csv",
}
ns_metricas.to_csv(NS_ARQUIVOS_SAIDA["metricas"], index=False)
ns_resumo.to_csv(NS_ARQUIVOS_SAIDA["resumo"], index=False)
ns_rotulos_tabela.to_parquet(NS_ARQUIVOS_SAIDA["rotulos"], index=False)
ns_perfil_kmeans.to_csv(NS_ARQUIVOS_SAIDA["perfil_kmeans"], index=False)
ns_perfil_hca.to_csv(NS_ARQUIVOS_SAIDA["perfil_hca"], index=False)

for ns_nome, ns_caminho in NS_ARQUIVOS_SAIDA.items():
    print("Gravado:", ns_caminho)
print("Tempo do perfil e da gravação:", f"{time.perf_counter() - ns_inicio:.1f} s", "| memória:", f"{memoria_processo_gb():.2f} GB")

In [ ]:
ns_inicio = time.perf_counter()

NS_COMORBIDADES = [
    "puerpera", "cardiopati", "hematologi", "sind_down", "hepatica", "asma", "diabetes",
    "neurologic", "pneumopati", "imunodepre", "renal", "obesidade", "out_morbi",
]
NS_SINTOMAS_SECUNDARIOS = ["diarreia", "vomito", "garganta", "perd_olft", "perd_pala", "dor_abd", "fadiga"]
NS_LIMITE_SINTOMAS_EM_BRANCO = 6
NS_ORDEM_FAIXAS = ["<1", "1-4", "5-11", "12-17", "18-39", "40-59", "60-79", "80+", "desconhecida"]
NS_MINIMO_FICHAS_PARA_PERCENTUAL = 100
NS_ALVO = ns_manifesto["target_column"]

ns_faltantes = set(NS_COMORBIDADES + NS_SINTOMAS_SECUNDARIOS + ["idade_anos"]) - set(
    NS_NUMERICAS + NS_CATEGORICAS + list(ns_manifesto.get("audit_columns") or [])
)
if ns_faltantes:
    raise KeyError(f"Variáveis ausentes no artefato: {sorted(ns_faltantes)}")

ns_base, _ = load_supervised_artifact(
    ROOT_DIR,
    expected_kind="final",
    partitions=("train_ref",),
    columns=["id_registro", NS_ALVO, "ano_internacao", "faixa_etaria", "fator_risc", "idade_anos",
             *NS_COMORBIDADES, *NS_SINTOMAS_SECUNDARIOS],
)
ns_caract = ns_rotulos_tabela.merge(ns_base, on="id_registro", how="left", validate="one_to_one")
if len(ns_caract) != len(ns_rotulos_tabela) or ns_caract["ano_internacao"].isna().any():
    raise ValueError("Há fichas da amostra que não estão no bloco train_ref.")
del ns_base
gc.collect()

ns_caract["idade_anos"] = pd.to_numeric(ns_caract["idade_anos"], errors="coerce")
ns_caract["faixa_etaria"] = ns_caract["faixa_etaria"].astype("string")


def ns_como_texto(coluna):
    return ns_caract[coluna].astype("string").str.strip().replace("", pd.NA)


ns_respostas = {coluna: ns_como_texto(coluna) for coluna in NS_COMORBIDADES + NS_SINTOMAS_SECUNDARIOS}
ns_em_branco = pd.DataFrame({coluna: serie.isna() for coluna, serie in ns_respostas.items()})
ns_respondeu = pd.DataFrame({coluna: serie.isin(["1", "2"]) for coluna, serie in ns_respostas.items()})
ns_sim = pd.DataFrame({coluna: serie.eq("1").fillna(False).astype(bool) for coluna, serie in ns_respostas.items()})
ns_nao = pd.DataFrame({coluna: serie.eq("2").fillna(False).astype(bool) for coluna, serie in ns_respostas.items()})

ns_caract["comorbidades_todas_em_branco"] = ns_em_branco[NS_COMORBIDADES].all(axis=1)
ns_caract["sintomas_secundarios_em_branco_n"] = ns_em_branco[NS_SINTOMAS_SECUNDARIOS].sum(axis=1)
ns_caract["pct_sintomas_secundarios_em_branco"] = (
    100 * ns_caract["sintomas_secundarios_em_branco_n"] / len(NS_SINTOMAS_SECUNDARIOS)
)
ns_caract["comorbidades_com_nao"] = ns_nao[NS_COMORBIDADES].any(axis=1)
ns_caract["fator_risc"] = ns_como_texto("fator_risc").fillna("AUSENTE")

ns_caract["modo_de_registro"] = np.select(
    [
        ns_caract["sintomas_secundarios_em_branco_n"] >= NS_LIMITE_SINTOMAS_EM_BRANCO,
        ~ns_caract["comorbidades_com_nao"],
    ],
    ["sintomas_em_branco", "comorbidades_sem_nao"],
    default="sim_e_nao",
)


def ns_resumo(coluna_grupo):
    tabela = ns_caract.groupby(coluna_grupo).agg(
        fichas=("id_registro", "size"),
        pct_comorbidades_todas_em_branco=("comorbidades_todas_em_branco", "mean"),
        pct_sintomas_secundarios_em_branco=("pct_sintomas_secundarios_em_branco", "mean"),
        idade_media=("idade_anos", "mean"),
        idade_mediana=("idade_anos", "median"),
        pct_obitos=(NS_ALVO, "mean"),
    )
    tabela["pct_das_fichas"] = 100 * tabela["fichas"] / tabela["fichas"].sum()
    for coluna in ("pct_comorbidades_todas_em_branco", "pct_obitos"):
        tabela[coluna] = 100 * tabela[coluna]
    tabela.insert(0, "metodo", coluna_grupo)
    return tabela.reset_index().rename(columns={coluna_grupo: "grupo"})


ns_t_resumo = pd.concat([ns_resumo("grupo_hca"), ns_resumo("grupo_kmeans")], ignore_index=True)

ns_t_modo = ns_caract.groupby("modo_de_registro").agg(
    fichas=("id_registro", "size"),
    idade_media=("idade_anos", "mean"),
    pct_obitos=(NS_ALVO, "mean"),
)
ns_t_modo["pct_das_fichas"] = 100 * ns_t_modo["fichas"] / ns_t_modo["fichas"].sum()
ns_t_modo["pct_obitos"] = 100 * ns_t_modo["pct_obitos"]
ns_t_modo = ns_t_modo.join(
    pd.crosstab(ns_caract["modo_de_registro"], ns_caract["grupo_hca"]).add_prefix("fichas_no_grupo_hca_")
).reset_index()

ns_t_ano = pd.concat(
    [
        (pd.crosstab(ns_caract["ano_internacao"], ns_caract[metodo], normalize="index") * 100)
        .rename_axis(columns="grupo")
        .reset_index()
        .melt(id_vars="ano_internacao", value_name="pct_das_fichas_do_ano")
        .assign(metodo=metodo)
        for metodo in ("grupo_hca", "grupo_kmeans")
    ],
    ignore_index=True,
)

ns_t_fator = ns_caract.groupby("fator_risc").agg(
    fichas=("id_registro", "size"),
    pct_comorbidades_todas_em_branco=("comorbidades_todas_em_branco", "mean"),
)
ns_t_fator["pct_comorbidades_todas_em_branco"] = 100 * ns_t_fator["pct_comorbidades_todas_em_branco"]
ns_t_fator = ns_t_fator.join(
    (pd.crosstab(ns_caract["fator_risc"], ns_caract["grupo_hca"], normalize="index") * 100).add_prefix("pct_no_grupo_hca_")
).reset_index()

ns_grupos_hca = ns_caract["grupo_hca"]
ns_n_respondentes = ns_respondeu.groupby(ns_grupos_hca).sum()
ns_n_sim = ns_sim.groupby(ns_grupos_hca).sum()
ns_pct_sim = 100 * ns_n_sim / ns_n_respondentes.where(ns_n_respondentes > 0)


def ns_formato_longo(tabela, nome):
    return tabela.rename_axis(columns="variavel").reset_index().melt(id_vars="grupo_hca", value_name=nome)


ns_t_resp = ns_formato_longo(ns_n_respondentes, "respondentes").merge(
    ns_formato_longo(ns_pct_sim, "pct_sim_entre_respondentes"), on=["grupo_hca", "variavel"]
)

ns_t_idade = (
    ns_caract.groupby(["grupo_hca", "faixa_etaria"])
    .agg(fichas=("id_registro", "size"), obitos=(NS_ALVO, "sum"))
    .reset_index()
)
ns_t_idade["pct_das_fichas_do_grupo"] = (
    100 * ns_t_idade["fichas"] / ns_t_idade.groupby("grupo_hca")["fichas"].transform("sum")
)
ns_t_idade["pct_obitos"] = (100 * ns_t_idade["obitos"] / ns_t_idade["fichas"]).where(
    ns_t_idade["fichas"] >= NS_MINIMO_FICHAS_PARA_PERCENTUAL
)
ns_t_idade["_ordem"] = ns_t_idade["faixa_etaria"].map({f: i for i, f in enumerate(NS_ORDEM_FAIXAS)}).fillna(len(NS_ORDEM_FAIXAS))
ns_t_idade = ns_t_idade.sort_values(["grupo_hca", "_ordem"]).drop(columns="_ordem")

NS_ARQUIVOS_CARACTERIZACAO = {
    "caracterizacao_grupos_resumo.csv": ns_t_resumo,
    "caracterizacao_grupos_modo_de_registro.csv": ns_t_modo,
    "caracterizacao_grupos_ano.csv": ns_t_ano,
    "caracterizacao_grupos_fator_risc.csv": ns_t_fator,
    "caracterizacao_grupos_respondentes.csv": ns_t_resp,
    "caracterizacao_grupos_idade.csv": ns_t_idade,
}
for ns_nome, ns_tabela in NS_ARQUIVOS_CARACTERIZACAO.items():
    ns_tabela.to_csv(NS_PASTA_FINAL / ns_nome, index=False)

print("1) Resumo por grupo (n, preenchimento, idade e óbitos no train_ref)")
display(ns_t_resumo.round(1))
print("2) Modo de registro por regra, e fichas de cada modo em cada grupo do hierárquico")
display(ns_t_modo.round(1))
print("3) Por ano: % das fichas de cada ano em cada grupo (hierárquico)")
display(ns_t_ano.query("metodo == 'grupo_hca'").pivot(index="ano_internacao", columns="grupo", values="pct_das_fichas_do_ano").round(1))
print("4) Por fator de risco: fichas, % com as 13 comorbidades em branco e distribuição nos grupos (%)")
display(ns_t_fator.round(1))
print("5) Entre quem respondeu sim ou não: % de sim e número de respondentes, por grupo do hierárquico")
display(
    ns_t_resp.pivot(index="variavel", columns="grupo_hca", values=["pct_sim_entre_respondentes", "respondentes"])
    .reindex(NS_COMORBIDADES + NS_SINTOMAS_SECUNDARIOS)
    .round(1)
)
print("6) Faixa etária dentro de cada grupo do hierárquico: fichas, % do grupo e % de óbitos")
display(ns_t_idade.round(1))

ns_faixa_de_corte = (
    float(ns_ligacao[-NS_K_ESCOLHIDO_HCA, 2]),
    float(ns_ligacao[-(NS_K_ESCOLHIDO_HCA - 1), 2]),
)
print(
    "Qualquer corte de Ward entre as alturas",
    round(ns_faixa_de_corte[0], 1), "e", round(ns_faixa_de_corte[1], 1),
    f"gera os mesmos {NS_K_ESCOLHIDO_HCA} grupos do nível de descrição.",
)
for ns_nome in NS_ARQUIVOS_CARACTERIZACAO:
    print("Gravado:", NS_PASTA_FINAL / ns_nome)
print("Tempo da caracterização:", f"{time.perf_counter() - ns_inicio:.1f} s", "| memória:", f"{memoria_processo_gb():.2f} GB")